In [102]:
# ==============================================================================
# SHORTAGE AI V2.0 - HYBRID PREDICTION SYSTEM (GBM + DEEP LEARNING ATTENTION)
# ==============================================================================
# Author: Capstone Team
# Date: January 14, 2026 (Simulation)
# Version: 2.0.0
# Description: 
#   This module initializes the environment for a hybrid drug shortage prediction 
#   system. It combines traditional Machine Learning (XGBoost, RandomForest) 
#   with Deep Learning (PyTorch Attention Networks) to predict supply risks 
#   at the DIN and Molecule levels.
# ==============================================================================

# ------------------------------------------------------------------------------
# SECTION 1: LIBRARY IMPORTS AND DEPENDENCY MANAGEMENT
# ------------------------------------------------------------------------------

# 1.1 Standard Library Imports
# ------------------------------------------------------------------------------
import os
import sys
import io
import time
import zipfile
import shutil          # High-level file operations (copying, archiving)
import warnings        # Warning control (suppressing deprecation warnings)
import random          # Python standard random number generator
import logging         # Professional logging facility
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Optional, Union, Any # Type hinting
from collections import Counter
import re
import json
# 1.2 Data Processing & Numerical Computing
# ------------------------------------------------------------------------------
import pandas as pd    # Data manipulation and analysis
import numpy as np     # Numerical computing and array operations
import requests        # HTTP library for API calls
import urllib3         # Low-level HTTP client
from urllib3.util import retry  # Robust retry logic for unstable APIs

# 1.3 Traditional Machine Learning Stack (Scikit-Learn & Boosters)
# ------------------------------------------------------------------------------
# Gradient Boosting Frameworks
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import (
    RandomForestClassifier, 
    HistGradientBoostingClassifier,
    GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression

# Model Selection & Preprocessing
from sklearn.model_selection import (
    TimeSeriesSplit,   # Crucial for time-dependent supply chain data
    train_test_split, 
    StratifiedKFold
)
from sklearn.preprocessing import (
    StandardScaler, 
    MinMaxScaler, 
    LabelEncoder,
    RobustScaler       # Robust to outliers in sales/inventory data
)

# Metrics & Evaluation
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import (
    classification_report, 
    roc_auc_score, 
    roc_curve,
    precision_recall_curve, 
    confusion_matrix, 
    auc,
    precision_score, 
    recall_score, 
    f1_score,
    average_precision_score
)
from sklearn.inspection import permutation_importance

# Optional: CatBoost (Handling Import Errors Gracefully)
try:
    from catboost import CatBoostClassifier, Pool
    CATBOOST_ACTIVE = True
except ImportError:
    CATBOOST_ACTIVE = False
    # We will log this warning in the setup phase

# 1.4 Deep Learning Stack (PyTorch)
# ------------------------------------------------------------------------------
# Core PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# 1.5 Visualization & Interpretability
# ------------------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import shap  # SHapley Additive exPlanations for model interpretability
except ImportError:
    pass

# ==============================================================================
# SECTION 2: SYSTEM CONFIGURATION & ARCHITECTURE
# ==============================================================================

class Config:
    """
    Centralized configuration management for ShortageAI v2.0.
    
    This class acts as a static container for all hyperparameters, file paths, 
    business logic thresholds, and system settings. It ensures consistency 
    across the training and inference pipelines.

    Attributes:
        SEED (int): Global random seed for reproducibility.
        DATA_DIR (Path): Directory containing raw input files (CSV/Excel).
        OUTPUT_DIR (Path): Directory for saving models, plots, and logs.
        THRESHOLD_DOS (float): 'Days of Supply' threshold used to define a shortage event.
        WINDOWS (Dict): Dictionary defining various rolling lookback windows for features.
        DL_PARAMS (Dict): Hyperparameters specific to the Deep Learning Attention model.
    """
    
    # --------------------------------------------------------------------------
    # 2.1 File System Paths
    # --------------------------------------------------------------------------
    # Using pathlib.Path for OS-agnostic path handling
    BASE_DIR = Path.cwd()
    DATA_DIR = BASE_DIR / 'Internal data'
    OUTPUT_DIR = BASE_DIR / 'Output_v2'
    LOG_DIR = OUTPUT_DIR / 'logs'

    # --------------------------------------------------------------------------
    # 2.2 Global Reproducibility
    # --------------------------------------------------------------------------
    SEED = 42

    # --------------------------------------------------------------------------
    # 2.3 Business Logic & Feature Engineering Parameters
    # --------------------------------------------------------------------------
    # Feature Engineering Windows (Weeks)
    # Different features require different historical contexts.
    WINDOWS = {
        'sales_history': 12,      # Look back 12 weeks for sales trends
        'supply_gap': 4,          # Look back 4 weeks for immediate supply gaps
        'rcv_trend': 8,           # Look back 8 weeks for receiving stability
        'po_horizon': 4,          # Look forward 4 weeks for purchase orders
        'forecast_horizon': 4     # Prediction target window (Next 4 weeks)
    }

    # --------------------------------------------------------------------------
    # 2.4 Model Hyperparameters
    # --------------------------------------------------------------------------
    # Deep Learning (Attention Mechanism) Configuration
    DL_PARAMS = {
        'batch_size': 32,
        'learning_rate': 0.001,
        'epochs': 20,
        'hidden_dim': 64,         # Size of hidden layers in DIN Encoder
        'dropout': 0.2,           # Regularization to prevent overfitting
        'attention_heads': 1      # Number of attention heads (1 for simple attention)
    }

    # Traditional ML Configuration
    ML_PARAMS = {
        'rf_estimators': 200,
        'xgb_learning_rate': 0.05,
        'cv_folds': 5             # Number of Cross-Validation folds
    }
    
    # --------------------------------------------------------------------------
    # 2.5 Infrastructure Settings
    # --------------------------------------------------------------------------
    API_RETRIES = 5
    API_BACKOFF = 0.5 # Factor for exponential backoff (0.5s, 1s, 2s...)

    @classmethod
    def setup_environment(cls) -> None:
        """
        Initializes the execution environment.

        This method performs three critical tasks:
        1. Creates necessary directories (Data, Output, Logs) if they don't exist.
        2. Configures the global logging format.
        3. Sets random seeds for Python, NumPy, and PyTorch to ensure reproducibility.
        
        Args:
            None
            
        Returns:
            None
            
        Raises:
            PermissionError: If the script lacks write permissions for directory creation.

        Example:
            >>> Config.setup_environment()
            [INFO] ShortageAI v2.0 Environment Initialized.
            [INFO] Device set to: cuda
        """
        # 1. Create Directories
        cls.DATA_DIR.mkdir(parents=True, exist_ok=True)
        cls.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        cls.LOG_DIR.mkdir(parents=True, exist_ok=True)

        # 2. Configure Logging
        # We write logs to both file and console
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - [%(levelname)s] - %(message)s',
            handlers=[
                logging.FileHandler(cls.LOG_DIR / f'run_{datetime.now():%Y%m%d_%H%M}.log'),
                logging.StreamHandler(sys.stdout)
            ]
        )
        logger = logging.getLogger(__name__)

        # 3. Set Random Seeds (Deterministic Behavior)
        random.seed(cls.SEED)
        np.random.seed(cls.SEED)
        torch.manual_seed(cls.SEED)
        
        # 4. Deep Learning Device Setup
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(cls.SEED)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
            device_name = "GPU (CUDA)"
        else:
            device_name = "CPU"
            
        # 5. Global Pandas Settings
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        pd.set_option('display.float_format', '{:.4f}'.format)
        warnings.filterwarnings('ignore')

        # Final Status Log
        logger.info("="*60)
        logger.info("SHORTAGE AI V2.0 - ENVIRONMENT INITIALIZED")
        logger.info("="*60)
        logger.info(f"System Time     : {datetime.now()}")
        logger.info(f"Torch Version   : {torch.__version__}")
        logger.info(f"Compute Device  : {device_name}")
        logger.info(f"Data Directory  : {cls.DATA_DIR}")
        logger.info(f"CatBoost Status : {'Active' if CATBOOST_ACTIVE else 'Inactive (Not Installed)'}")
        logger.info("-" * 60)

# ==============================================================================
# EXECUTION ENTRY POINT (Initialization)
# ==============================================================================
if __name__ == "__main__":
    # Initialize the environment immediately upon script load
    Config.setup_environment()

2026-01-17 18:15:50,793 - [INFO] - ============================================================
2026-01-17 18:15:50,795 - [INFO] - SHORTAGE AI V2.0 - ENVIRONMENT INITIALIZED
2026-01-17 18:15:50,796 - [INFO] - ============================================================
2026-01-17 18:15:50,799 - [INFO] - System Time     : 2026-01-17 18:15:50.798895
2026-01-17 18:15:50,805 - [INFO] - Torch Version   : 2.8.0
2026-01-17 18:15:50,810 - [INFO] - Compute Device  : CPU
2026-01-17 18:15:50,811 - [INFO] - Data Directory  : /Users/yuyangchen/Internal data
2026-01-17 18:15:50,811 - [INFO] - CatBoost Status : Active
2026-01-17 18:15:50,812 - [INFO] - ------------------------------------------------------------


In [51]:
# ==============================================================================
# SECTION 1: EXTERNAL DATA ACQUISITION & INGESTION
# ==============================================================================
# Description:
#   This module handles the extraction/transform/load (ETL) process for external
#   datasets. It implements a robust, 'stealth' scraping architecture designed
#   to bypass basic WAFs (Web Application Firewalls) and ensure data freshness
#   without redundant network calls.
# ==============================================================================
# Initialize Logger (Inherits settings from Section 0)
logger = logging.getLogger(__name__)

# ------------------------------------------------------------------------------
# COMPATIBILITY PATCH: URLLIB3 & PYTRENDS
# ------------------------------------------------------------------------------
# Newer versions of urllib3 (v2.0+) removed 'method_whitelist' in favor of 
# 'allowed_methods'. This patch intercepts the Retry initialization to 
# ensure backward compatibility for libraries like pytrends.
# ------------------------------------------------------------------------------
if not hasattr(retry.Retry, '_original_init'):
    retry.Retry._original_init = retry.Retry.__init__

def patched_retry_init(self, *args, **kwargs):
    if 'method_whitelist' in kwargs:
        kwargs['allowed_methods'] = kwargs.pop('method_whitelist')
    self._original_init(*args, **kwargs)

retry.Retry.__init__ = patched_retry_init


class ExternalDataIngestor:
    """
    Orchestrates the acquisition of external regulatory and market data.
    
    This class serves as the gateway to outside information. It encapsulates
    all network logic, retry mechanisms, and file system interactions required
    to maintain a fresh 'Data Lake' for the modeling pipeline.
    
    Attributes:
        data_dir (Path): The root directory for persistent data storage.
        session (requests.Session): A stateful HTTP session with retry logic.
    """

    # --------------------------------------------------------------------------
    # CONSTANTS
    # --------------------------------------------------------------------------
    
    # Health Canada Drug Product Database (DPD) Endpoint
    HC_BASE_URL = "https://www.canada.ca/content/dam/hc-sc/migration/hc-sc/dhp-mps/alt_formats/zip/prodpharma/databasdon/"
    
    # WHO ATC Level 1 Keywords for Market Sentiment Analysis (Google Trends)
    # These map broad therapeutic classes to consumer search terms.
    ATC_KEYWORDS = {
        'A': ['Stomach pain', 'Ozempic', 'Antacid', 'Digestive health'],
        'B': ['Blood thinner', 'Anticoagulant', 'Blood clot'],
        'C': ['Blood pressure', 'Heart medication', 'Beta blocker'],
        'D': ['Eczema cream', 'Skin rash', 'Cortisone'],
        'G': ['Birth control', 'Hormone replacement', 'Menopause'],
        'H': ['Thyroid medicine', 'Steroids', 'Prednisone'],
        'J': ['Antibiotics', 'Amoxicillin', 'Penicillin', 'Infection med'],
        'L': ['Chemotherapy', 'Cancer treatment', 'Immunotherapy'],
        'M': ['Muscle pain', 'Back pain', 'Arthritis relief'],
        'N': ['Pain killer', 'Tylenol', 'Advil', 'Ibuprofen'],
        'P': ['Lice treatment', 'Worm medicine'],
        'R': ['Cold medicine', 'Cough syrup', 'Flu medicine', 'Inhaler'],
        'S': ['Eye drops', 'Pink eye', 'Ear drops'],
        'V': ['Medical supply', 'First aid']
    }

    def __init__(self):
        """
        Initializes the Ingestor by binding it to the global configuration
        and establishing a robust network session.
        """
        self.data_dir = Config.DATA_DIR
        self.session = self._create_robust_session()

    def _create_robust_session(self) -> requests.Session:
        """
        Configures a requests.Session with industrial-grade resilience.

        It applies two layers of protection:
        1. Network Stability: Automatic retries on 500/502/503/504 errors.
        2. Anti-Blocking: Sets 'User-Agent' headers to mimic Chrome on Windows,
           preventing immediate rejection by server-side security rules.

        Returns:
            requests.Session: A configured session object ready for scraping.
        """
        session = requests.Session()
        
        # Configure Retry Strategy (Exponential Backoff)
        retries = retry.Retry(
            total=Config.API_RETRIES,
            backoff_factor=Config.API_BACKOFF,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["HEAD", "GET", "OPTIONS"]
        )
        
        # Mount the adapter to both HTTP and HTTPS protocols
        adapter = requests.adapters.HTTPAdapter(max_retries=retries)
        session.mount("https://", adapter)
        session.mount("http://", adapter)

        # Apply Browser Masquerading
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1"
        })
        
        return session

    def fetch_health_canada_data(self, update_frequency_days: int = 7) -> None:
        """
        Manages the download of Health Canada Regulatory Data (DPD).

        This function implements a 'Smart Cache' logic:
        1. Checks if the target file exists locally.
        2. Inspects the file's 'Last Modified' metadata.
        3. Only initiates a network request if the file is older than 'update_frequency_days'.
        
        Downloads are performed using stream=True to minimize memory footprint 
        when handling large ZIP archives (~50MB+).

        Args:
            update_frequency_days (int): The validity period (in days) for the local cache.
        """
        logger.info("[Ingestion] Starting Health Canada DPD Acquisition...")

        # Mapping of Zip archives to specific text files needed for analysis
        targets = {
            "allfiles.zip": ["drug.txt", "ingred.txt", "status.txt", "comp.txt"], 
            "allfiles_ia.zip": ["drug_ia.txt", "comp_ia.txt", "status_ia.txt"]
        }

        for zip_name, files_to_extract in targets.items():
            primary_file = files_to_extract[0]
            primary_path = self.data_dir / primary_file
            
            # --- INTELLIGENT CACHE VERIFICATION ---
            should_download = True
            
            if primary_path.exists() and primary_path.stat().st_size > 1024:
                # Retrieve OS-level file modification time
                last_modified_ts = primary_path.stat().st_mtime
                last_modified_date = datetime.fromtimestamp(last_modified_ts)
                
                # Calculate Freshness
                file_age_days = (datetime.now() - last_modified_date).days
                
                if file_age_days < update_frequency_days:
                    logger.info(f"   ---> Cache Hit: {zip_name} is fresh ({file_age_days} days old). Skipping.")
                    should_download = False
                else:
                    logger.info(f"   ---> Cache Expired: {zip_name} is {file_age_days} days old. Refreshing...")
            else:
                logger.info(f"   ---> Missing Data: {zip_name} not found. Initiating download...")

            # --- EXECUTE DOWNLOAD & EXTRACTION ---
            if should_download:
                try:
                    url = self.HC_BASE_URL + zip_name
                    logger.info(f"   ---> Connecting to {url}...")
                    
                    # Stream download to avoid loading entire file into RAM
                    with self.session.get(url, stream=True, timeout=90) as r:
                        r.raise_for_status()
                        
                        # In-memory decompression
                        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                            for target_file in files_to_extract:
                                # Robust logic to find file regardless of internal folder structure
                                match = next((f for f in z.namelist() if f.lower().endswith(target_file.lower())), None)
                                
                                if match:
                                    with z.open(match) as source, open(self.data_dir / target_file, "wb") as dest:
                                        shutil.copyfileobj(source, dest)
                                    logger.info(f"        Extracted: {target_file}")
                                else:
                                    logger.warning(f"        Warning: {target_file} not found inside archive.")
                                    
                except Exception as e:
                    logger.error(f"   ---> Critical Error downloading {zip_name}: {e}")

    def fetch_google_trends(self, update_frequency_days: int = 7) -> None:
        """
        Acquires consumer search sentiment via Google Trends.

        This function prioritizes data accuracy and retrieval stability.
        It uses 'Jitter' to avoid rate-limits and validates data completeness.
        It also includes a COOL-DOWN loop for 429 errors.

        Logic:
        1. Check 'ATC_COMPOSITE_TRENDS.csv' cache freshness.
        2. If stale, instantiate 'pytrends' with specific timezone (US CST).
        3. Loop through ATC codes with randomized sleep intervals (Jitter).
        4. If 429 Error occurs, wait 60s+ and retry (do not skip).
        
        Args:
            update_frequency_days (int): Data freshness threshold.
        """
        output_file = self.data_dir / "ATC_COMPOSITE_TRENDS.csv"
        logger.info("[Ingestion] Starting Google Trends Acquisition...")

        # --- STEP 1: CACHE VALIDATION ---
        if output_file.exists():
            try:
                df_cache = pd.read_csv(output_file)
                if 'date' in df_cache.columns and not df_cache.empty:
                    last_date = pd.to_datetime(df_cache['date']).max()
                    days_diff = (pd.Timestamp.now() - last_date).days
                    
                    if days_diff < update_frequency_days:
                        logger.info(f"   ---> Cache Valid ({days_diff} days old). Skipping API.")
                        return
                    else:
                        logger.info(f"   ---> Cache Expired ({days_diff} days old). Refreshing...")
            except Exception as e:
                logger.warning(f"   ---> Cache corrupted ({e}). Forcing refresh.")

        # --- STEP 2: ROBUST API EXECUTION ---
        fetched_dfs = []
        api_success = False

        # Implicit check for pytrends availability
        if 'TrendReq' in globals() or 'TrendReq' in locals() or sys.modules.get('pytrends.request'):
             try:
                 from pytrends.request import TrendReq
                 pytrends_available = True
             except ImportError:
                 pytrends_available = False
        else:
             pytrends_available = False

        if pytrends_available:
            logger.info("   ---> Connecting to Google Trends API...")
            # Timezone 360 = US CST (Stable endpoint)
            pytrends = TrendReq(hl='en-US', tz=360, timeout=(10, 25), retries=2, backoff_factor=1)
            
            for idx, (atc_code, keywords) in enumerate(self.ATC_KEYWORDS.items()):
                success = False
                attempts = 0
                max_retries = 3 

                # --- 429 COOL-DOWN LOOP ---
                while not success and attempts < max_retries:
                    try:
                        attempts += 1
                        
                        # [JITTER MECHANISM]
                        # Randomly sleep between 10.0 and 20.0 seconds (Increased for safety)
                        sleep_time = random.uniform(10.0, 20.0)
                        logger.info(f"        Querying ATC-{atc_code} [{idx+1}/{len(self.ATC_KEYWORDS)}] (Waiting {sleep_time:.1f}s)...")
                        time.sleep(sleep_time)
                        
                        # Request 5-year history for Canada (geo='CA')
                        pytrends.build_payload(keywords, cat=0, timeframe='today 5-y', geo='CA')
                        data = pytrends.interest_over_time()
                        
                        if not data.empty:
                            # [ACCURACY CHECK: DROP PARTIAL DATA]
                            if 'isPartial' in data.columns:
                                data = data[data['isPartial'].astype(str) == 'False']
                                data = data.drop(columns=['isPartial'], errors='ignore')
                                
                            # Calculate Composite Index
                            data['COMPOSITE_INDEX'] = data.max(axis=1)
                            
                            # Format for storage
                            subset = data.reset_index()[['date', 'COMPOSITE_INDEX']]
                            subset['ATC_LEVEL1'] = atc_code
                            fetched_dfs.append(subset)
                        
                        success = True # Exit Loop
                        
                    except Exception as e:
                        error_msg = str(e)
                        # [AUTO-COOL-DOWN]: Handle 429 Errors specifically
                        if "429" in error_msg or "Too Many Requests" in error_msg:
                            wait_time = 60 + (attempts * 45) # 60s, 105s, 150s
                            logger.warning(f"         HIT RATE LIMIT (429) for ATC-{atc_code}. Cooling down for {wait_time}s...")
                            time.sleep(wait_time)
                        else:
                            logger.error(f"        Failed to fetch ATC-{atc_code}: {e}")
                            break # Break loop for other errors

            if fetched_dfs:
                api_success = True
                final_df = pd.concat(fetched_dfs)
                logger.info("   ---> Google Trends Data Successfully Acquired.")
            else:
                 logger.error("   ---> Critical: No data fetched from Google Trends API. Check connection.")

        else:
             logger.error("   ---> Pytrends library not available. Skipping Google Trends ingestion.")

        # --- STEP 3: STORAGE ---
        if api_success:
             # Ensure date format consistency
             final_df['date'] = pd.to_datetime(final_df['date'])
             # Add YEAR_WEEK key for downstream merging (e.g., 202401)
             final_df['YEAR_WEEK'] = final_df['date'].dt.strftime('%Y%U').astype(int)
             
             final_df.to_csv(output_file, index=False)
             logger.info(f"   ---> Market Sentiment Data saved to {output_file}")
        else:
             logger.warning("   ---> Market Sentiment Data NOT updated due to API failure.")

# ==============================================================================
# EXECUTION ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    # Instantiate the Ingestor
    ingestor = ExternalDataIngestor()
    
    # Run Pipelines with 7-Day Freshness Enforcement
    # To force re-download, change to update_frequency_days=0
    ingestor.fetch_health_canada_data(update_frequency_days=7)
    ingestor.fetch_google_trends(update_frequency_days=7)
    
    print("✅ Section 1 Complete: External Data Ingested.")

2026-01-16 16:57:34,865 - [INFO] - [Ingestion] Starting Health Canada DPD Acquisition...
2026-01-16 16:57:34,869 - [INFO] -    ---> Cache Hit: allfiles.zip is fresh (2 days old). Skipping.
2026-01-16 16:57:34,871 - [INFO] -    ---> Cache Hit: allfiles_ia.zip is fresh (2 days old). Skipping.
2026-01-16 16:57:34,873 - [INFO] - [Ingestion] Starting Google Trends Acquisition...
2026-01-16 16:57:34,949 - [INFO] -    ---> Cache Valid (5 days old). Skipping API.
✅ Section 1 Complete: External Data Ingested.


In [52]:
# ==============================================================================
# SECTION 2: DATA LOADING & HIERARCHICAL PRE-PROCESSING
# ==============================================================================
# Description:
#   Primary ETL engine acting as the single source of truth.
#   Responsibilities:
#   1. Ingest raw internal data (Item, Supply, Sales, Target) and external regulatory data (DPD).
#   2. Standardize critical join keys (DIN, ITEM_NUM) across ALL datasets.
#   3. Resolve entity attributes (Molecules, ATC Classes, Manufacturers).
# ==============================================================================

# NLP Library Import
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    NLP_AVAILABLE = True
except ImportError:
    NLP_AVAILABLE = False

# Initialize Logger
logger = logging.getLogger(__name__)

# ------------------------------------------------------------------------------
# CLASS 1: TEXT NORMALIZER
# ------------------------------------------------------------------------------
class TextNormalizer:
    """
    Static utility class for parsing free-text descriptions into a conservative
    bag-of-words style molecule string.

    Important:
      - This parser is a FALLBACK only (used when internal molecule is missing).
      - Do NOT drop salt/counter-ion tokens (SODIUM, CHLORIDE, CALCIUM, etc.).
      - Preserve concentration tokens like 50% / 0.9%.
    """

    # Packaging / form / vendor noise only (do NOT include chemical tokens here)
    NOISE_WORDS = {
        'BAG', 'VIAL', 'KIT', 'BOX', 'BTL', 'UNT', 'IU', 'PCT', 'MEQ', 'VOL',
        'ML', 'MG', 'G', 'L', 'MCG', 'KG',
        'TAB', 'CAP', 'SOL', 'SOLUTION', 'INJ', 'INJECTION', 'CREAM', 'OINTMENT',
        'SUSP', 'SYRUP', 'ELIXIR', 'SUPP', 'PATCH', 'AEROSOL', 'DROPS',
        'USP', 'STERILE', 'PRESERVATIVE-FREE', 'CONCENTRATE', 'IRRIGATION',
        'APO', 'TEVA', 'PMS', 'SDZ', 'MYLAN', 'SANDOZ', 'JAMP', 'MINT', 'RIVA', 'MAR',
        'PHS', 'REX', 'OPTION', 'ATOMA', 'EQUATE', 'TRULY', 'MEME', 'NOV',
        'DPLY', 'XST', 'CLD', 'BLST', 'STRP', 'MGUD', 'ASTD', 'RRX', 'DS', 'RG', 'SP'
    }

    ABBREVIATIONS = {
        'SOD': 'SODIUM', 'CHLOR': 'CHLORIDE', 'CALC': 'CALCIUM',
        'POT': 'POTASSIUM', 'MAG': 'MAGNESIUM', 'DEXT': 'DEXTROSE',
        'BICARB': 'BICARBONATE', 'ACET': 'ACETAMINOPHEN', 'IBUP': 'IBUPROFEN',
        'SALB': 'SALBUTAMOL', 'VIT': 'VITAMIN', 'MULTI': 'MULTIVITAMIN',
        'ISOP': 'ISOPROPYL', 'PEROX': 'PEROXIDE', 'ALC': 'ALCOHOL',
        'DIPHEN': 'DIPHENHYDRAMINE', 'VACC': 'VACCINE'
    }

    BRAND_MAP = {
        'TYLENOL': 'ACETAMINOPHEN', 'ADVIL': 'IBUPROFEN', 'MOTRIN': 'IBUPROFEN',
        'VOLTAREN': 'DICLOFENAC', 'ASPIRIN': 'ACETYLSALICYLIC ACID',
        'ANACIN': 'ACETYLSALICYLIC ACID|CAFFEINE',
        'PAXLOVID': 'NIRMATRELVIR|RITONAVIR'
    }

    SUPPLY_KEYWORDS = {
        'SYR', 'SYRINGE', 'NEEDLE', 'LANCET', 'STRIP', 'TEST',
        'METER', 'MONITOR', 'WIPE', 'SWAB', 'GAUZE', 'BANDAGE',
        'DEVICE', 'SPACER', 'AEROCHAMBER'
    }

    @staticmethod
    def parse(description: str) -> tuple:
        """
        Parses a raw product description into a standardized molecule string.

        Args:
            description (str): Raw text description.

        Returns:
            tuple: (parsed_molecule_string, atc_flag_if_supply)
        """
        if not isinstance(description, str) or not description:
            return np.nan, None

        desc = description.upper()

        # Supply detection (non-drug items)
        for kw in TextNormalizer.SUPPLY_KEYWORDS:
            if kw in desc:
                return "SUPPLY", 'V'

        # Brand-to-generic fallback (coarse mapping)
        for brand, generic in TextNormalizer.BRAND_MAP.items():
            if brand in desc:
                return generic, None

        # Tokenization normalization (parser only)
        desc = re.sub(r'\s(W/|WITH)\s', '|', desc)
        desc = re.sub(r'[+/&]', '|', desc)

        raw_tokens = desc.split('|')
        clean_tokens = []

        for token in raw_tokens:
            token = token.strip()
            if not token:
                continue

            words = []
            parts = token.split()
            i = 0

            while i < len(parts):
                w = parts[i].strip()

                # Abbreviation expansion
                if w in TextNormalizer.ABBREVIATIONS:
                    w = TextNormalizer.ABBREVIATIONS[w]

                # Merge "50 %", "0.9 %"
                if i + 1 < len(parts) and parts[i + 1].strip() == '%':
                    num = w.replace('.', '', 1)
                    if num.isdigit():
                        w = f"{w}%"
                        i += 2
                    else:
                        i += 1
                else:
                    i += 1

                # Remove packaging/form/vendor noise
                if w in TextNormalizer.NOISE_WORDS:
                    continue

                # Drop pure numbers only, KEEP percent tokens like 50%
                if w.replace('.', '', 1).isdigit():
                    continue

                words.append(w)

            if words:
                clean_tokens.append(' '.join(words))

        if not clean_tokens:
            return np.nan, None

        return '|'.join(sorted(set(clean_tokens))), None


# ------------------------------------------------------------------------------
# CLASS 2: AI SMART MATCHER
# ------------------------------------------------------------------------------
class SmartMatcher:
    """
    TF-IDF + cosine similarity matcher used to map noisy molecule strings
    to a canonical DPD-derived molecule vocabulary.
    """

    def __init__(self, known_molecules: list, similarity_threshold: float = 0.55):
        self.threshold = similarity_threshold
        self.known_molecules = [m for m in known_molecules if isinstance(m, str) and len(m) > 0]
        self.vectorizer = None
        self.molecule_vectors = None

        if NLP_AVAILABLE and self.known_molecules:
            self.vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
            self.molecule_vectors = self.vectorizer.fit_transform(self.known_molecules)
            logger.info(f"   [SmartMatcher] Trained on {len(self.known_molecules)} unique molecules.")

    def match_batch(self, series: pd.Series) -> pd.Series:
        if self.vectorizer is None or series is None or len(series) == 0:
            return pd.Series([np.nan] * (0 if series is None else len(series)),
                             index=None if series is None else series.index)

        X = self.vectorizer.transform(series.fillna('').astype(str))
        sim = cosine_similarity(X, self.molecule_vectors)

        out = []
        for row in sim:
            i = int(row.argmax())
            out.append(self.known_molecules[i] if row[i] >= self.threshold else np.nan)

        return pd.Series(out, index=series.index)


# ------------------------------------------------------------------------------
# CLASS 3: MAIN DATA PROCESSOR
# ------------------------------------------------------------------------------
class DataPreProcessor:
    """
    Orchestrates Section 2 ETL.
    """

    AHFS_TO_ATC_MAP = {
        '04': 'R', '08': 'J', '10': 'L', '12': 'N', '20': 'B', '24': 'C',
        '28': 'N', '40': 'A', '48': 'R', '52': 'S', '56': 'A', '68': 'G',
        '84': 'D', '88': 'A', '92': 'V'
    }

    KEYWORD_RULES = {
        'STATIN': 'C', 'PRIL': 'C', 'SARTAN': 'C', 'OLOL': 'C', 'DIPINE': 'C',
        'CILLIN': 'J', 'MYCIN': 'J', 'VIR': 'J', 'CEF': 'J', 'FLOXACIN': 'J',
        'ACETAMINOPHEN': 'N', 'ASPIRIN': 'N', 'BENZOCAINE': 'N', 'GABAPENTIN': 'N',
        'SERTRALINE': 'N', 'CITALOPRAM': 'N', 'FLUOXETINE': 'N', 'NICOTINE': 'N',
        'FENTANIL': 'N', 'FENTANYL': 'N', 'REMIFENTANIL': 'N',
        'PROFEN': 'M', 'COXIB': 'M', 'IBUPROFEN': 'M', 'CAPSAICIN': 'M',
        'GLIPTIN': 'A', 'INSULIN': 'A', 'PRAZOLE': 'A', 'METFORMIN': 'A',
        'CALCIUM': 'A', 'MAGNESIUM': 'A', 'ZINC': 'A', 'VITAMIN': 'A', 'OMEGA': 'A',
        'ALGIN': 'A', 'BICARBONATE': 'A', 'LECITHIN': 'A', 'TUMS': 'A',
        'TESTOSTERONE': 'G', 'ESTRADIOL': 'G', 'PROGESTERONE': 'G',
        'SODIUM': 'B', 'POTASSIUM': 'B', 'CHLORIDE': 'B',
        'HEPARIN': 'B', 'WARFARIN': 'B', 'ALBUMIN': 'B',
        'SALBUTAMOL': 'R', 'IPRATROPIUM': 'R', 'FLUTICASONE': 'R',
        'FORMOTEROL': 'R', 'MOMETASONE': 'R', 'DIPHENHYDRAMINE': 'R',
        'TIMOLOL': 'S', 'DORZOLAMIDE': 'S', 'LATANOPROST': 'S',
        'SOAP': 'D', 'WASH': 'D', 'SHAMPOO': 'D', 'LOTION': 'D',
        'CREAM': 'D', 'PEROXIDE': 'D', 'ALCOHOL': 'D',
        'WATER': 'V', 'CONTRAST': 'V', 'SUPPLY': 'V'
    }

    def __init__(self):
        self.data_dir = Config.DATA_DIR

    def load_dataset(self, filename: str) -> pd.DataFrame:
        path = self.data_dir / filename
        if not path.exists():
            logger.warning(f"File {filename} not found.")
            return pd.DataFrame()

        logger.info(f"Loading {filename}...")
        try:
            return pd.read_csv(path, low_memory=False)
        except Exception as e:
            logger.error(f"Error reading {filename}: {e}")
            return pd.DataFrame()

    @staticmethod
    def normalize_key(series: pd.Series, width: int = 8) -> pd.Series:
        x = pd.to_numeric(series, errors='coerce').astype('Int64')
        return x.astype('string').str.zfill(width)

    @staticmethod
    def yearweek_to_monday(year_week: pd.Series) -> pd.Series:
        yw = pd.to_numeric(year_week, errors='coerce').astype('Int64')
        year = (yw // 100).astype('Int64')
        week = (yw % 100).astype('Int64')

        def _iso_monday(y, w):
            if pd.isna(y) or pd.isna(w):
                return pd.NaT
            try:
                return datetime.fromisocalendar(int(y), int(w), 1).date()
            except Exception:
                return pd.NaT

        return pd.to_datetime([_iso_monday(y, w) for y, w in zip(year, week)])

    @staticmethod
    def collapse_dpd_ingredient_set(dpd_str: str) -> str:
        if not isinstance(dpd_str, str) or dpd_str.strip() == "":
            return pd.NA

        tokens = [t.strip() for t in dpd_str.split('|') if t.strip()]
        if len(tokens) <= 1:
            return tokens[0] if tokens else pd.NA

        joined = " ".join(tokens).upper()

        if any(x in joined for x in ['VITAMIN', 'BIOTIN', 'FOLIC', 'FOLATE']):
            if any(x in joined for x in ['IRON', 'CALCIUM', 'ZINC', 'MAGNESIUM', 'MINERAL']):
                return 'MULTIVIT_MINERALS'
            return 'MULTIVITAMIN'

        if any(x in joined for x in ['SODIUM', 'POTASSIUM', 'CHLORIDE']):
            return 'ELECTROLYTE_COMBINATION'

        return 'COMBINATION_PRODUCT'

    def process_dpd_data(self):
        logger.info("Processing DPD Files & Building Knowledge Base...")

        cols_drug = ['DRUG_CODE', 'PROD_CATEG', 'CLASS', 'DIN', 'BRAND_NAME', 'DESCRIPTOR',
                    'PEDIATRIC_FLAG', 'ACCESSION_NUMBER', 'NUMBER_OF_AIS', 'LAST_UPDATE_DATE',
                    'AI_GROUP_NO', 'CLASS_F', 'BRAND_NAME_F', 'DESCRIPTOR_F']
        cols_ther = ['DRUG_CODE', 'TC_ATC_NUMBER', 'TC_ATC', 'TC_ATC_F',
                    'TC_AHFS_NUMBER', 'TC_AHFS', 'TC_AHFS_F']
        cols_ingred = ['DRUG_CODE', 'ACTIVE_INGREDIENT_CODE', 'INGREDIENT', 'INGREDIENT_SUPPLIED_IND',
                      'STRENGTH', 'STRENGTH_UNIT', 'STRENGTH_TYPE', 'DOSAGE_VALUE', 'BASE', 'YESNO',
                      'NOTES', 'INGREDIENT_F', 'STRENGTH_UNIT_F', 'STRENGTH_TYPE_F', 'DOSAGE_VALUE_F']
        cols_comp = ['DRUG_CODE', 'MFR_CODE', 'COMPANY_CODE', 'COMPANY_NAME', 'COMPANY_TYPE',
                    'ADDRESS_MAILING_FLAG', 'BILLING_FLAG', 'FINANCE_FLAG', 'TE_FLAG']

        def ingest(f1, f2, cols):
            dfs = []
            for f in [f1, f2]:
                p = self.data_dir / f
                if p.exists():
                    try:
                        dfs.append(pd.read_csv(p, names=cols, header=None, quotechar='"', encoding='latin1'))
                    except Exception:
                        continue
            return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

        df_drug = ingest("drug.txt", "drug_ia.txt", cols_drug)
        df_ther = ingest("ther.txt", "ther_ia.txt", cols_ther)
        df_ingred = ingest("ingred.txt", "ingred_ia.txt", cols_ingred)
        df_comp = ingest("comp.txt", "comp_ia.txt", cols_comp)

        if df_drug.empty:
            return pd.DataFrame(), {}, []

        for df in [df_drug, df_ther, df_ingred, df_comp]:
            if not df.empty and 'DRUG_CODE' in df.columns:
                df['DRUG_CODE'] = pd.to_numeric(df['DRUG_CODE'], errors='coerce').fillna(0).astype(int)

        df_drug['DIN_KEY'] = self.normalize_key(df_drug['DIN'], width=8)
        df_base = df_drug[['DRUG_CODE', 'DIN_KEY']].copy()

        if not df_ingred.empty and 'INGREDIENT' in df_ingred.columns:
            df_ingred['INGREDIENT'] = df_ingred['INGREDIENT'].astype(str).str.upper().str.strip()
            mol_map = (
                df_ingred.groupby('DRUG_CODE')['INGREDIENT']
                         .apply(lambda x: '|'.join(sorted(set([t for t in x if isinstance(t, str) and t.strip()]))))
                         .reset_index(name='DPD_MOLECULE')
            )
            df_base = df_base.merge(mol_map, on='DRUG_CODE', how='left')
        else:
            df_base['DPD_MOLECULE'] = pd.NA

        if not df_ther.empty and 'TC_ATC_NUMBER' in df_ther.columns:
            df_ther['TC_ATC_NUMBER'] = df_ther['TC_ATC_NUMBER'].astype(str).str.strip().str.upper()
            df_ther['TC_AHFS_NUMBER'] = df_ther['TC_AHFS_NUMBER'].astype(str).str.strip().str.upper()

            ther_map = (
                df_ther.groupby('DRUG_CODE')[['TC_ATC_NUMBER', 'TC_AHFS_NUMBER']]
                       .first()
                       .reset_index()
            )
            ther_map['DPD_ATC_L1'] = ther_map['TC_ATC_NUMBER'].str[0]
            ther_map['AHFS_CLASS'] = ther_map['TC_AHFS_NUMBER'].str[:2]
            ther_map['ATC_LEVEL3'] = ther_map['TC_ATC_NUMBER'].str[:4]
            df_base = df_base.merge(
                ther_map[['DRUG_CODE', 'DPD_ATC_L1', 'AHFS_CLASS', 'ATC_LEVEL3']],
                on='DRUG_CODE', how='left'
            )
        else:
            df_base['DPD_ATC_L1'] = pd.NA
            df_base['AHFS_CLASS'] = pd.NA
            df_base['ATC_LEVEL3'] = pd.NA

        if not df_comp.empty and 'COMPANY_NAME' in df_comp.columns:
            comp_map = df_comp.groupby('DRUG_CODE')['COMPANY_NAME'].first().reset_index(name='MFR_NAME_DPD')
            df_base = df_base.merge(comp_map, on='DRUG_CODE', how='left')
        else:
            df_base['MFR_NAME_DPD'] = 'UNKNOWN'

        def _mode_or_nan(s):
            s = s.dropna()
            if len(s) == 0:
                return np.nan
            return s.value_counts().index[0]

        df_dpd_din = (
            df_base.groupby('DIN_KEY')
                   .agg({
                       'DPD_MOLECULE': _mode_or_nan,
                       'DPD_ATC_L1': _mode_or_nan,
                       'AHFS_CLASS': _mode_or_nan,
                       'ATC_LEVEL3': _mode_or_nan,
                       'MFR_NAME_DPD': _mode_or_nan
                   })
                   .reset_index()
        )

        tmp = df_dpd_din.copy()
        tmp['DPD_MOLECULE_COLLAPSED'] = tmp['DPD_MOLECULE'].apply(self.collapse_dpd_ingredient_set)
        kb_df = tmp.dropna(subset=['DPD_MOLECULE_COLLAPSED', 'DPD_ATC_L1'])

        knowledge_base = (
            kb_df.groupby('DPD_MOLECULE_COLLAPSED')['DPD_ATC_L1']
                 .agg(lambda x: x.mode().iat[0] if len(x.mode()) > 0 else x.dropna().iat[0])
                 .to_dict()
        )

        known_molecules = sorted([m for m in kb_df['DPD_MOLECULE_COLLAPSED'].dropna().unique().tolist()
                                  if isinstance(m, str) and len(m) > 0])

        return df_dpd_din, knowledge_base, known_molecules

    def load_google_trends(self) -> pd.DataFrame:
        path = self.data_dir / "ATC_COMPOSITE_TRENDS.csv"
        if not path.exists():
            logger.warning("Google Trends file ATC_COMPOSITE_TRENDS.csv not found; df_trends will be empty.")
            return pd.DataFrame()

        try:
            df = pd.read_csv(path, low_memory=False)
        except Exception as e:
            logger.error(f"Error reading ATC_COMPOSITE_TRENDS.csv: {e}")
            return pd.DataFrame()

        if df.empty:
            return df

        if 'ATC_LEVEL1' not in df.columns:
            logger.warning("Google Trends file missing ATC_LEVEL1; df_trends will be empty.")
            return pd.DataFrame()

        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df['date_monday'] = df['date'] - pd.to_timedelta(df['date'].dt.weekday, unit='D')
        elif 'YEAR_WEEK' in df.columns:
            df['date_monday'] = self.yearweek_to_monday(df['YEAR_WEEK'])
        else:
            logger.warning("Google Trends file missing both date and YEAR_WEEK; df_trends will be empty.")
            return pd.DataFrame()

        if 'COMPOSITE_INDEX' not in df.columns:
            logger.warning("Google Trends file missing COMPOSITE_INDEX; df_trends will be empty.")
            return pd.DataFrame()

        df = df[['ATC_LEVEL1', 'date_monday', 'COMPOSITE_INDEX']].copy()
        df['ATC_LEVEL1'] = df['ATC_LEVEL1'].astype(str).str.strip().str.upper()
        df = df.dropna(subset=['ATC_LEVEL1', 'date_monday'])
        return df

    def run_pipeline(self):
        logger.info("Executing Section 2 Pipeline (Standardization Engine)...")

        df_item   = self.load_dataset("promitto_item_info_produc_descriptions.csv")
        df_supply = self.load_dataset("promitto_actual_drug_shortages.csv")
        df_sales  = self.load_dataset("promitto_history_product_level_sales_trend_past_year.csv")
        df_target = self.load_dataset("promitto_canada_drug_shortage_output.csv")
        df_raw    = self.load_dataset("Promitto_canada_drug_shortages_raw.csv")
        df_trends = self.load_google_trends()

        if 'DIN_NUM' in df_item.columns:
            df_item['DIN_NUM_CLEAN'] = pd.to_numeric(df_item['DIN_NUM'], errors='coerce')
            df_item = df_item.dropna(subset=['DIN_NUM_CLEAN']).copy()
            df_item['DIN_KEY'] = self.normalize_key(df_item['DIN_NUM_CLEAN'], width=8)
        else:
            df_item['DIN_KEY'] = pd.NA

        if 'ITEM_NUM' in df_item.columns:
            df_item['ITEM_NUM_CLEAN'] = pd.to_numeric(
                df_item['ITEM_NUM'].astype(str).str.replace(r'^I', '', regex=True),
                errors='coerce'
            )

        target_din_col = next((c for c in df_target.columns if 'DIN' in str(c).upper()), None)
        if target_din_col:
            df_target['DIN_KEY'] = self.normalize_key(df_target[target_din_col], width=8)
        else:
            logger.warning("No DIN column found in Target File. Merging will fail.")
            df_target['DIN_KEY'] = pd.Series([pd.NA] * len(df_target), dtype="string")

        if 'ITEM_NUM' in df_supply.columns:
            df_supply['ITEM_NUM_CLEAN'] = pd.to_numeric(
                df_supply['ITEM_NUM'].astype(str).str.replace(r'^I', '', regex=True),
                errors='coerce'
            )

        if ('ITEM_NUM_CLEAN' in df_supply.columns and
            'ITEM_NUM_CLEAN' in df_item.columns and
            'DIN_KEY' in df_item.columns):

            pre_rows = len(df_supply)
            df_map_item_to_din = df_item[['ITEM_NUM_CLEAN', 'DIN_KEY']].drop_duplicates('ITEM_NUM_CLEAN').copy()

            df_supply = df_supply.merge(
                df_map_item_to_din,
                on='ITEM_NUM_CLEAN',
                how='left',
                validate='many_to_one'
            )

            post_rows = len(df_supply)
            if pre_rows != post_rows:
                raise ValueError(f"Row count changed after supply->item merge: {pre_rows} -> {post_rows}")
        else:
            logger.warning("Supply->Item DIN mapping skipped (missing ITEM_NUM_CLEAN or DIN_KEY).")

        if 'YEAR_WEEK' in df_supply.columns:
            df_supply['date_monday'] = self.yearweek_to_monday(df_supply['YEAR_WEEK'])
        else:
            logger.warning("Supply file missing YEAR_WEEK; date_monday will be unavailable.")

        if not df_raw.empty:
            raw_din_col = "DRUG_DIN" if "DRUG_DIN" in df_raw.columns else ("DIN" if "DIN" in df_raw.columns else None)
            if raw_din_col is not None:
                df_raw["DIN_KEY"] = self.normalize_key(df_raw[raw_din_col], width=8)
            else:
                logger.warning("Raw file found but has no DIN/DRUG_DIN column; DIN_KEY cannot be standardized.")
        else:
            logger.warning("Raw file is empty or missing; df_raw will be empty.")

        df_dpd_din, knowledge_base, known_molecules = self.process_dpd_data()
        if not df_dpd_din.empty and 'DIN_KEY' in df_item.columns:
            df_master = df_item.merge(df_dpd_din, on='DIN_KEY', how='left')
        else:
            df_master = df_item.copy()
            df_master['DPD_MOLECULE'] = pd.NA
            df_master['DPD_ATC_L1'] = pd.NA
            df_master['AHFS_CLASS'] = pd.NA
            df_master['ATC_LEVEL3'] = pd.NA
            df_master['MFR_NAME_DPD'] = pd.NA

        df_master['DPD_MOLECULE_COLLAPSED'] = df_master['DPD_MOLECULE'].apply(self.collapse_dpd_ingredient_set)

        logger.info("Running Advanced Molecule Resolution...")

        desc_col = 'ITEM_EN_DESC' if 'ITEM_EN_DESC' in df_master.columns else ('SHORT_EN_DESC' if 'SHORT_EN_DESC' in df_master.columns else None)
        if desc_col is not None:
            parsed_results = df_master[desc_col].apply(TextNormalizer.parse)
            df_master['PARSED_MOLECULE'] = parsed_results.apply(lambda x: x[0])
            df_master['PARSED_ATC_FLAG'] = parsed_results.apply(lambda x: x[1])
        else:
            df_master['PARSED_MOLECULE'] = pd.NA
            df_master['PARSED_ATC_FLAG'] = pd.NA

        if 'MOLECULE_NM' not in df_master.columns:
            df_master['MOLECULE_NM'] = pd.NA

        df_master['MOLECULE_NM_STD'] = (
            df_master['MOLECULE_NM']
            .astype('string')
            .str.strip()
            .str.upper()
        )
        df_master.loc[df_master['MOLECULE_NM_STD'].isin(['', 'UNKNOWN', '<NA>']), 'MOLECULE_NM_STD'] = pd.NA

        din_molecule_mode = (
            df_master.dropna(subset=['DIN_KEY', 'MOLECULE_NM_STD'])
                     .groupby('DIN_KEY')['MOLECULE_NM_STD']
                     .agg(lambda s: s.value_counts().index[0])
        )
        df_master['MOLECULE_NM_CANON'] = df_master['DIN_KEY'].map(din_molecule_mode)
        df_master['MOLECULE_NM_FILLED'] = df_master['MOLECULE_NM_STD'].fillna(df_master['MOLECULE_NM_CANON'])

        df_master['FINAL_MOLECULE'] = (
            df_master['MOLECULE_NM_FILLED']
                .fillna(df_master['DPD_MOLECULE_COLLAPSED'])
                .fillna(df_master['PARSED_MOLECULE'])
                .fillna('UNKNOWN')
                .astype(str)
                .str.upper()
        )

        logger.info("Resolving ATC Classifications...")

        df_master['ATC_LEVEL1'] = df_master.get('DPD_ATC_L1')
        df_master['DATA_SOURCE'] = np.where(df_master['ATC_LEVEL1'].notna(), 'DPD_OFFICIAL', 'UNMAPPED')

        mask_supply = df_master['ATC_LEVEL1'].isna() & df_master['PARSED_ATC_FLAG'].notna()
        df_master.loc[mask_supply, 'ATC_LEVEL1'] = df_master.loc[mask_supply, 'PARSED_ATC_FLAG']
        df_master.loc[mask_supply, 'DATA_SOURCE'] = 'SUPPLY_DETECTION'

        mask_kb = df_master['ATC_LEVEL1'].isna() & (df_master['FINAL_MOLECULE'] != 'UNKNOWN')
        if mask_kb.any():
            inferred_kb = df_master.loc[mask_kb, 'FINAL_MOLECULE'].map(knowledge_base)
            update_mask = mask_kb & inferred_kb.notna()
            df_master.loc[update_mask, 'ATC_LEVEL1'] = inferred_kb
            df_master.loc[update_mask, 'DATA_SOURCE'] = 'KNOWLEDGE_BASE_MATCH'

        if NLP_AVAILABLE:
            mask_nlp = df_master['ATC_LEVEL1'].isna() & (df_master['FINAL_MOLECULE'] != 'UNKNOWN')
            if mask_nlp.any() and len(known_molecules) > 0:
                logger.info(f"   [AI] Attempting TF-IDF matching for {mask_nlp.sum()} items...")
                matcher = SmartMatcher(known_molecules, similarity_threshold=0.55)
                best_matches = matcher.match_batch(df_master.loc[mask_nlp, 'FINAL_MOLECULE'])
                matched_atc = best_matches.map(knowledge_base)
                update_mask = mask_nlp & matched_atc.notna()
                df_master.loc[update_mask, 'ATC_LEVEL1'] = matched_atc
                df_master.loc[update_mask, 'DATA_SOURCE'] = 'AI_TFIDF_MATCH'
                logger.info(f"   [AI] Successfully recovered {int(update_mask.sum())} items.")

        if 'AHFS_CLASS' in df_master.columns:
            mask_ahfs = df_master['ATC_LEVEL1'].isna() & df_master['AHFS_CLASS'].notna()
            if mask_ahfs.any():
                mapped = df_master.loc[mask_ahfs, 'AHFS_CLASS'].map(self.AHFS_TO_ATC_MAP)
                df_master.loc[mask_ahfs, 'ATC_LEVEL1'] = mapped
                df_master.loc[mask_ahfs & df_master['ATC_LEVEL1'].notna(), 'DATA_SOURCE'] = 'AHFS_CONVERSION'

        mask_kw = df_master['ATC_LEVEL1'].isna() & (df_master['FINAL_MOLECULE'] != 'UNKNOWN')
        if mask_kw.any():
            def infer_atc(mol_str):
                for token in str(mol_str).split('|'):
                    for kw, code in self.KEYWORD_RULES.items():
                        if kw in token:
                            return code
                return np.nan

            inferred = df_master.loc[mask_kw, 'FINAL_MOLECULE'].apply(infer_atc)
            update_mask = mask_kw & inferred.notna()
            df_master.loc[update_mask, 'ATC_LEVEL1'] = inferred
            df_master.loc[update_mask, 'DATA_SOURCE'] = 'KEYWORD_INFERRED'

        df_master['ATC_LEVEL1'] = df_master['ATC_LEVEL1'].fillna('X')

        if 'MY_DATE' in df_target.columns:
            df_target['MY_DATE'] = pd.to_datetime(df_target['MY_DATE'], errors='coerce')
            df_target['date_monday'] = df_target['MY_DATE'] - pd.to_timedelta(df_target['MY_DATE'].dt.weekday, unit='D')

        if 'CAL_YEAR' in df_sales.columns and 'CAL_MONTH' in df_sales.columns:
            df_sales['date_month'] = pd.to_datetime(
                df_sales['CAL_YEAR'].astype(str) + '-' + df_sales['CAL_MONTH'].astype(str) + '-01',
                errors='coerce'
            )

        return df_master, df_supply, df_sales, df_target, df_raw, df_trends


# ==============================================================================
# EXECUTION ENTRY POINT
# ==============================================================================
processor = DataPreProcessor()
df_model_scope, df_supply, df_sales, df_target, df_raw, df_trends = processor.run_pipeline()

total = len(df_model_scope)
unmapped = len(df_model_scope[df_model_scope['ATC_LEVEL1'] == 'X'])
coverage = (1 - unmapped / total) * 100 if total > 0 else 0.0

print("-" * 60)
print("SECTION 2 COMPLETE: MASTER UNIVERSE GENERATED")
print("-" * 60)
print(f"Total Unique Products : {total:,}")
print(f"Fully Mapped (ATC)    : {total - unmapped:,} ({coverage:.1f}%)")

print("\n[Unique Molecule Summary]")
unique_pairs = df_model_scope[['DIN_KEY', 'FINAL_MOLECULE']].dropna(subset=['DIN_KEY']).drop_duplicates()
print(f"Unique Products (DIN) : {unique_pairs['DIN_KEY'].nunique():,}")
print(f"Unique Molecules      : {unique_pairs['FINAL_MOLECULE'].nunique():,}")
print("\nTop 20 Molecules by Product Count")
print(unique_pairs['FINAL_MOLECULE'].value_counts().head(20))

print("\n[Source Attribution - Mapping Performance]")
if 'DATA_SOURCE' in df_model_scope.columns:
    print(df_model_scope['DATA_SOURCE'].value_counts())

print("-" * 60)


2026-01-16 16:57:36,017 - [INFO] - Executing Section 2 Pipeline (Standardization Engine)...
2026-01-16 16:57:36,019 - [INFO] - Loading promitto_item_info_produc_descriptions.csv...
2026-01-16 16:57:36,504 - [INFO] - Loading promitto_actual_drug_shortages.csv...
2026-01-16 16:57:36,514 - [INFO] - Loading promitto_history_product_level_sales_trend_past_year.csv...
2026-01-16 16:57:36,615 - [INFO] - Loading promitto_canada_drug_shortage_output.csv...
2026-01-16 16:57:36,623 - [INFO] - Loading Promitto_canada_drug_shortages_raw.csv...
2026-01-16 16:57:37,309 - [WARNING] - No DIN column found in Target File. Merging will fail.
2026-01-16 16:57:37,451 - [INFO] - Processing DPD Files & Building Knowledge Base...
2026-01-16 16:58:03,033 - [INFO] - Running Advanced Molecule Resolution...
2026-01-16 16:58:07,955 - [INFO] - Resolving ATC Classifications...
2026-01-16 16:58:07,996 - [INFO] -    [AI] Attempting TF-IDF matching for 4027 items...
2026-01-16 16:58:08,152 - [INFO] -    [SmartMatcher] T

In [53]:
tests = [
    # -----------------------------
    # 1) Percent handling (keep %)
    # -----------------------------
    "DEXTROSE 50 % IN WATER",
    "SODIUM CHLORIDE 0.9 %",
    "SODIUM CHLORIDE 3 %",
    "POTASSIUM CHLORIDE 20 %",
    "DEXTROSE 5 % IN WATER",
    "DEXTROSE 10 % IN WATER",
    "SODIUM CHLORIDE 0.45 %",
    "SODIUM CHLORIDE 0.225 %",

    # ------------------------------------------------
    # 2) Salt / counter-ion must be preserved (critical)
    # ------------------------------------------------
    "ATORVASTATIN CALCIUM",
    "ROSUVASTATIN CALCIUM",
    "DILTIAZEM HCL",
    "METOPROLOL TARTRATE",
    "AMLODIPINE BESYLATE",
    "LOSARTAN POTASSIUM",
    "DICLOFENAC SODIUM",
    "NAPROXEN SODIUM",
    "MORPHINE SULFATE",
    "GENTAMICIN SULFATE",
    "CLINDAMYCIN PHOSPHATE",
    "CEFTRIAXONE SODIUM",

    # ---------------------------------------------
    # 3) Combination / separators (/, +, &, WITH)
    # ---------------------------------------------
    "METHENAMINE/SOD BICARB/TART AC",
    "ACETAMINOPHEN/CAFFEINE",
    "ACETAMINOPHEN + CODEINE",
    "ACETAMINOPHEN & IBUPROFEN",
    "AMOXICILLIN WITH CLAVULANIC ACID",
    "SODIUM CHLORIDE W/ DEXTROSE 5 %",
    "DEXTROSE 5 % + SODIUM CHLORIDE 0.9 %",
    "NIRMATRELVIR/RITONAVIR",

    # ---------------------------------------------
    # 4) Brand mapping (BRAND_MAP)
    # ---------------------------------------------
    "ADVIL 200 MG TAB",
    "MOTRIN 400 MG",
    "TYLENOL EXTRA STRENGTH",
    "VOLTAREN GEL",
    "ASPIRIN 81 MG",
    "ANACIN TAB",
    "PAXLOVID",

    # ---------------------------------------------
    # 5) Supply detection (SUPPLY_KEYWORDS)
    # ---------------------------------------------
    "INSULIN SYRINGE 1 ML",
    "NEEDLE 25G",
    "ALCOHOL SWAB",
    "GLUCOSE TEST STRIP",
    "AEROCHAMBER SPACER",
    "BANDAGE ROLL GAUZE",

    # ---------------------------------------------
    # 6) Dosage form / packaging noise removal
    # ---------------------------------------------
    "IBUPROFEN 200 MG TAB",
    "AMOXICILLIN 250 MG CAPSULE",
    "SALBUTAMOL INHALER 100 MCG",
    "DIPHENHYDRAMINE SYRUP",
    "HYDROCORTISONE CREAM",
    "CIPROFLOXACIN SOLUTION",
    "EPINEPHRINE INJECTION",

    # ------------------------------------------------
    # 7) Edge patterns: parentheses / hyphens / commas
    # ------------------------------------------------
    "SODIUM CHLORIDE (0.9 %) INJECTION",
    "DEXTROSE (5 %) IN WATER",
    "SODIUM-CHLORIDE 0.9 %",
    "ACETAMINOPHEN-CAFFEINE",
    "IBUPROFEN, SODIUM",  # comma should remain as token separator only if you pre-clean; here parser keeps it in token

    # ------------------------------------------------
    # 8) Numeric-only tokens should drop (except %)
    # ------------------------------------------------
    "AMOXICILLIN 500 250",   # pure numbers should be dropped
    "VITAMIN D3 1000 IU",    # IU is noise; 1000 dropped; keep VITAMIN D3? (D3 stays if not purely numeric)
    "vitamin a palm/d3/b1/b2/b3/c"
    "flu vac 2019(6-24mos)/MF59C/PF"
]

for t in tests:
    print(f"{t} -> {TextNormalizer.parse(t)[0]}")


DEXTROSE 50 % IN WATER -> DEXTROSE 50% IN WATER
SODIUM CHLORIDE 0.9 % -> SODIUM CHLORIDE 0.9%
SODIUM CHLORIDE 3 % -> SODIUM CHLORIDE 3%
POTASSIUM CHLORIDE 20 % -> POTASSIUM CHLORIDE 20%
DEXTROSE 5 % IN WATER -> DEXTROSE 5% IN WATER
DEXTROSE 10 % IN WATER -> DEXTROSE 10% IN WATER
SODIUM CHLORIDE 0.45 % -> SODIUM CHLORIDE 0.45%
SODIUM CHLORIDE 0.225 % -> SODIUM CHLORIDE 0.225%
ATORVASTATIN CALCIUM -> ATORVASTATIN CALCIUM
ROSUVASTATIN CALCIUM -> ROSUVASTATIN CALCIUM
DILTIAZEM HCL -> DILTIAZEM HCL
METOPROLOL TARTRATE -> METOPROLOL TARTRATE
AMLODIPINE BESYLATE -> AMLODIPINE BESYLATE
LOSARTAN POTASSIUM -> LOSARTAN POTASSIUM
DICLOFENAC SODIUM -> DICLOFENAC SODIUM
NAPROXEN SODIUM -> NAPROXEN SODIUM
MORPHINE SULFATE -> MORPHINE SULFATE
GENTAMICIN SULFATE -> GENTAMICIN SULFATE
CLINDAMYCIN PHOSPHATE -> CLINDAMYCIN PHOSPHATE
CEFTRIAXONE SODIUM -> CEFTRIAXONE SODIUM
METHENAMINE/SOD BICARB/TART AC -> METHENAMINE|SODIUM BICARBONATE|TART AC
ACETAMINOPHEN/CAFFEINE -> ACETAMINOPHEN|CAFFEINE
ACETAMINOPHE

In [54]:
df_model_scope.head(50)

,ITEM_NUM,ITEM_EN_DESC,SHORT_EN_DESC,VENDOR_NUM,DIN_NUM,GEN_ITEM_IND,NWDA_CD,NWDA_EN_DESC,GEN_NUM,GEN_EN_DESC_1,THRP_CL_CD,THRP_CL_EN_DESC,ITEM_CL_GRP_CD,ITEM_CL_GRP_EN_SHORT_DESC,MBA_ST_CD,ITEM_TYP_CD,ITEM_TYP_EN_DESC,REFRG_IND,REC_CUR_IND,PHARMACEUTICAL_CD,NWDA_DETAIL_CD,INJECTABLE_ITM_CD,REFRG_EN_DESC,INJECTABLE_ITM_EN_DESC,ITEM_GRP_ID,AHFS_CD,NWDA_GRP_CD,NWDA_GRP_EN_DESC,PRECURSOR_INFO_CD,CNTRL_DRUG_CD,MOLECULE_NM,DIN_NUM_CLEAN,DIN_KEY,ITEM_NUM_CLEAN,DPD_MOLECULE,DPD_ATC_L1,AHFS_CLASS,ATC_LEVEL3,MFR_NAME_DPD,DPD_MOLECULE_COLLAPSED,PARSED_MOLECULE,PARSED_ATC_FLAG,MOLECULE_NM_STD,MOLECULE_NM_CANON,MOLECULE_NM_FILLED,FINAL_MOLECULE,ATC_LEVEL1,DATA_SOURCE
0,84,/ LEVETIRACETAM TB 250MG 100,LEVETIRACETAM,172587,2353342.0000,O,864,OTHER RX GENERIC,44632.0000,LEVETIRACETAM TABLET 250 MG ORAL,281292.0000,MISCELLANEOUS ANTICONVULSANTS,1,RX,D,R,REGULAR,NaN,Y,A,0.0000,NaN,NaN,NaN,224173,281292.0000,860.0000,PHARMACY RX GENERIC,NaN,NaN,levETIRAcetam,2353342.0000,02353342,84,LEVETIRACETAM,N,MI,N03A,NaN,LEVETIRACETAM,LEVETIRACETAM TB 250MG,None,LEVETIRACETAM,LEVETIRACETAM,LEVETIRACETAM,LEVETIRACETAM,N,DPD_OFFICIAL
1,87,/ LEVETIRACETAM TB 500MG 100,LEVETIRACETAM,172587,2353350.0000,O,864,OTHER RX GENERIC,44633.0000,LEVETIRACETAM TABLET 500 MG ORAL,281292.0000,MISCELLANEOUS ANTICONVULSANTS,1,RX,D,R,REGULAR,NaN,Y,A,0.0000,NaN,NaN,NaN,224176,281292.0000,860.0000,PHARMACY RX GENERIC,NaN,NaN,levETIRAcetam,2353350.0000,02353350,87,LEVETIRACETAM,N,MI,N03A,NaN,LEVETIRACETAM,LEVETIRACETAM TB 500MG,None,LEVETIRACETAM,LEVETIRACETAM,LEVETIRACETAM,LEVETIRACETAM,N,DPD_OFFICIAL
2,88,/ LEVETIRACETAM TB 750MG 100,LEVETIRACETAM,172587,2353369.0000,O,864,OTHER RX GENERIC,45652.0000,LEVETIRACETAM TABLET 750 MG ORAL,281292.0000,MISCELLANEOUS ANTICONVULSANTS,1,RX,D,R,REGULAR,NaN,Y,A,0.0000,NaN,NaN,NaN,224177,281292.0000,860.0000,PHARMACY RX GENERIC,NaN,NaN,levETIRAcetam,2353369.0000,02353369,88,LEVETIRACETAM,N,MI,N03A,NaN,LEVETIRACETAM,LEVETIRACETAM TB 750MG,None,LEVETIRACETAM,LEVETIRACETAM,LEVETIRACETAM,LEVETIRACETAM,N,DPD_OFFICIAL
3,90,/ LOVASTATIN TB 20MG 100,LOVASTATIN,172587,2353229.0000,O,864,OTHER RX GENERIC,6460.0000,LOVASTATIN TABLET 20 MG ORAL,240608.0000,HMG-COA REDUCTASE INHIBITORS,1,RX,D,R,REGULAR,NaN,Y,A,0.0000,NaN,NaN,NaN,224179,240608.0000,860.0000,PHARMACY RX GENERIC,NaN,NaN,lovastatin,2353229.0000,02353229,90,LOVASTATIN,C,HM,C10A,NaN,LOVASTATIN,LOVASTATIN TB 20MG,None,LOVASTATIN,LOVASTATIN,LOVASTATIN,LOVASTATIN,C,DPD_OFFICIAL
4,99,/ MELOXICAM TB 15MG 100,MELOXICAM,172587,2353156.0000,O,864,OTHER RX GENERIC,29157.0000,MELOXICAM TABLET 15 MG ORAL,280804.0000,OTHER NONSTEROIDAL ANTI-IMFLAMMATORY AGENTS,1,RX,F,R,REGULAR,NaN,Y,A,0.0000,NaN,NaN,NaN,224187,28080492.0000,860.0000,PHARMACY RX GENERIC,NaN,NaN,meloxicam,2353156.0000,02353156,99,MELOXICAM,M,OT,M01A,NaN,MELOXICAM,MELOXICAM TB 15MG,None,MELOXICAM,MELOXICAM,MELOXICAM,MELOXICAM,M,DPD_OFFICIAL
5,102,/ MELOXICAM TB 7.5MG 100,MELOXICAM,172587,2353148.0000,O,864,OTHER RX GENERIC,29156.0000,MELOXICAM TABLET 7.5 MG ORAL,280804.0000,OTHER NONSTEROIDAL ANTI-IMFLAMMATORY AGENTS,1,RX,F,R,REGULAR,NaN,Y,A,0.0000,NaN,NaN,NaN,224190,28080492.0000,860.0000,PHARMACY RX GENERIC,NaN,NaN,meloxicam,2353148.0000,02353148,102,MELOXICAM,M,OT,M01A,NaN,MELOXICAM,MELOXICAM TB 7.5MG,None,MELOXICAM,MELOXICAM,MELOXICAM,MELOXICAM,M,DPD_OFFICIAL
6,105,/ PROSH PL OIN DIMETH TB115G,PROSHIELD PLUS,38558,2256975.0000,N,651,SKINCARE - MEDICATIONS FOR SKIN DISEASE AND ACNE,59632.0000,DIMETHICONE OINT. (G) 1% TOPICAL,842412.0000,BASIC OINTMENTS AND PROTECTANTS,2,OTC,D,S,SEASONAL,NaN,Y,D,0.0000,NaN,NaN,NaN,224193,842412.0000,650.0000,SKIN CARE,NaN,NaN,dimethicone,2256975.0000,02256975,105,DIMETHICONE,D,BA,D02A,NaN,DIMETHICONE,PROSH PL OIN DIMETH TB115G,None,DIMETHICONE,DIMETHICONE,DIMETHICONE,DIMETHICONE,D,DPD_OFFICIAL
7,1073,/ STIEVA A GEL 0.05% 25G,STIEVA A,188201,641863.0000,N,854,OTHER RX NATIONAL BRAND,21108.0000,TRETINOIN GEL (GRAM) 0.05% TOPICAL,841600.0000,CELL STIMULANTS AND PROLIFERANTS,1,RX,F,R,REGULAR,NaN,Y,A,0.0000,N

In [80]:
# ==============================================================================
# EDA: Distribution of molecule token count per DIN
# ==============================================================================

def token_count(s):
    if not isinstance(s, str) or s.strip() == "":
        return 0
    return len([t for t in s.split('|') if t.strip()])

din_mol = (
    df_model_scope[['DIN_KEY', 'FINAL_MOLECULE']]
      .dropna(subset=['DIN_KEY'])
      .drop_duplicates()
)

din_mol['N_MOLECULE_TOKENS'] = din_mol['FINAL_MOLECULE'].apply(token_count)

print("\n=== EDA: DIN → #Molecule Tokens (Distribution) ===")
print(din_mol['N_MOLECULE_TOKENS'].value_counts().sort_index())

print("\n[Percent Distribution]")
pct = (
    din_mol['N_MOLECULE_TOKENS']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)
print(pct)

print("\n[Summary Stats]")
print(din_mol['N_MOLECULE_TOKENS'].describe())



=== EDA: DIN → #Molecule Tokens (Distribution) ===
N_MOLECULE_TOKENS
1    17601
2      259
3       40
4        6
5        2
Name: count, dtype: int64

[Percent Distribution]
N_MOLECULE_TOKENS
1   98.2900
2    1.4500
3    0.2200
4    0.0300
5    0.0100
Name: proportion, dtype: float64

[Summary Stats]
count   17908.0000
mean        1.0204
std         0.1667
min         1.0000
25%         1.0000
50%         1.0000
75%         1.0000
max         5.0000
Name: N_MOLECULE_TOKENS, dtype: float64


In [81]:
# ==============================================================================
# SECTION 3: FEATURE ENGINEERING
# ==============================================================================

logger = logging.getLogger(__name__)
logger.info("Executing Section 3 (Feature Engineering)...")

from tqdm.auto import tqdm


class ShortageLogic:
    """
    Centralized shortage definition logic.
    
    This class encapsulates the business logic for determining whether a DIN
    is in shortage for a given week. All shortage labels throughout the pipeline
    use this single implementation to ensure consistency.
    
    The current implementation uses fuzzy substring matching. When the client
    confirms the final shortage definition, update the is_shortage_week method
    and all dependent features will automatically inherit the change.
    """
    
    @staticmethod
    def is_shortage_week(final_molecule: str, shortage_molecules: list) -> int:
        """
        Determine if a DIN is in shortage for a given week.
        
        Args:
            final_molecule: The DIN's FINAL_MOLECULE value
            shortage_molecules: List of molecules in shortage this week
            
        Returns:
            1 if shortage, 0 otherwise
            
        Notes:
            Current implementation uses fuzzy substring matching. To update to
            exact matching or DIN-based logic, replace this method's body.
        """
        if not shortage_molecules:
            return 0
        fm = str(final_molecule).upper()
        if fm in ["", "UNKNOWN", "<NA>", "NAN"]:
            return 0
        for m in shortage_molecules:
            if m and (m in fm):
                return 1
        return 0


class SupplyGapLogic:
    """
    Centralized supply gap calculation logic.
    
    This class encapsulates the business logic for calculating supply gaps
    between expected purchase orders and target receive quantities. All supply
    gap features use this single implementation.
    
    To modify the gap calculation (e.g., to percentage-based), update the
    calculate_gap method and all dependent features will automatically inherit
    the change.
    """
    
    @staticmethod
    def calculate_gap(exp_po: float, target_rcv: float) -> float:
        """
        Calculate supply gap between expected PO and target receive.
        
        Args:
            exp_po: Expected purchase orders (4-week forward)
            target_rcv: Target receive quantity (4-week forward)
            
        Returns:
            Supply gap in units (negative indicates deficit)
            
        Notes:
            Current implementation: gap = exp_po - target_rcv
            To change to percentage: return (exp_po - target_rcv) / target_rcv
        """
        return exp_po - target_rcv
    
    @staticmethod
    def is_deficit(gap: float) -> int:
        """Binary indicator of deficit (gap < 0)."""
        return 1 if gap < 0 else 0


class FeatureConfig:
    """Configuration parameters for feature engineering pipeline."""
    
    WEEK_FREQ = "W-MON"
    HORIZONS = [1, 2, 3, 4]
    GAP_ROLL_WINDOWS = [4]
    TRENDS_ROLL_WINDOW = 4
    VENDOR_GROUPS = {
        1: "1_vendor",
        2: "2_vendors", 
        3: "3_vendors",
        4: "4_vendors",
        5: "5+_vendors"
    }
    VENDOR_CREDIT_LOOKBACK_WEEKS = 12
    WINSORIZE_PERCENTILE = 0.99


def _coerce_numeric(s: pd.Series) -> pd.Series:
    """Convert series to numeric, coercing errors to NaN."""
    return pd.to_numeric(s, errors="coerce")


def _to_monday(dt: pd.Series) -> pd.Series:
    """Convert datetime series to Monday (start of week)."""
    dt = pd.to_datetime(dt, errors="coerce")
    return dt - pd.to_timedelta(dt.dt.weekday, unit="D")


def _common_week_window(*dfs, date_col="date_monday"):
    """
    Determine common week window across multiple dataframes.
    
    Args:
        *dfs: Variable number of dataframes
        date_col: Name of date column to use
        
    Returns:
        Tuple of (max_start_date, min_end_date) or (None, None) if no overlap
    """
    mins, maxs = [], []
    for d in dfs:
        if isinstance(d, pd.DataFrame) and (not d.empty) and (date_col in d.columns):
            mins.append(pd.to_datetime(d[date_col], errors="coerce").min())
            maxs.append(pd.to_datetime(d[date_col], errors="coerce").max())
    if not mins or not maxs:
        return None, None
    return max(mins), min(maxs)


def _make_vendor_group(n_vendors: pd.Series, groups: dict) -> pd.Series:
    """
    Categorize vendor counts into discrete groups.
    
    Args:
        n_vendors: Series of vendor counts per DIN
        groups: Mapping of count thresholds to group labels
        
    Returns:
        Categorical series with vendor group labels
    """
    def _categorize(n):
        if pd.isna(n):
            return pd.NA
        n = int(n)
        if n <= 0:
            return pd.NA
        if n in groups:
            return groups[n]
        return groups[5]
    return n_vendors.apply(_categorize).astype("string")


class FeatureBuilder:
    """
    Production feature engineering pipeline for drug shortage prediction.
    
    This class implements a complete feature engineering workflow that transforms
    raw data sources into a modeling-ready DIN-week panel. The pipeline ensures
    temporal consistency, prevents data leakage, and maintains referential
    integrity across all transformations.
    
    Key Design Principles:
        - All features are lagged appropriately to prevent leakage
        - Business logic is centralized for easy modification
        - Statistical transformations ensure numerical stability
        - Documentation maintains production code standards
    
    Attributes:
        cfg: Configuration object containing pipeline parameters
        manifest: Dictionary mapping feature names to descriptions
        shortage_logic: Centralized shortage definition implementation
        supply_logic: Centralized supply gap calculation implementation
        vendor_din_map: Mapping of vendors to DINs for credit features
        global_trend_mean: Fallback value for unmapped ATC trends
    """
    
    def __init__(self, cfg: FeatureConfig):
        """
        Initialize feature builder.
        
        Args:
            cfg: FeatureConfig instance containing pipeline parameters
        """
        self.cfg = cfg
        self.manifest = {}
        self.shortage_logic = ShortageLogic()
        self.supply_logic = SupplyGapLogic()

    def build_din_master(self, df_model_scope: pd.DataFrame) -> pd.DataFrame:
        """
        Construct static DIN-level master attributes.
        
        Creates one row per DIN with time-invariant features including vendor
        diversity metrics, ATC classification, and therapeutic diversity indicators.
        These features capture structural characteristics of each drug product.
        
        Args:
            df_model_scope: Product universe dataframe from Section 2
            
        Returns:
            DataFrame with one row per DIN containing static attributes
            
        Raises:
            ValueError: If required columns are missing from input
        """
        logger.info("Section 3.1: Building DIN master attributes...")
        
        req = ["DIN_KEY", "ATC_LEVEL1", "ATC_LEVEL3", "FINAL_MOLECULE"]
        for c in req:
            if c not in df_model_scope.columns:
                raise ValueError(f"df_model_scope missing required column: {c}")
        
        base_cols = ["DIN_KEY", "ATC_LEVEL1", "ATC_LEVEL3", "FINAL_MOLECULE"]
        df_din = df_model_scope[base_cols].dropna(subset=["DIN_KEY"]).drop_duplicates("DIN_KEY").copy()
        
        if "VENDOR_NUM" in df_model_scope.columns:
            v = df_model_scope[["DIN_KEY", "VENDOR_NUM"]].copy()
            v["VENDOR_NUM"] = v["VENDOR_NUM"].astype("string").str.strip()
            v.loc[v["VENDOR_NUM"].isin(["", "UNKNOWN", "<NA>"]), "VENDOR_NUM"] = pd.NA
            
            n_vendors = v.dropna(subset=["DIN_KEY", "VENDOR_NUM"]).groupby("DIN_KEY")["VENDOR_NUM"].nunique()
            df_din["n_vendors"] = df_din["DIN_KEY"].map(n_vendors).fillna(0).astype("Int64")
            df_din["is_sole_vendor"] = (df_din["n_vendors"] == 1).astype("Int64")
            df_din["vendor_group"] = _make_vendor_group(df_din["n_vendors"], self.cfg.VENDOR_GROUPS)
            
            self.vendor_din_map = v.dropna(subset=["DIN_KEY", "VENDOR_NUM"])[["DIN_KEY", "VENDOR_NUM"]].drop_duplicates()
            
            logger.info(f"Vendor distribution: {df_din['vendor_group'].value_counts().to_dict()}")
        else:
            logger.warning("No VENDOR_NUM column found. Vendor features will be NA.")
            df_din["n_vendors"] = pd.Series([pd.NA] * len(df_din), dtype="Int64")
            df_din["is_sole_vendor"] = pd.Series([pd.NA] * len(df_din), dtype="Int64")
            df_din["vendor_group"] = pd.Series([pd.NA] * len(df_din), dtype="string")
            self.vendor_din_map = pd.DataFrame()
        
        atc_l3_mol_cnt = df_model_scope.groupby("ATC_LEVEL3")["FINAL_MOLECULE"].nunique()
        df_din["atc_l3_molecule_count"] = df_din["ATC_LEVEL3"].map(atc_l3_mol_cnt).fillna(1).astype("Int64")
        
        unique_molecules = df_model_scope.groupby("DIN_KEY")["FINAL_MOLECULE"].nunique()
        df_din["unique_molecules_per_din"] = df_din["DIN_KEY"].map(unique_molecules).fillna(1).astype("Int64")
        
        self.manifest["ATC_LEVEL1"] = "ATC level 1 therapeutic class"
        self.manifest["n_vendors"] = "Number of distinct vendors supplying this DIN"
        self.manifest["is_sole_vendor"] = "Binary indicator: 1 if sole-sourced"
        self.manifest["vendor_group"] = "Categorical vendor diversity: 1/2/3/4/5+ vendors"
        self.manifest["atc_l3_molecule_count"] = "Therapeutic diversity: unique molecules in ATC_LEVEL3"
        self.manifest["unique_molecules_per_din"] = "Formulation complexity: unique molecules per DIN"
        
        return df_din

    def build_backbone(self, df_din: pd.DataFrame, common_start: pd.Timestamp, common_end: pd.Timestamp) -> pd.DataFrame:
        """
        Create DIN-week Cartesian product backbone.
        
        Args:
            df_din: DIN master with static attributes
            common_start: First Monday in analysis window
            common_end: Last Monday in analysis window
            
        Returns:
            Complete DIN-week panel with static attributes
        """
        logger.info("Section 3.2: Building DIN-week backbone...")
        weeks = pd.date_range(start=common_start, end=common_end, freq=self.cfg.WEEK_FREQ)
        df_weeks = pd.DataFrame({"date_monday": weeks})
        df_panel = df_din.merge(df_weeks, how="cross")
        logger.info(f"Backbone: {len(df_panel):,} rows, {df_din['DIN_KEY'].nunique():,} DINs, {len(weeks):,} weeks")
        return df_panel

    def build_supply_features(self, df_supply: pd.DataFrame) -> pd.DataFrame:
        """
        Engineer supply chain features with statistical transformations.
        
        Aggregates supply data to DIN-week level and applies transformations to
        ensure numerical stability and model convergence. Raw quantities are
        log-transformed to handle high skewness. Supply gap statistics are
        winsorized to mitigate outlier influence.
        
        All features are lagged by 1 week minimum to prevent temporal leakage.
        Rolling statistics use expanding windows with minimum periods to handle
        sparse data gracefully.
        
        Args:
            df_supply: Supply data with quantities and dates
            
        Returns:
            DataFrame with DIN-week supply features
            
        Raises:
            ValueError: If required columns are missing
        """
        logger.info("Section 3.3: Engineering supply features...")
        if df_supply is None or df_supply.empty:
            return pd.DataFrame(columns=["DIN_KEY", "date_monday"])
        
        need_cols = ["DIN_KEY", "date_monday", "TARGET_RCV_QTY_4_WEEKS", "EXP_PO_4_WEEKS"]
        missing = [c for c in need_cols if c not in df_supply.columns]
        if missing:
            raise ValueError(f"df_supply missing required columns: {missing}")
        
        sup = df_supply.copy()
        sup["date_monday"] = pd.to_datetime(sup["date_monday"], errors="coerce")
        sup["target_rcv_qty_4w"] = _coerce_numeric(sup["TARGET_RCV_QTY_4_WEEKS"]).fillna(0.0)
        sup["exp_po_4w"] = _coerce_numeric(sup["EXP_PO_4_WEEKS"]).fillna(0.0)
        
        df_sup = (
            sup.dropna(subset=["DIN_KEY", "date_monday"])
               .groupby(["DIN_KEY", "date_monday"], as_index=False)[["target_rcv_qty_4w", "exp_po_4w"]]
               .sum()
               .sort_values(["DIN_KEY", "date_monday"])
        )
        
        df_sup["supply_gap_raw"] = df_sup.apply(
            lambda r: self.supply_logic.calculate_gap(r["exp_po_4w"], r["target_rcv_qty_4w"]), 
            axis=1
        )
        df_sup["gap_is_deficit"] = df_sup["supply_gap_raw"].apply(self.supply_logic.is_deficit).astype("Int64")
        
        g = df_sup.groupby("DIN_KEY", sort=False)
        
        raw_target = g["target_rcv_qty_4w"].shift(1)
        raw_po = g["exp_po_4w"].shift(1)
        df_sup["target_rcv_qty_4w_lag1"] = np.log1p(raw_target.clip(lower=0))
        df_sup["exp_po_4w_lag1"] = np.log1p(raw_po.clip(lower=0))
        
        df_sup["supply_gap_raw_lag1"] = g["supply_gap_raw"].shift(1)
        df_sup["gap_is_deficit_lag1"] = g["gap_is_deficit"].shift(1)
        
        w = self.cfg.GAP_ROLL_WINDOWS[0]
        df_sup["supply_gap_roll_mean_w4"] = g["supply_gap_raw"].transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean()
        )
        df_sup["supply_gap_roll_std_w4"] = g["supply_gap_raw"].transform(
            lambda x: x.shift(1).rolling(w, min_periods=2).std()
        )
        
        p99_mean = df_sup["supply_gap_roll_mean_w4"].abs().quantile(self.cfg.WINSORIZE_PERCENTILE)
        p99_std = df_sup["supply_gap_roll_std_w4"].quantile(self.cfg.WINSORIZE_PERCENTILE)
        df_sup["supply_gap_roll_mean_w4"] = df_sup["supply_gap_roll_mean_w4"].clip(lower=-p99_mean, upper=p99_mean)
        df_sup["supply_gap_roll_std_w4"] = df_sup["supply_gap_roll_std_w4"].clip(upper=p99_std)
        
        df_sup["supply_volatility_w4"] = df_sup["supply_gap_roll_std_w4"] / (
            df_sup["supply_gap_roll_mean_w4"].abs() + 1e-6
        )
        df_sup["supply_volatility_w4"] = df_sup["supply_volatility_w4"].clip(upper=100)
        
        lag = df_sup["supply_gap_raw_lag1"]
        df_sup["supply_gap_log_signed_lag1"] = np.sign(lag) * np.log1p(np.abs(lag))
        
        p99_gap = df_sup["supply_gap_raw_lag1"].abs().quantile(self.cfg.WINSORIZE_PERCENTILE)
        df_sup["supply_gap_raw_lag1_winsorized"] = df_sup["supply_gap_raw_lag1"].clip(
            lower=-p99_gap, upper=p99_gap
        )
        
        denom = raw_target.replace(0, np.nan)
        df_sup["po_to_target_ratio_lag1"] = (raw_po / denom).replace([np.inf, -np.inf], np.nan)
        
        keep = [
            "DIN_KEY", "date_monday",
            "target_rcv_qty_4w_lag1", "exp_po_4w_lag1",
            "supply_gap_raw_lag1_winsorized", "gap_is_deficit_lag1",
            "supply_gap_roll_mean_w4", "supply_gap_roll_std_w4",
            "supply_volatility_w4", "supply_gap_log_signed_lag1",
            "po_to_target_ratio_lag1",
        ]
        
        self.manifest["target_rcv_qty_4w_lag1"] = "Log-transformed target receive quantity, lag 1w"
        self.manifest["exp_po_4w_lag1"] = "Log-transformed expected PO, lag 1w"
        self.manifest["supply_gap_raw_lag1_winsorized"] = "Supply gap winsorized at 99th percentile, lag 1w"
        self.manifest["gap_is_deficit_lag1"] = "Binary deficit indicator, lag 1w"
        self.manifest["supply_gap_roll_mean_w4"] = "Rolling 4w mean gap (winsorized), lag 1w"
        self.manifest["supply_gap_roll_std_w4"] = "Rolling 4w std gap (winsorized), lag 1w"
        self.manifest["supply_volatility_w4"] = "Supply volatility ratio (capped at 100), lag 1w"
        self.manifest["supply_gap_log_signed_lag1"] = "Signed log-transformed absolute gap, lag 1w"
        self.manifest["po_to_target_ratio_lag1"] = "PO to target ratio, lag 1w"
        
        logger.info(f"Supply: 9 features, {df_sup['DIN_KEY'].nunique():,} DINs")
        return df_sup[keep].copy()

    def build_trends_features(self, df_trends: pd.DataFrame) -> pd.DataFrame:
        """
        Engineer Google Trends features with hierarchical imputation.
        
        Constructs ATC-level trend indicators using hierarchical filling strategy:
        first fills with weekly cross-ATC mean, then falls back to global mean
        for unmapped ATCs. This ensures all DINs have trend values while
        preserving temporal variation.
        
        Args:
            df_trends: Google Trends data with ATC and composite index
            
        Returns:
            DataFrame with ATC-week trend features
            
        Raises:
            ValueError: If required columns are missing
        """
        logger.info("Section 3.4: Engineering trends features...")
        if df_trends is None or df_trends.empty:
            return pd.DataFrame(columns=["ATC_LEVEL1", "date_monday"])
        
        need = ["ATC_LEVEL1", "date_monday", "COMPOSITE_INDEX"]
        missing = [c for c in need if c not in df_trends.columns]
        if missing:
            raise ValueError(f"df_trends missing required columns: {missing}")
        
        t = df_trends[need].copy()
        t["ATC_LEVEL1"] = t["ATC_LEVEL1"].astype(str).str.strip().str.upper()
        t["date_monday"] = pd.to_datetime(t["date_monday"], errors="coerce")
        t["COMPOSITE_INDEX"] = _coerce_numeric(t["COMPOSITE_INDEX"])
        t = t.dropna(subset=["ATC_LEVEL1", "date_monday"]).sort_values(["ATC_LEVEL1", "date_monday"])
        
        g = t.groupby("ATC_LEVEL1", sort=False)
        t["trend_lag1"] = g["COMPOSITE_INDEX"].shift(1)
        
        w = self.cfg.TRENDS_ROLL_WINDOW
        t["trend_prev4w_mean"] = g["COMPOSITE_INDEX"].transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean()
        )
        t["trend_momentum_4w"] = t["trend_lag1"] - t["trend_prev4w_mean"]
        
        t["trend_lag5"] = g["COMPOSITE_INDEX"].shift(5)
        denom = t["trend_lag5"].replace(0, np.nan)
        t["trend_pct_change_4w"] = (t["trend_lag1"] - t["trend_lag5"]) / denom
        
        out = t[["ATC_LEVEL1", "date_monday", "trend_lag1", "trend_prev4w_mean", 
                 "trend_momentum_4w", "trend_pct_change_4w"]].copy()
        
        weekly_mean = out.groupby("date_monday")["trend_lag1"].mean()
        global_mean = float(out["trend_lag1"].mean()) if out["trend_lag1"].notna().any() else 60.0
        
        self.global_trend_mean = global_mean
        
        for c in ["trend_lag1", "trend_prev4w_mean", "trend_momentum_4w", "trend_pct_change_4w"]:
            out[c] = out[c].where(out[c].notna(), out["date_monday"].map(weekly_mean))
            out[c] = out[c].fillna(global_mean if c in ["trend_lag1", "trend_prev4w_mean"] else 0.0)
        
        self.manifest["trend_lag1"] = "Google Trends index, lag 1w, hierarchically filled"
        self.manifest["trend_prev4w_mean"] = "Mean trends over 4w, lag 1w, hierarchically filled"
        self.manifest["trend_pct_change_4w"] = "Trends percent change vs 4w prior, hierarchically filled"
        self.manifest["trend_momentum_4w"] = "Momentum: current minus 4w mean, hierarchically filled"
        
        logger.info(f"Trends: 4 features, {out['ATC_LEVEL1'].nunique()} ATCs, global_mean={global_mean:.2f}")
        return out

    def build_sales_features(self, df_sales: pd.DataFrame) -> pd.DataFrame:
        """
        Engineer national sales features with log transformation.
        
        Transforms monthly national sales into weekly indicators by allocating
        monthly totals across constituent Mondays. Log transformation handles
        the large magnitude (hundreds of millions) and ensures numerical stability.
        
        All features use previous month data with 1-week lag to prevent leakage.
        
        Args:
            df_sales: Sales data with year, month, and total sales
            
        Returns:
            DataFrame with weekly sales features
            
        Raises:
            ValueError: If required columns are missing
        """
        logger.info("Section 3.5: Engineering sales features...")
        if df_sales is None or df_sales.empty:
            return pd.DataFrame(columns=["date_monday"])
        
        need = ["CAL_YEAR", "CAL_MONTH", "Sales"]
        missing = [c for c in need if c not in df_sales.columns]
        if missing:
            raise ValueError(f"df_sales missing required columns: {missing}")
        
        s = df_sales[need].copy()
        s["CAL_YEAR"] = _coerce_numeric(s["CAL_YEAR"])
        s["CAL_MONTH"] = _coerce_numeric(s["CAL_MONTH"])
        s["Sales"] = _coerce_numeric(s["Sales"]).fillna(0.0)
        s = s.dropna(subset=["CAL_YEAR", "CAL_MONTH"])
        s["date_month"] = pd.to_datetime(
            s["CAL_YEAR"].astype(int).astype(str) + "-" + 
            s["CAL_MONTH"].astype(int).astype(str) + "-01",
            errors="coerce"
        )
        m = s.groupby("date_month", as_index=False)["Sales"].sum().sort_values("date_month")
        m["sales_prev_month_total"] = m["Sales"].shift(1)
        
        rows = []
        for _, r in m.iterrows():
            month_start = pd.Timestamp(r["date_month"])
            month_end = (month_start + pd.offsets.MonthEnd(1)).normalize()
            mondays = pd.date_range(month_start, month_end, freq="W-MON")
            if len(mondays) == 0:
                continue
            prev_total = r["sales_prev_month_total"]
            weekly = (prev_total / len(mondays)) if pd.notna(prev_total) else np.nan
            rows.append(pd.DataFrame({
                "date_monday": mondays,
                "sales_prev_month_weekly_raw": weekly
            }))
        
        w = pd.concat(rows, ignore_index=True).sort_values("date_monday") if rows else \
            pd.DataFrame(columns=["date_monday", "sales_prev_month_weekly_raw"])
        w["sales_prev_month_weekly_raw_lag1"] = w["sales_prev_month_weekly_raw"].shift(1)
        w["sales_prev_month_weekly_lag1"] = np.log1p(w["sales_prev_month_weekly_raw_lag1"].clip(lower=0))
        
        m["sales_prevprev_month_total"] = m["Sales"].shift(2)
        denom = m["sales_prevprev_month_total"].replace(0, np.nan)
        m["sales_pct_change_m1"] = (m["sales_prev_month_total"] - m["sales_prevprev_month_total"]) / denom
        
        pct_rows = []
        for _, r in m.iterrows():
            month_start = pd.Timestamp(r["date_month"])
            month_end = (month_start + pd.offsets.MonthEnd(1)).normalize()
            mondays = pd.date_range(month_start, month_end, freq="W-MON")
            if len(mondays) == 0:
                continue
            pct_rows.append(pd.DataFrame({
                "date_monday": mondays,
                "sales_prev_month_weekly_pct_change_m1": r["sales_pct_change_m1"]
            }))
        
        pct = pd.concat(pct_rows, ignore_index=True).sort_values("date_monday") if pct_rows else \
              pd.DataFrame(columns=["date_monday", "sales_prev_month_weekly_pct_change_m1"])
        pct["sales_prev_month_weekly_pct_change_m1"] = pct["sales_prev_month_weekly_pct_change_m1"].shift(1)
        
        out = w[["date_monday", "sales_prev_month_weekly_lag1"]].merge(
            pct[["date_monday", "sales_prev_month_weekly_pct_change_m1"]], 
            on="date_monday", 
            how="left"
        )
        out["sales_prev_month_weekly_lag1"] = out["sales_prev_month_weekly_lag1"].fillna(0.0)
        out["sales_prev_month_weekly_pct_change_m1"] = out["sales_prev_month_weekly_pct_change_m1"].fillna(0.0)
        
        self.manifest["sales_prev_month_weekly_lag1"] = "Log-transformed weekly national sales, lag 1w"
        self.manifest["sales_prev_month_weekly_pct_change_m1"] = "Month-over-month sales percent change, lag 1w"
        
        logger.info(f"Sales: 2 features, {len(out):,} weeks")
        return out

    def build_vendor_credit_features(self, df_panel: pd.DataFrame) -> pd.DataFrame:
        """
        Engineer vendor reliability features using vectorized approach.
        
        Constructs six complementary vendor credit metrics that capture different
        dimensions of supplier reliability. Uses vectorized operations for
        computational efficiency (100x faster than nested loops).
        
        Features measure: central tendency (avg), tail risk (max/min), size-adjusted
        risk (weighted), stability (consistency), and dynamics (trend). These multiple
        perspectives enable the model to learn nuanced vendor risk patterns.
        
        Args:
            df_panel: Panel with y_shortage_t computed
            
        Returns:
            df_panel with 6 vendor credit features added
        """
        logger.info("Section 3.6: Engineering vendor credit features...")
        
        if self.vendor_din_map.empty:
            logger.warning("No vendor mapping available. Filling vendor features with 0.")
            for col in ['vendor_avg_shortage_12w', 'vendor_max_shortage_12w', 
                        'vendor_min_shortage_12w', 'vendor_weighted_shortage_12w',
                        'vendor_consistency', 'vendor_recent_trend']:
                df_panel[col] = 0.0
            return df_panel
        
        if "y_shortage_t" not in df_panel.columns:
            logger.warning("y_shortage_t not computed. Skipping vendor credit.")
            return df_panel
        
        print("Computing vendor features (vectorized)...")
        
        vendor_panel = self.vendor_din_map.merge(
            df_panel[["DIN_KEY", "date_monday", "y_shortage_t"]],
            on="DIN_KEY",
            how="inner"
        ).sort_values(["VENDOR_NUM", "date_monday"])
        
        lookback = self.cfg.VENDOR_CREDIT_LOOKBACK_WEEKS
        
        vendor_panel["vendor_shortage_rate"] = (
            vendor_panel.groupby("VENDOR_NUM")["y_shortage_t"]
            .transform(lambda x: x.shift(1).rolling(lookback, min_periods=1).mean())
        )
        
        vendor_panel["vendor_shortage_recent_4w"] = (
            vendor_panel.groupby("VENDOR_NUM")["y_shortage_t"]
            .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
        )
        vendor_panel["vendor_shortage_prev_8w"] = (
            vendor_panel.groupby("VENDOR_NUM")["y_shortage_t"]
            .transform(lambda x: x.shift(5).rolling(8, min_periods=1).mean())
        )
        vendor_panel["vendor_recent_trend"] = (
            vendor_panel["vendor_shortage_recent_4w"] - vendor_panel["vendor_shortage_prev_8w"]
        ).fillna(0.0)
        
        vendor_panel["vendor_consistency"] = (
            vendor_panel.groupby("VENDOR_NUM")["y_shortage_t"]
            .transform(lambda x: 1 - x.shift(1).rolling(lookback, min_periods=2).std().fillna(0))
        ).clip(0, 1)
        
        vendor_size = self.vendor_din_map.groupby("VENDOR_NUM")["DIN_KEY"].nunique().to_dict()
        vendor_panel["vendor_din_count"] = vendor_panel["VENDOR_NUM"].map(vendor_size)
        
        vendor_agg = (
            vendor_panel.groupby(["DIN_KEY", "date_monday"], as_index=False)
            .agg({
                'vendor_shortage_rate': ['mean', 'max', 'min'],
                'vendor_recent_trend': 'mean',
                'vendor_consistency': 'mean'
            })
        )
        vendor_agg.columns = ['DIN_KEY', 'date_monday', 
                              'vendor_avg_shortage_12w', 
                              'vendor_max_shortage_12w', 
                              'vendor_min_shortage_12w',
                              'vendor_recent_trend',
                              'vendor_consistency']
        
        vendor_weighted = (
            vendor_panel.assign(
                weighted_shortage=vendor_panel['vendor_shortage_rate'] * vendor_panel['vendor_din_count']
            )
            .groupby(["DIN_KEY", "date_monday"], as_index=False)
            .apply(lambda x: pd.Series({
                'vendor_weighted_shortage_12w': 
                    (x['weighted_shortage'].sum() / x['vendor_din_count'].sum()) 
                    if x['vendor_din_count'].sum() > 0 else 0
            }), include_groups=False)
            .reset_index()
        )
        
        vendor_features = vendor_agg.merge(vendor_weighted, on=["DIN_KEY", "date_monday"], how="left")
        
        df_panel = df_panel.merge(vendor_features, on=["DIN_KEY", "date_monday"], how="left")
        
        for col in ['vendor_avg_shortage_12w', 'vendor_max_shortage_12w', 
                    'vendor_min_shortage_12w', 'vendor_weighted_shortage_12w',
                    'vendor_consistency', 'vendor_recent_trend']:
            df_panel[col] = df_panel[col].fillna(0.0)
        
        self.manifest["vendor_avg_shortage_12w"] = "Average vendor shortage rate over 12w"
        self.manifest["vendor_max_shortage_12w"] = "Maximum vendor shortage rate (tail risk)"
        self.manifest["vendor_min_shortage_12w"] = "Minimum vendor shortage rate (fallback option)"
        self.manifest["vendor_weighted_shortage_12w"] = "Vendor shortage rate weighted by vendor size"
        self.manifest["vendor_consistency"] = "Vendor reliability score: 1 - std(shortage_rate)"
        self.manifest["vendor_recent_trend"] = "Vendor trend: recent 4w minus prior 8w shortage rate"
        
        logger.info(f"Vendor: 6 features, avg={df_panel['vendor_avg_shortage_12w'].mean():.4f}, "
                    f"max={df_panel['vendor_max_shortage_12w'].mean():.4f}")
        
        return df_panel

    def build_labels(self, df_panel: pd.DataFrame, df_target: pd.DataFrame) -> pd.DataFrame:
        """
        Construct shortage labels using centralized logic.
        
        Creates multi-horizon binary targets (t+1 through t+4) and historical
        features. All labels use the centralized ShortageLogic implementation
        to ensure consistency across the pipeline.
        
        Args:
            df_panel: Panel with FINAL_MOLECULE column
            df_target: Shortage data with dates and molecule flags
            
        Returns:
            df_panel with labels and history features added
            
        Raises:
            ValueError: If required columns are missing
        """
        logger.info("Section 3.7: Building shortage labels...")
        
        need = ["MY_DATE", "SHORTAGE_FLAG", "MOLECULE_NM"]
        missing = [c for c in need if c not in df_target.columns]
        if missing:
            raise ValueError(f"df_target missing required columns: {missing}")
        
        tgt = df_target[need].copy()
        tgt["date_monday"] = _to_monday(tgt["MY_DATE"])
        tgt["SHORTAGE_FLAG"] = _coerce_numeric(tgt["SHORTAGE_FLAG"]).fillna(0).astype(int)
        tgt["MOLECULE_NM"] = tgt["MOLECULE_NM"].astype("string").str.strip().str.upper()
        tgt = tgt.dropna(subset=["date_monday"])
        tgt = tgt[tgt["SHORTAGE_FLAG"] == 1].copy()
        
        if tgt.empty:
            logger.warning("No positive SHORTAGE_FLAG found. All labels will be 0.")
            out = df_panel.copy()
            for h in self.cfg.HORIZONS:
                out[f"y_shortage_t_plus_{h}"] = 0
            out["y_shortage_t"] = 0
            out["y_shortage_t_lag1"] = 0
            out["y_shortage_rate_prev4w"] = 0.0
            return out
        
        mol_by_week = tgt.groupby("date_monday")["MOLECULE_NM"].apply(
            lambda s: sorted(set([x for x in s.dropna().tolist() if x]))
        ).to_dict()
        
        def _check_shortage(row):
            wk = row["date_monday"]
            shortage_mols = mol_by_week.get(wk, [])
            return self.shortage_logic.is_shortage_week(row.get("FINAL_MOLECULE", ""), shortage_mols)
        
        print("Computing shortage labels...")
        tqdm.pandas(desc="Applying shortage logic", ncols=100)
        out = df_panel.copy()
        out["y_shortage_t"] = out.progress_apply(_check_shortage, axis=1).astype(int)
        
        print("Creating multi-horizon targets...")
        out = out.sort_values(["DIN_KEY", "date_monday"])
        g = out.groupby("DIN_KEY", sort=False)
        
        for h in self.cfg.HORIZONS:
            out[f"y_shortage_t_plus_{h}"] = g["y_shortage_t"].shift(-h).fillna(0).astype(int)
        
        out["y_shortage_t_lag1"] = g["y_shortage_t"].shift(1).fillna(0).astype(int)
        out["y_shortage_rate_prev4w"] = g["y_shortage_t"].transform(
            lambda x: x.shift(1).rolling(4, min_periods=1).mean()
        ).fillna(0.0)
        
        self.manifest["y_shortage_t_lag1"] = "Historical shortage indicator at t, lag 1w"
        self.manifest["y_shortage_rate_prev4w"] = "Historical shortage rate over 4w, lag 1w"
        for h in self.cfg.HORIZONS:
            self.manifest[f"y_shortage_t_plus_{h}"] = f"Target: shortage at t+{h}"
        
        logger.info(f"Labels: {len(self.cfg.HORIZONS)} horizons + 2 history features")
        return out

    def run(self, df_model_scope, df_supply, df_sales, df_target, df_trends):
        """
        Execute complete feature engineering pipeline.
        
        Orchestrates the full transformation workflow from raw data to modeling-ready
        panel. Handles date alignment, feature construction, merging, and final
        validation. Returns a complete DIN-week panel with all features and labels.
        
        Args:
            df_model_scope: Product universe from Section 2
            df_supply: Supply chain data
            df_sales: National sales data
            df_target: Shortage labels
            df_trends: Google Trends data
            
        Returns:
            Tuple of (df_panel, feature_manifest)
        """
        
        if not df_supply.empty and "date_monday" not in df_supply.columns:
            if "YEAR_WEEK" in df_supply.columns:
                df_supply = df_supply.copy()
                df_supply["date_monday"] = DataPreProcessor.yearweek_to_monday(df_supply["YEAR_WEEK"])
        
        sup_for_win = df_supply[["date_monday"]].dropna() if (isinstance(df_supply, pd.DataFrame) and "date_monday" in df_supply.columns) else pd.DataFrame()
        tr_for_win = df_trends[["date_monday"]].dropna() if (isinstance(df_trends, pd.DataFrame) and "date_monday" in df_trends.columns) else pd.DataFrame()
        common_start, common_end = _common_week_window(sup_for_win, tr_for_win, date_col="date_monday")
        if common_start is None or common_end is None:
            common_start, common_end = _common_week_window(sup_for_win, date_col="date_monday")
        if common_start is None or common_end is None:
            raise ValueError("Unable to determine common week window from available data.")
        
        df_din = self.build_din_master(df_model_scope)
        df_panel = self.build_backbone(df_din, common_start, common_end)
        df_sup_feat = self.build_supply_features(df_supply)
        df_tr_feat = self.build_trends_features(df_trends)
        df_sales_feat = self.build_sales_features(df_sales)
        
        print("\n" + "─" * 80)
        print("Merging features to panel...")
        print("─" * 80)
        
        df_panel = df_panel.merge(df_sup_feat, on=["DIN_KEY", "date_monday"], how="left")
        supply_cols = ["target_rcv_qty_4w_lag1", "exp_po_4w_lag1", "supply_gap_raw_lag1_winsorized", 
                      "gap_is_deficit_lag1", "supply_gap_roll_mean_w4", "supply_gap_roll_std_w4", 
                      "supply_volatility_w4", "supply_gap_log_signed_lag1", "po_to_target_ratio_lag1"]
        for c in supply_cols:
            if c in df_panel.columns:
                df_panel[c] = df_panel[c].fillna(0.0 if c != "gap_is_deficit_lag1" else 0)
        
        df_panel = df_panel.merge(df_tr_feat, on=["ATC_LEVEL1", "date_monday"], how="left")
        
        global_trend_mean = getattr(self, 'global_trend_mean', 60.0)
        for c in ["trend_lag1", "trend_prev4w_mean", "trend_momentum_4w", "trend_pct_change_4w"]:
            if c in df_panel.columns:
                df_panel[c] = df_panel[c].fillna(global_trend_mean if c in ["trend_lag1", "trend_prev4w_mean"] else 0.0)
        
        df_panel = df_panel.merge(df_sales_feat, on="date_monday", how="left")
        
        df_panel = self.build_labels(df_panel, df_target)
        df_panel = self.build_vendor_credit_features(df_panel)
        
        print("\n" + "─" * 80)
        print("Finalizing panel...")
        print("─" * 80)
        max_h = max(self.cfg.HORIZONS)
        df_panel = df_panel.sort_values(["DIN_KEY", "date_monday"])
        df_panel["_rank_desc"] = df_panel.groupby("DIN_KEY").cumcount(ascending=False)
        df_panel = df_panel[df_panel["_rank_desc"] >= max_h].drop(columns=["_rank_desc"])
        
        logger.info("Section 3 complete. Label statistics:")
        for h in self.cfg.HORIZONS:
            rate = float(df_panel[f"y_shortage_t_plus_{h}"].mean())
            print(f"  t+{h} positive rate: {rate:.6f}")
        
        return df_panel, self.manifest


cfg = FeatureConfig()
builder = FeatureBuilder(cfg)

print("\n" + "=" * 80)
print("SECTION 3: FEATURE ENGINEERING")
print("=" * 80)

df_panel, feature_manifest = builder.run(
    df_model_scope=df_model_scope,
    df_supply=df_supply,
    df_sales=df_sales,
    df_target=df_target,
    df_trends=df_trends
)

print("\n" + "=" * 80)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 80)
print(f"Panel Shape          : {df_panel.shape}")
print(f"Rows (DIN-week)      : {len(df_panel):,}")
print(f"Unique DINs          : {df_panel['DIN_KEY'].nunique():,}")
unique_mols = df_panel[['DIN_KEY', 'unique_molecules_per_din']].drop_duplicates('DIN_KEY')['unique_molecules_per_din'].sum()
print(f"Total Molecule Count : {int(unique_mols):,}")
print(f"Week Range           : {df_panel['date_monday'].min().date()} to {df_panel['date_monday'].max().date()}")
print(f"Total Weeks          : {df_panel['date_monday'].nunique()}")
print("=" * 80)

print("\n" + "=" * 80)
print("FEATURE INVENTORY (33 features)")
print("=" * 80)

feature_groups = {
    "STATIC ATTRIBUTES (6)": [
        "ATC_LEVEL1", "n_vendors", "is_sole_vendor", "vendor_group", 
        "atc_l3_molecule_count", "unique_molecules_per_din"
    ],
    "SUPPLY FEATURES (9)": [
        "target_rcv_qty_4w_lag1", "exp_po_4w_lag1", "supply_gap_raw_lag1_winsorized", 
        "gap_is_deficit_lag1", "supply_gap_roll_mean_w4", "supply_gap_roll_std_w4",
        "supply_volatility_w4", "supply_gap_log_signed_lag1", "po_to_target_ratio_lag1"
    ],
    "TRENDS FEATURES (4)": [
        "trend_lag1", "trend_prev4w_mean", "trend_pct_change_4w", "trend_momentum_4w"
    ],
    "SALES FEATURES (2)": [
        "sales_prev_month_weekly_lag1", "sales_prev_month_weekly_pct_change_m1"
    ],
    "VENDOR CREDIT (6)": [
        "vendor_avg_shortage_12w", "vendor_max_shortage_12w", "vendor_min_shortage_12w",
        "vendor_weighted_shortage_12w", "vendor_consistency", "vendor_recent_trend"
    ],
    "HISTORY FEATURES (2)": [
        "y_shortage_t_lag1", "y_shortage_rate_prev4w"
    ],
    "TARGET LABELS (4)": [
        "y_shortage_t_plus_1", "y_shortage_t_plus_2", "y_shortage_t_plus_3", "y_shortage_t_plus_4"
    ]
}

for group_name, features in feature_groups.items():
    print(f"\n{group_name}")
    print("─" * 80)
    for feat in features:
        if feat in df_panel.columns:
            if feat in feature_manifest:
                desc = feature_manifest[feat]
                print(f"  {feat:<40} {desc}")
            else:
                print(f"  {feat:<40} (in panel)")
        else:
            print(f"  {feat:<40} MISSING FROM PANEL")

print("\n" + "=" * 80)
print("DATA QUALITY VALIDATION")
print("=" * 80)
print(f"Missing values:")
print(f"  trend_lag1           : {df_panel['trend_lag1'].isna().sum():,} ({df_panel['trend_lag1'].isna().mean():.2%})")
print(f"  atc_l3_molecule_count: {df_panel['atc_l3_molecule_count'].isna().sum():,}")
print(f"  supply features      : {df_panel['target_rcv_qty_4w_lag1'].isna().sum():,}")
print(f"\nFeature ranges:")
print(f"  supply (log)         : [{df_panel['target_rcv_qty_4w_lag1'].min():.2f}, {df_panel['target_rcv_qty_4w_lag1'].max():.2f}]")
print(f"  gap (winsorized)     : [{df_panel['supply_gap_raw_lag1_winsorized'].min():.0f}, {df_panel['supply_gap_raw_lag1_winsorized'].max():.0f}]")
print(f"  sales (log)          : [{df_panel['sales_prev_month_weekly_lag1'].min():.2f}, {df_panel['sales_prev_month_weekly_lag1'].max():.2f}]")
print(f"  vendor_avg           : [{df_panel['vendor_avg_shortage_12w'].min():.4f}, {df_panel['vendor_avg_shortage_12w'].max():.4f}]")
print(f"  vendor_max           : [{df_panel['vendor_max_shortage_12w'].min():.4f}, {df_panel['vendor_max_shortage_12w'].max():.4f}]")
print("=" * 80)

print("\nFeature engineering complete. Panel ready for modeling.")

2026-01-17 10:52:03,157 - [INFO] - Executing Section 3 (Feature Engineering)...

SECTION 3: FEATURE ENGINEERING
2026-01-17 10:52:03,244 - [INFO] - Section 3.1: Building DIN master attributes...
2026-01-17 10:52:03,372 - [INFO] - Vendor distribution: {'1_vendor': 12232, '2_vendors': 3683, '3_vendors': 608, '5+_vendors': 204, '4_vendors': 132}
2026-01-17 10:52:03,412 - [INFO] - Section 3.2: Building DIN-week backbone...
2026-01-17 10:52:03,548 - [INFO] - Backbone: 775,514 rows, 16,859 DINs, 46 weeks
2026-01-17 10:52:03,548 - [INFO] - Section 3.3: Engineering supply features...
2026-01-17 10:52:03,695 - [INFO] - Supply: 9 features, 208 DINs
2026-01-17 10:52:03,697 - [INFO] - Section 3.4: Engineering trends features...
2026-01-17 10:52:03,711 - [INFO] - Trends: 4 features, 13 ATCs, global_mean=60.19
2026-01-17 10:52:03,712 - [INFO] - Section 3.5: Engineering sales features...
2026-01-17 10:52:03,882 - [INFO] - Sales: 2 features, 248 weeks

──────────────────────────────────────────────────

Applying shortage logic:   0%|                                           | 0/775514 [00:00<?, ?it/s]

Creating multi-horizon targets...
2026-01-17 10:52:13,644 - [INFO] - Labels: 4 horizons + 2 history features
2026-01-17 10:52:13,671 - [INFO] - Section 3.6: Engineering vendor credit features...
Computing vendor features (vectorized)...
2026-01-17 10:54:28,794 - [INFO] - Vendor: 6 features, avg=0.0157, max=0.0194

────────────────────────────────────────────────────────────────────────────────
Finalizing panel...
────────────────────────────────────────────────────────────────────────────────
2026-01-17 10:54:29,610 - [INFO] - Section 3 complete. Label statistics:
  t+1 positive rate: 0.016132
  t+2 positive rate: 0.016075
  t+3 positive rate: 0.015963
  t+4 positive rate: 0.015851

FEATURE ENGINEERING COMPLETE
Panel Shape          : (708078, 39)
Rows (DIN-week)      : 708,078
Unique DINs          : 16,859
Total Molecule Count : 17,908
Week Range           : 2023-01-02 to 2023-10-16
Total Weeks          : 42

FEATURE INVENTORY (33 features)

STATIC ATTRIBUTES (6)
──────────────────────

In [82]:
df_panel.head(100)

,DIN_KEY,ATC_LEVEL1,ATC_LEVEL3,FINAL_MOLECULE,n_vendors,is_sole_vendor,vendor_group,atc_l3_molecule_count,unique_molecules_per_din,date_monday,target_rcv_qty_4w_lag1,exp_po_4w_lag1,supply_gap_raw_lag1_winsorized,gap_is_deficit_lag1,supply_gap_roll_mean_w4,supply_gap_roll_std_w4,supply_volatility_w4,supply_gap_log_signed_lag1,po_to_target_ratio_lag1,trend_lag1,trend_prev4w_mean,trend_momentum_4w,trend_pct_change_4w,sales_prev_month_weekly_lag1,sales_prev_month_weekly_pct_change_m1,y_shortage_t,y_shortage_t_plus_1,y_shortage_t_plus_2,y_shortage_t_plus_3,y_shortage_t_plus_4,y_shortage_t_lag1,y_shortage_rate_prev4w,vendor_avg_shortage_12w,vendor_max_shortage_12w,vendor_min_shortage_12w,vendor_recent_trend,vendor_consistency,index,vendor_weighted_shortage_12w
0,00000151,X,NaN,CYTOSPRAY 2OZ 180ML,1,1,1_vendor,1,1,2023-01-02,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,60.1945,60.1945,0.0000,0.0000,19.6400,0.0424,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0,0.0000
1,00000151,X,NaN,CYTOSPRAY 2OZ 180ML,1,1,1_vendor,1,1,2023-01-09,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,60.1945,60.1945,0.0000,0.0000,19.4107,-0.0061,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1,0.0000
2,00000151,X,NaN,CYTOSPRAY 2OZ 180ML,1,1,1_vendor,1,1,2023-01-16,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,60.1945,60.1945,0.0000,0.0000,19.4107,-0.0061,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2,0.0000
3,00000151,X,NaN,CYTOSPRAY 2OZ 180ML,1,1,1_vendor,1,1,2023-01-23,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,60.1945,60.1945,0.0000,0.0000,19.4107,-0.0061,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,3,0.0000
4,00000151,X,NaN,CYTOSPRAY 2OZ 180ML,1,1,1_vendor,1,1,2023-01-30,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,60.1945,60.1945,0.0000,0.0000,19.4107,-0.0061,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,4,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,00000655,S,S01E,CARBACHOL,1,1,1_vendor,36,1,2023-03-20,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,100.0000,73.2500,26.7500,0.5625,19.4874,-0.1059,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,103,0.0000
104,00000655,S,S01E,CARBACHOL,1,1,1_vendor,36,1,2023-03-27,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,84.0000,78.5000,5.5000,0.3333,19.4874,-0.1059,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,104,0.0000
105,00000655,S,S01E,CARBACHOL,1,1,1_vendor,36,1,2023-04-03,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,62.0000,77.2500,-15.2500,-0.0746,19.4874,-0.1059,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,105,0.0000
106,00000655,S,S01E,CARBACHOL,1,1,1_vendor,36,1,2023-04-10,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,66.0000,78.0000,-12.0000,0.0476,19.6649,0.1942,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,106,0.0000


In [83]:
df_panel.describe()

,n_vendors,is_sole_vendor,atc_l3_molecule_count,unique_molecules_per_din,date_monday,target_rcv_qty_4w_lag1,exp_po_4w_lag1,supply_gap_raw_lag1_winsorized,gap_is_deficit_lag1,supply_gap_roll_mean_w4,supply_gap_roll_std_w4,supply_volatility_w4,supply_gap_log_signed_lag1,po_to_target_ratio_lag1,trend_lag1,trend_prev4w_mean,trend_momentum_4w,trend_pct_change_4w,sales_prev_month_weekly_lag1,sales_prev_month_weekly_pct_change_m1,y_shortage_t,y_shortage_t_plus_1,y_shortage_t_plus_2,y_shortage_t_plus_3,y_shortage_t_plus_4,y_shortage_t_lag1,y_shortage_rate_prev4w,vendor_avg_shortage_12w,vendor_max_shortage_12w,vendor_min_shortage_12w,vendor_recent_trend,vendor_consistency,index,vendor_weighted_shortage_12w
count,708078.0000,708078.0000,708078.0000,708078.0000,708078,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000,708078.0000
mean,1.3933,0.7255,23.4449,1.0622,2023-05-25 12:00:00,0.0389,0.0455,3.4243,0.0026,3.9966,9.6394,0.1287,0.0097,0.0205,55.2273,55.1058,0.1215,0.0075,19.5217,0.0025,0.0162,0.0161,0.0161,0.0160,0.0159,0.0157,0.0157,0.0159,0.0197,0.0124,-0.0002,0.9640,387754.5000,0.0163
min,1.0000,0.0000,1.0000,1.0000,2023-01-02 00:00:00,0.0000,0.0000,-16336.3700,0.0000,-12161.9375,0.0000,0.0000,-10.9490,-7.9118,10.0000,10.7500,-15.2500,-0.3860,19.3040,-0.1287,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,-1.0000,0.2929,0.0000,0.0000
25%,1.0000,0.0000,4.0000,1.0000,2023-03-13 00:00:00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,19.0000,19.0000,-1.5000,-0.0588,19.4107,-0.0743,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,193875.2500,0.0000
50%,1.0000,1.0000,19.0000,1.0000,2023-05-25 12:00:00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,63.0000,64.2500,0.0000,0.0000,19.5346,-0.0210,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,387754.5000,0.0000
75%,2.0000,1.0000,36.0000,1.0000,2023-08-07 00:00:00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,75.0000,73.2500,1.0000,0.0471,19.6649,0.0948,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,581633.7500,0.0000
max,16.0000,1.0000,78.0000,11.0000,2023-10-16 00:00:00,12.1244,12.3864,16336.3700,1.0000,12161.9375,18205.4545,100.0000,11.6461,3560.8696,100.0000,92.5000,26.7500,1.2000,19.6802,0.1942,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,775509.0000,1.0000
std,0.9379,0.4462,21.5360,0.3360,NaN,0.5092,0.5530,248.3238,0.0511,179.7530,285.7929,3.0897,0.4691,4.3587,26.1175,25.8293,3.7957,0.1503,0.1293,0.1049,0.1262,0.1260,0.1258,0.1253,0.1249,0.1244,0.1230,0.0535,0.0620,0.0508,0.0933,0.1039,223871.7663,0.0551


In [87]:
# ==============================================================================
# SECTION 4: DATA PREPARATION FOR MODELING
# ==============================================================================

logger = logging.getLogger(__name__)
logger.info("Executing Section 4 (Data Preparation)...")


class DataPreparator:
    """
    Prepares feature-engineered panel for machine learning modeling.
    
    This class implements industry-standard data preparation workflows including
    temporal train/validation/test splitting, feature matrix construction, class
    imbalance analysis, and data versioning for reproducibility.
    
    Key Responsibilities:
        - Temporal split to prevent data leakage
        - Feature matrix extraction and validation
        - Class imbalance quantification and weight computation
        - Data export with versioning and manifest generation
    
    Design Principles:
        - Temporal integrity: No future information in training
        - Reproducibility: All splits are deterministic and documented
        - Validation: Extensive checks at each stage
        - Production-ready: Outputs compatible with deployment pipelines
    """
    
    def __init__(self, target_horizon: int = 4, random_state: int = 42):
        """
        Initialize data preparator.
        
        Args:
            target_horizon: Prediction horizon (1-4 weeks ahead)
            random_state: Random seed for reproducibility
        """
        self.target_horizon = target_horizon
        self.target_col = f"y_shortage_t_plus_{target_horizon}"
        self.random_state = random_state
        self.feature_cols = None
        self.categorical_cols = None
        self.numerical_cols = None
        
    def temporal_split(
        self, 
        df_panel: pd.DataFrame,
        train_end: str = "2023-08-06",
        val_end: str = "2023-09-10"
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """
        Split panel into train/validation/test using temporal cutoffs.
        
        This prevents data leakage by ensuring training data always precedes
        validation data, which precedes test data. This mimics production
        deployment where models predict future outcomes.
        
        Args:
            df_panel: Feature-engineered panel from Section 3
            train_end: Last date in training set (inclusive)
            val_end: Last date in validation set (inclusive)
            
        Returns:
            Tuple of (train_df, val_df, test_df)
            
        Validation Checks:
            - No temporal overlap between splits
            - Similar class distributions across splits
            - All DINs present in training appear in val/test
        """
        logger.info("Step 4.1: Temporal train/validation/test split")
        
        df = df_panel.copy()
        df['date_monday'] = pd.to_datetime(df['date_monday'])
        train_end_date = pd.to_datetime(train_end)
        val_end_date = pd.to_datetime(val_end)
        
        train_df = df[df['date_monday'] <= train_end_date].copy()
        val_df = df[(df['date_monday'] > train_end_date) & 
                    (df['date_monday'] <= val_end_date)].copy()
        test_df = df[df['date_monday'] > val_end_date].copy()
        
        logger.info(f"  Training:   {train_df['date_monday'].min().date()} to "
                   f"{train_df['date_monday'].max().date()} ({len(train_df):,} rows)")
        logger.info(f"  Validation: {val_df['date_monday'].min().date()} to "
                   f"{val_df['date_monday'].max().date()} ({len(val_df):,} rows)")
        logger.info(f"  Test:       {test_df['date_monday'].min().date()} to "
                   f"{test_df['date_monday'].max().date()} ({len(test_df):,} rows)")
        
        assert len(train_df) > 0, "Training set is empty"
        assert len(val_df) > 0, "Validation set is empty"
        assert len(test_df) > 0, "Test set is empty"
        assert train_df['date_monday'].max() < val_df['date_monday'].min(), "Temporal overlap: train/val"
        assert val_df['date_monday'].max() < test_df['date_monday'].min(), "Temporal overlap: val/test"
        
        train_pct = len(train_df) / len(df) * 100
        val_pct = len(val_df) / len(df) * 100
        test_pct = len(test_df) / len(df) * 100
        
        logger.info(f"  Split proportions: Train {train_pct:.1f}%, Val {val_pct:.1f}%, Test {test_pct:.1f}%")
        
        return train_df, val_df, test_df
    
    def validate_target(self, df: pd.DataFrame) -> Dict[str, Any]:
        """
        Validate target variable and compute statistics.
        
        Args:
            df: DataFrame with target column
            
        Returns:
            Dictionary with target statistics
        """
        if self.target_col not in df.columns:
            raise ValueError(f"Target column {self.target_col} not found in dataframe")
        
        target = df[self.target_col]
        
        if target.isnull().any():
            raise ValueError(f"Target column {self.target_col} contains NaN values")
        
        n_positive = (target == 1).sum()
        n_negative = (target == 0).sum()
        positive_rate = n_positive / len(target)
        
        stats = {
            'n_samples': len(target),
            'n_positive': int(n_positive),
            'n_negative': int(n_negative),
            'positive_rate': float(positive_rate),
            'class_balance_ratio': float(n_negative / n_positive) if n_positive > 0 else float('inf')
        }
        
        return stats
    
    def identify_features(self, df_panel: pd.DataFrame) -> None:
        """
        Identify and categorize features for modeling.
        
        Separates features into categorical and numerical types. Excludes
        keys, dates, and target variables from feature matrix.
        
        Args:
            df_panel: Full panel with all columns
            
        Side Effects:
            Sets self.feature_cols, self.categorical_cols, self.numerical_cols
        """
        logger.info("Step 4.3: Feature matrix identification")
        
        exclude_cols = [
            'DIN_KEY', 'date_monday', 'FINAL_MOLECULE', 'ATC_LEVEL3',
            'y_shortage_t', 'y_shortage_t_lag1', 'y_shortage_rate_prev4w',
            'y_shortage_t_plus_1', 'y_shortage_t_plus_2', 
            'y_shortage_t_plus_3', 'y_shortage_t_plus_4'
        ]
        
        all_cols = df_panel.columns.tolist()
        feature_cols = [c for c in all_cols if c not in exclude_cols]
        
        categorical_cols = []
        numerical_cols = []
        
        for col in feature_cols:
            if df_panel[col].dtype == 'object' or df_panel[col].dtype.name == 'string':
                categorical_cols.append(col)
            elif df_panel[col].dtype.name == 'category':
                categorical_cols.append(col)
            else:
                numerical_cols.append(col)
        
        self.feature_cols = feature_cols
        self.categorical_cols = categorical_cols
        self.numerical_cols = numerical_cols
        
        logger.info(f"  Total features: {len(feature_cols)}")
        logger.info(f"  Categorical:    {len(categorical_cols)} {categorical_cols}")
        logger.info(f"  Numerical:      {len(numerical_cols)}")
        
    def prepare_features(
        self, 
        train_df: pd.DataFrame,
        val_df: pd.DataFrame,
        test_df: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
        """
        Extract feature matrices and target vectors.
        
        Args:
            train_df: Training split
            val_df: Validation split
            test_df: Test split
            
        Returns:
            Tuple of (X_train, y_train, X_val, y_val, X_test, y_test)
            
        Validation Checks:
            - No NaN values in features
            - No infinite values
            - Feature dtypes consistent across splits
            - Target is binary (0/1)
        """
        logger.info("Step 4.3 (continued): Extract feature matrices")
        
        if self.feature_cols is None:
            raise ValueError("Must call identify_features() first")
        
        X_train = train_df[self.feature_cols].copy()
        y_train = train_df[self.target_col].copy()
        
        X_val = val_df[self.feature_cols].copy()
        y_val = val_df[self.target_col].copy()
        
        X_test = test_df[self.feature_cols].copy()
        y_test = test_df[self.target_col].copy()
        
        for split_name, X, y in [('Train', X_train, y_train), 
                                  ('Val', X_val, y_val), 
                                  ('Test', X_test, y_test)]:
            
            nan_counts = X.isnull().sum()
            if nan_counts.any():
                logger.warning(f"  {split_name} NaN counts:\n{nan_counts[nan_counts > 0]}")
                raise ValueError(f"{split_name} features contain NaN values")
            
            numeric_cols = X.select_dtypes(include=[np.number]).columns
            float_cols = [c for c in numeric_cols if X[c].dtype in ['float64', 'float32']]
            if len(float_cols) > 0:
                inf_mask = np.isinf(X[float_cols].values).any()
                if inf_mask:
                    raise ValueError(f"{split_name} features contain infinite values")
            
            if not y.isin([0, 1]).all():
                raise ValueError(f"{split_name} target contains non-binary values")
        
        logger.info(f"  Feature matrices extracted successfully")
        logger.info(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
        logger.info(f"  X_val:   {X_val.shape}, y_val:   {y_val.shape}")
        logger.info(f"  X_test:  {X_test.shape}, y_test:  {y_test.shape}")
        
        return X_train, y_train, X_val, y_val, X_test, y_test
    
    def analyze_imbalance(
        self,
        y_train: pd.Series,
        y_val: pd.Series,
        y_test: pd.Series
    ) -> Dict[str, Any]:
        """
        Analyze class imbalance and compute weighting strategies.
        
        Computes multiple class weighting schemes for handling severe imbalance.
        Models can select the most appropriate strategy during hyperparameter tuning.
        
        Args:
            y_train: Training labels
            y_val: Validation labels
            y_test: Test labels
            
        Returns:
            Dictionary with imbalance statistics and class weights
        """
        logger.info("Step 4.4: Class imbalance analysis")
        
        train_stats = self.validate_target(pd.DataFrame({self.target_col: y_train}))
        val_stats = self.validate_target(pd.DataFrame({self.target_col: y_val}))
        test_stats = self.validate_target(pd.DataFrame({self.target_col: y_test}))
        
        logger.info(f"  Training:   {train_stats['n_positive']:,} positive ({train_stats['positive_rate']:.4f})")
        logger.info(f"  Validation: {val_stats['n_positive']:,} positive ({val_stats['positive_rate']:.4f})")
        logger.info(f"  Test:       {test_stats['n_positive']:,} positive ({test_stats['positive_rate']:.4f})")
        
        n_samples = len(y_train)
        n_positive = train_stats['n_positive']
        n_negative = train_stats['n_negative']
        
        weights_inv = {
            0: n_samples / (2 * n_negative),
            1: n_samples / (2 * n_positive)
        }
        
        weights_sqrt = {
            0: np.sqrt(n_samples / (2 * n_negative)),
            1: np.sqrt(n_samples / (2 * n_positive))
        }
        
        weights_log = {
            0: np.log1p(n_samples / (2 * n_negative)),
            1: np.log1p(n_samples / (2 * n_positive))
        }
        
        scale_pos_weight = n_negative / n_positive
        
        imbalance_report = {
            'train': train_stats,
            'val': val_stats,
            'test': test_stats,
            'class_weights': {
                'inverse': weights_inv,
                'sqrt': weights_sqrt,
                'log': weights_log
            },
            'scale_pos_weight': float(scale_pos_weight)
        }
        
        logger.info(f"  Class weight (inverse):  neg={weights_inv[0]:.3f}, pos={weights_inv[1]:.3f}")
        logger.info(f"  Class weight (sqrt):     neg={weights_sqrt[0]:.3f}, pos={weights_sqrt[1]:.3f}")
        logger.info(f"  Class weight (log):      neg={weights_log[0]:.3f}, pos={weights_log[1]:.3f}")
        logger.info(f"  Scale pos weight (XGBoost/LightGBM): {scale_pos_weight:.2f}")
        
        return imbalance_report
    
    def export_datasets(
        self,
        train_df: pd.DataFrame,
        val_df: pd.DataFrame,
        test_df: pd.DataFrame,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_val: pd.DataFrame,
        y_val: pd.Series,
        X_test: pd.DataFrame,
        y_test: pd.Series,
        imbalance_report: Dict[str, Any],
        output_dir: str = "data/modeling"
    ) -> None:
        """
        Export processed datasets with versioning and manifest.
        
        Saves data in Parquet format for efficiency and creates a manifest
        documenting dataset properties for reproducibility.
        
        Args:
            train_df, val_df, test_df: Full dataframes with all columns
            X_train, y_train, etc: Feature matrices and targets
            imbalance_report: Class imbalance analysis results
            output_dir: Directory for outputs
        """
        logger.info("Step 4.5: Data export and versioning")
        
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        train_df.to_parquet(output_path / "train_full.parquet", index=False)
        val_df.to_parquet(output_path / "val_full.parquet", index=False)
        test_df.to_parquet(output_path / "test_full.parquet", index=False)
        
        X_train.to_parquet(output_path / "X_train.parquet", index=False)
        X_val.to_parquet(output_path / "X_val.parquet", index=False)
        X_test.to_parquet(output_path / "X_test.parquet", index=False)
        
        y_train.to_frame().to_parquet(output_path / "y_train.parquet", index=False)
        y_val.to_frame().to_parquet(output_path / "y_val.parquet", index=False)
        y_test.to_frame().to_parquet(output_path / "y_test.parquet", index=False)
        
        with open(output_path / "imbalance_report.json", 'w') as f:
            json.dump(imbalance_report, f, indent=2)
        
        manifest = {
            'target_horizon': self.target_horizon,
            'target_column': self.target_col,
            'random_state': self.random_state,
            'n_features': len(self.feature_cols),
            'feature_columns': self.feature_cols,
            'categorical_features': self.categorical_cols,
            'numerical_features': self.numerical_cols,
            'train_shape': list(X_train.shape),
            'val_shape': list(X_val.shape),
            'test_shape': list(X_test.shape),
            'train_dates': {
                'start': str(train_df['date_monday'].min().date()),
                'end': str(train_df['date_monday'].max().date())
            },
            'val_dates': {
                'start': str(val_df['date_monday'].min().date()),
                'end': str(val_df['date_monday'].max().date())
            },
            'test_dates': {
                'start': str(test_df['date_monday'].min().date()),
                'end': str(test_df['date_monday'].max().date())
            }
        }
        
        with open(output_path / "data_manifest.json", 'w') as f:
            json.dump(manifest, f, indent=2)
        
        logger.info(f"  Datasets exported to: {output_path}")
        logger.info(f"  Files created:")
        logger.info(f"    - train_full.parquet, val_full.parquet, test_full.parquet")
        logger.info(f"    - X_train.parquet, X_val.parquet, X_test.parquet")
        logger.info(f"    - y_train.parquet, y_val.parquet, y_test.parquet")
        logger.info(f"    - imbalance_report.json")
        logger.info(f"    - data_manifest.json")
    
    def run(self, df_panel: pd.DataFrame, output_dir: str = "data/modeling") -> Dict[str, Any]:
        """
        Execute complete data preparation pipeline.
        
        Args:
            df_panel: Feature-engineered panel from Section 3
            output_dir: Directory for outputs
            
        Returns:
            Dictionary with all prepared datasets and metadata
        """
        print("\n" + "=" * 80)
        print("SECTION 4: DATA PREPARATION FOR MODELING")
        print("=" * 80)
        
        train_df, val_df, test_df = self.temporal_split(df_panel)
        
        self.identify_features(df_panel)
        
        X_train, y_train, X_val, y_val, X_test, y_test = self.prepare_features(
            train_df, val_df, test_df
        )
        
        imbalance_report = self.analyze_imbalance(y_train, y_val, y_test)
        
        self.export_datasets(
            train_df, val_df, test_df,
            X_train, y_train, X_val, y_val, X_test, y_test,
            imbalance_report, output_dir
        )
        
        print("\n" + "=" * 80)
        print("SECTION 4 COMPLETE")
        print("=" * 80)
        print(f"Target: {self.target_col}")
        print(f"Features: {len(self.feature_cols)} ({len(self.categorical_cols)} categorical, {len(self.numerical_cols)} numerical)")
        print(f"Training samples: {len(X_train):,} ({imbalance_report['train']['positive_rate']:.4f} positive)")
        print(f"Validation samples: {len(X_val):,} ({imbalance_report['val']['positive_rate']:.4f} positive)")
        print(f"Test samples: {len(X_test):,} ({imbalance_report['test']['positive_rate']:.4f} positive)")
        print("=" * 80)
        
        return {
            'train_df': train_df,
            'val_df': val_df,
            'test_df': test_df,
            'X_train': X_train,
            'y_train': y_train,
            'X_val': X_val,
            'y_val': y_val,
            'X_test': X_test,
            'y_test': y_test,
            'imbalance_report': imbalance_report,
            'feature_cols': self.feature_cols,
            'categorical_cols': self.categorical_cols,
            'numerical_cols': self.numerical_cols
        }


preparator = DataPreparator(target_horizon=4, random_state=42)
prepared_data = preparator.run(df_panel, output_dir="data/modeling")

train_df = prepared_data['train_df']
val_df = prepared_data['val_df']
test_df = prepared_data['test_df']
X_train = prepared_data['X_train']
y_train = prepared_data['y_train']
X_val = prepared_data['X_val']
y_val = prepared_data['y_val']
X_test = prepared_data['X_test']
y_test = prepared_data['y_test']
imbalance_report = prepared_data['imbalance_report']
feature_cols = prepared_data['feature_cols']
categorical_cols = prepared_data['categorical_cols']
numerical_cols = prepared_data['numerical_cols']

2026-01-17 14:21:26,511 - [INFO] - Executing Section 4 (Data Preparation)...

SECTION 4: DATA PREPARATION FOR MODELING
2026-01-17 14:21:26,516 - [INFO] - Step 4.1: Temporal train/validation/test split
2026-01-17 14:21:27,271 - [INFO] -   Training:   2023-01-02 to 2023-07-31 (522,629 rows)
2026-01-17 14:21:27,277 - [INFO] -   Validation: 2023-08-07 to 2023-09-04 (84,295 rows)
2026-01-17 14:21:27,280 - [INFO] -   Test:       2023-09-11 to 2023-10-16 (101,154 rows)
2026-01-17 14:21:27,285 - [INFO] -   Split proportions: Train 73.8%, Val 11.9%, Test 14.3%
2026-01-17 14:21:27,327 - [INFO] - Step 4.3: Feature matrix identification
2026-01-17 14:21:27,332 - [INFO] -   Total features: 28
2026-01-17 14:21:27,334 - [INFO] -   Categorical:    2 ['ATC_LEVEL1', 'vendor_group']
2026-01-17 14:21:27,339 - [INFO] -   Numerical:      26
2026-01-17 14:21:27,352 - [INFO] - Step 4.3 (continued): Extract feature matrices
2026-01-17 14:21:27,860 - [INFO] -   Feature matrices extracted successfully
2026-01-17

In [88]:
# ==============================================================================
# SECTION 4.5: VALIDATION CHECKS
# ==============================================================================

print("\n" + "=" * 80)
print("SECTION 4 VALIDATION CHECKS")
print("=" * 80)

validation_passed = True
validation_results = []


def check(condition: bool, test_name: str, details: str = "") -> bool:
    """Helper function to perform validation check."""
    global validation_passed
    status = "PASS" if condition else "FAIL"
    symbol = "✓" if condition else "✗"
    
    result = {
        'test': test_name,
        'status': status,
        'details': details
    }
    validation_results.append(result)
    
    print(f"{symbol} {test_name}: {status}")
    if details:
        print(f"  {details}")
    
    if not condition:
        validation_passed = False
    
    return condition


print("\n" + "-" * 80)
print("1. TEMPORAL INTEGRITY CHECKS")
print("-" * 80)

train_dates = pd.to_datetime(train_df['date_monday'])
val_dates = pd.to_datetime(val_df['date_monday'])
test_dates = pd.to_datetime(test_df['date_monday'])

check(
    train_dates.max() < val_dates.min(),
    "No temporal overlap between train and validation",
    f"Train ends: {train_dates.max().date()}, Val starts: {val_dates.min().date()}"
)

check(
    val_dates.max() < test_dates.min(),
    "No temporal overlap between validation and test",
    f"Val ends: {val_dates.max().date()}, Test starts: {test_dates.min().date()}"
)

check(
    len(train_df) + len(val_df) + len(test_df) == len(df_panel),
    "All rows accounted for in splits",
    f"Original: {len(df_panel):,}, Sum of splits: {len(train_df) + len(val_df) + len(test_df):,}"
)


print("\n" + "-" * 80)
print("2. FEATURE MATRIX CHECKS")
print("-" * 80)

check(
    X_train.shape[1] == X_val.shape[1] == X_test.shape[1],
    "Consistent feature count across splits",
    f"Train: {X_train.shape[1]}, Val: {X_val.shape[1]}, Test: {X_test.shape[1]}"
)

check(
    list(X_train.columns) == list(X_val.columns) == list(X_test.columns),
    "Identical feature columns across splits"
)

check(
    X_train.isnull().sum().sum() == 0,
    "No NaN values in training features",
    f"NaN count: {X_train.isnull().sum().sum()}"
)

check(
    X_val.isnull().sum().sum() == 0,
    "No NaN values in validation features",
    f"NaN count: {X_val.isnull().sum().sum()}"
)

check(
    X_test.isnull().sum().sum() == 0,
    "No NaN values in test features",
    f"NaN count: {X_test.isnull().sum().sum()}"
)

numerical_features = X_train.select_dtypes(include=[np.number]).columns
float_features = [c for c in numerical_features if X_train[c].dtype in ['float64', 'float32']]

if len(float_features) > 0:
    inf_train = np.isinf(X_train[float_features].values).any()
    inf_val = np.isinf(X_val[float_features].values).any()
    inf_test = np.isinf(X_test[float_features].values).any()
else:
    inf_train = inf_val = inf_test = False

check(
    not (inf_train or inf_val or inf_test),
    "No infinite values in features",
    f"Train: {inf_train}, Val: {inf_val}, Test: {inf_test}"
)


print("\n" + "-" * 80)
print("3. TARGET VARIABLE CHECKS")
print("-" * 80)

check(
    y_train.isin([0, 1]).all(),
    "Training target is binary (0/1)",
    f"Unique values: {sorted(y_train.unique())}"
)

check(
    y_val.isin([0, 1]).all(),
    "Validation target is binary (0/1)",
    f"Unique values: {sorted(y_val.unique())}"
)

check(
    y_test.isin([0, 1]).all(),
    "Test target is binary (0/1)",
    f"Unique values: {sorted(y_test.unique())}"
)

train_pos_rate = y_train.mean()
val_pos_rate = y_val.mean()
test_pos_rate = y_test.mean()

check(
    0.01 < train_pos_rate < 0.05,
    "Training positive rate in expected range (1-5%)",
    f"Actual: {train_pos_rate:.4f}"
)

pos_rate_diff_val = abs(train_pos_rate - val_pos_rate)
pos_rate_diff_test = abs(train_pos_rate - test_pos_rate)

check(
    pos_rate_diff_val < 0.005,
    "Validation positive rate similar to training (±0.5%)",
    f"Train: {train_pos_rate:.4f}, Val: {val_pos_rate:.4f}, Diff: {pos_rate_diff_val:.4f}"
)

check(
    pos_rate_diff_test < 0.005,
    "Test positive rate similar to training (±0.5%)",
    f"Train: {train_pos_rate:.4f}, Test: {test_pos_rate:.4f}, Diff: {pos_rate_diff_test:.4f}"
)


print("\n" + "-" * 80)
print("4. CLASS IMBALANCE CHECKS")
print("-" * 80)

scale_pos_weight = imbalance_report['scale_pos_weight']
check(
    scale_pos_weight > 10,
    "Severe imbalance detected (scale_pos_weight > 10)",
    f"scale_pos_weight = {scale_pos_weight:.2f}"
)

inv_weights = imbalance_report['class_weights']['inverse']
check(
    inv_weights[1] > inv_weights[0],
    "Positive class has higher weight than negative",
    f"Neg: {inv_weights[0]:.2f}, Pos: {inv_weights[1]:.2f}"
)

check(
    inv_weights[1] < 100,
    "Positive class weight is reasonable (< 100)",
    f"Pos weight: {inv_weights[1]:.2f}"
)


print("\n" + "-" * 80)
print("5. DATA EXPORT CHECKS")
print("-" * 80)

output_path = Path("data/modeling")
required_files = [
    "train_full.parquet", "val_full.parquet", "test_full.parquet",
    "X_train.parquet", "X_val.parquet", "X_test.parquet",
    "y_train.parquet", "y_val.parquet", "y_test.parquet",
    "imbalance_report.json", "data_manifest.json"
]

all_files_exist = all((output_path / f).exists() for f in required_files)
check(
    all_files_exist,
    "All required files exported",
    f"Location: {output_path}"
)

if (output_path / "data_manifest.json").exists():
    with open(output_path / "data_manifest.json", 'r') as f:
        manifest = json.load(f)
    
    check(
        manifest['n_features'] == len(feature_cols),
        "Manifest feature count matches actual",
        f"Manifest: {manifest['n_features']}, Actual: {len(feature_cols)}"
    )
    
    check(
        manifest['train_shape'][0] == len(X_train),
        "Manifest train shape matches actual",
        f"Manifest: {manifest['train_shape']}, Actual: {X_train.shape}"
    )


print("\n" + "-" * 80)
print("6. FEATURE CATEGORIZATION CHECKS")
print("-" * 80)

check(
    len(categorical_cols) > 0,
    "Categorical features identified",
    f"Count: {len(categorical_cols)}"
)

check(
    len(numerical_cols) > 0,
    "Numerical features identified",
    f"Count: {len(numerical_cols)}"
)

check(
    len(categorical_cols) + len(numerical_cols) == len(feature_cols),
    "All features categorized (categorical + numerical = total)",
    f"Cat: {len(categorical_cols)}, Num: {len(numerical_cols)}, Total: {len(feature_cols)}"
)

expected_cat_features = ['ATC_LEVEL1', 'vendor_group']
actual_cat_features = [c for c in expected_cat_features if c in categorical_cols]
check(
    len(actual_cat_features) > 0,
    "Expected categorical features present",
    f"Found: {actual_cat_features}"
)


print("\n" + "-" * 80)
print("7. SPLIT PROPORTION CHECKS")
print("-" * 80)

total_samples = len(df_panel)
train_pct = len(train_df) / total_samples
val_pct = len(val_df) / total_samples
test_pct = len(test_df) / total_samples

check(
    0.65 <= train_pct <= 0.75,
    "Training set is ~70% of data",
    f"Actual: {train_pct:.1%}"
)

check(
    0.10 <= val_pct <= 0.20,
    "Validation set is ~15% of data",
    f"Actual: {val_pct:.1%}"
)

check(
    0.10 <= test_pct <= 0.20,
    "Test set is ~15% of data",
    f"Actual: {test_pct:.1%}"
)


print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)

passed_count = sum(1 for r in validation_results if r['status'] == 'PASS')
total_count = len(validation_results)

print(f"Passed: {passed_count}/{total_count}")
print(f"Status: {'ALL CHECKS PASSED' if validation_passed else 'SOME CHECKS FAILED'}")

if not validation_passed:
    print("\nFailed checks:")
    for r in validation_results:
        if r['status'] == 'FAIL':
            print(f"  - {r['test']}")
            if r['details']:
                print(f"    {r['details']}")

print("\n" + "=" * 80)
print("GATE CRITERIA FOR SECTION 4")
print("=" * 80)

gate_criteria = {
    'temporal_integrity': all([
        train_dates.max() < val_dates.min(),
        val_dates.max() < test_dates.min()
    ]),
    'no_data_leakage': all([
        train_dates.max() < val_dates.min(),
        val_dates.max() < test_dates.min()
    ]),
    'class_distribution_documented': imbalance_report is not None,
    'no_missing_values': all([
        X_train.isnull().sum().sum() == 0,
        X_val.isnull().sum().sum() == 0,
        X_test.isnull().sum().sum() == 0
    ]),
    'all_deliverables_generated': all_files_exist
}

for criterion, passed in gate_criteria.items():
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {criterion.replace('_', ' ').title()}")

all_gates_passed = all(gate_criteria.values())

print("\n" + "=" * 80)
if all_gates_passed and validation_passed:
    print("SECTION 4 VALIDATION: PASSED")
    print("Ready to proceed to Section 5 (Baseline Models)")
else:
    print("SECTION 4 VALIDATION: FAILED")
    print("Please fix issues before proceeding")
print("=" * 80)


SECTION 4 VALIDATION CHECKS

--------------------------------------------------------------------------------
1. TEMPORAL INTEGRITY CHECKS
--------------------------------------------------------------------------------
✓ No temporal overlap between train and validation: PASS
  Train ends: 2023-07-31, Val starts: 2023-08-07
✓ No temporal overlap between validation and test: PASS
  Val ends: 2023-09-04, Test starts: 2023-09-11
✓ All rows accounted for in splits: PASS
  Original: 708,078, Sum of splits: 708,078

--------------------------------------------------------------------------------
2. FEATURE MATRIX CHECKS
--------------------------------------------------------------------------------
✓ Consistent feature count across splits: PASS
  Train: 28, Val: 28, Test: 28
✓ Identical feature columns across splits: PASS
✓ No NaN values in training features: PASS
  NaN count: 0
✓ No NaN values in validation features: PASS
  NaN count: 0
✓ No NaN values in test features: PASS
  NaN count: 

In [91]:
# ==============================================================================
# SECTION 5: BASELINE MODELS AND EVALUATION FRAMEWORK
# ==============================================================================

logger = logging.getLogger(__name__)
logger.info("Executing Section 5 (Baseline Models)...")


class ModelEvaluator:
    """
    Standardized evaluation framework for shortage prediction models.
    
    This class provides consistent metrics computation and reporting across all
    models (baselines, traditional ML, deep learning). It handles class imbalance
    appropriately by prioritizing PR-AUC over ROC-AUC and includes business metrics.
    
    Key Responsibilities:
        - Compute classification metrics (PR-AUC, ROC-AUC, F1, Precision, Recall)
        - Generate confusion matrices and classification reports
        - Create visualization plots (PR curves, ROC curves)
        - Save results in standardized format
    
    Design Principles:
        - Imbalance-aware: PR-AUC as primary metric
        - Business-aligned: Include lead time and coverage metrics
        - Reproducible: All results saved with timestamps
        - Comparable: Consistent format across all models
    """
    
    def __init__(self, output_dir: str = "models/baselines"):
        """
        Initialize evaluator.
        
        Args:
            output_dir: Directory for saving evaluation results
        """
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
    def compute_metrics(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        y_prob: np.ndarray,
        model_name: str
    ) -> Dict[str, Any]:
        """
        Compute comprehensive evaluation metrics.
        
        Args:
            y_true: True labels (0/1)
            y_pred: Predicted labels (0/1)
            y_prob: Predicted probabilities for positive class
            model_name: Name of model being evaluated
            
        Returns:
            Dictionary with all metrics
        """
        metrics = {
            'model_name': model_name,
            'n_samples': len(y_true),
            'n_positive': int(y_true.sum()),
            'positive_rate': float(y_true.mean())
        }
        
        precision, recall, pr_thresholds = precision_recall_curve(y_true, y_prob)
        fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)
        
        if len(np.unique(y_prob)) <= 1:
            metrics['pr_auc'] = float(y_true.mean())
            metrics['average_precision'] = float(y_true.mean())
        else:
            metrics['pr_auc'] = auc(recall, precision)
            metrics['average_precision'] = average_precision_score(y_true, y_prob)
        
        metrics['roc_auc'] = auc(fpr, tpr)
        
        metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
        metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
        
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        metrics['confusion_matrix'] = {
            'tn': int(tn), 'fp': int(fp),
            'fn': int(fn), 'tp': int(tp)
        }
        metrics['false_alarm_rate'] = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0
        metrics['coverage_rate'] = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
        
        metrics['pr_curve'] = {
            'precision': precision.tolist(),
            'recall': recall.tolist(),
            'thresholds': pr_thresholds.tolist()
        }
        metrics['roc_curve'] = {
            'fpr': fpr.tolist(),
            'tpr': tpr.tolist(),
            'thresholds': roc_thresholds.tolist()
        }
        
        return metrics
    
    def save_metrics(self, metrics: Dict[str, Any], filename: str) -> None:
        """
        Save metrics to JSON file.
        
        Args:
            metrics: Metrics dictionary
            filename: Output filename
        """
        output_path = self.output_dir / filename
        with open(output_path, 'w') as f:
            json.dump(metrics, f, indent=2)
        logger.info(f"  Metrics saved to: {output_path}")
    
    def plot_pr_curve(
        self,
        metrics_list: List[Dict[str, Any]],
        filename: str = "pr_curves_comparison.png"
    ) -> None:
        """
        Plot PR curves for multiple models.
        
        Args:
            metrics_list: List of metrics dictionaries from different models
            filename: Output filename for plot
        """
        plt.figure(figsize=(10, 6))
        
        for metrics in metrics_list:
            precision = np.array(metrics['pr_curve']['precision'])
            recall = np.array(metrics['pr_curve']['recall'])
            pr_auc = metrics['pr_auc']
            label = f"{metrics['model_name']} (PR-AUC={pr_auc:.3f})"
            plt.plot(recall, precision, label=label, linewidth=2)
        
        plt.xlabel('Recall', fontsize=12)
        plt.ylabel('Precision', fontsize=12)
        plt.title('Precision-Recall Curves - Baseline Models', fontsize=14, fontweight='bold')
        plt.legend(loc='best', fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        output_path = self.output_dir / filename
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"  PR curves saved to: {output_path}")
    
    def generate_comparison_table(
        self,
        metrics_list: List[Dict[str, Any]]
    ) -> pd.DataFrame:
        """
        Generate comparison table for multiple models.
        
        Args:
            metrics_list: List of metrics dictionaries
            
        Returns:
            DataFrame with comparison metrics
        """
        comparison_data = []
        for metrics in metrics_list:
            row = {
                'Model': metrics['model_name'],
                'PR-AUC': f"{metrics['pr_auc']:.4f}",
                'ROC-AUC': f"{metrics['roc_auc']:.4f}",
                'Precision': f"{metrics['precision']:.4f}",
                'Recall': f"{metrics['recall']:.4f}",
                'F1': f"{metrics['f1']:.4f}",
                'TP': metrics['confusion_matrix']['tp'],
                'FP': metrics['confusion_matrix']['fp'],
                'FN': metrics['confusion_matrix']['fn'],
                'Coverage': f"{metrics['coverage_rate']:.4f}"
            }
            comparison_data.append(row)
        
        df = pd.DataFrame(comparison_data)
        return df


class LogisticRegressionBaseline:
    """
    Simple logistic regression baseline.
    
    This is the most basic ML baseline - a logistic regression model using all
    available features with balanced class weights to handle imbalance. This
    establishes a simple linear baseline that more complex models must beat.
    """
    
    def __init__(self, class_weight_strategy: str = 'balanced'):
        """
        Initialize logistic regression baseline.
        
        Args:
            class_weight_strategy: Class weighting strategy
        """
        self.name = "Logistic Regression"
        self.class_weight_strategy = class_weight_strategy
        self.model = LogisticRegression(
            class_weight=class_weight_strategy,
            random_state=42,
            max_iter=1000,
            solver='lbfgs'
        )
        self.label_encoders = {}
        
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series) -> None:
        """
        Train logistic regression on all features.
        
        Args:
            X_train: Training features
            y_train: Training labels
        """
        X_encoded = X_train.copy()
        
        for col in X_train.columns:
            if X_train[col].dtype == 'object' or X_train[col].dtype.name == 'string':
                le = LabelEncoder()
                X_encoded[col] = le.fit_transform(X_train[col].astype(str))
                self.label_encoders[col] = le
        
        self.model.fit(X_encoded, y_train)
        logger.info(f"  Logistic regression trained on {X_train.shape[1]} features")
        
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """Predict probabilities."""
        X_encoded = X.copy()
        
        for col in X.columns:
            if col in self.label_encoders:
                le = self.label_encoders[col]
                X_encoded[col] = X[col].astype(str).map(
                    lambda x: le.transform([x])[0] if x in le.classes_ else -1
                )
        
        return self.model.predict_proba(X_encoded)
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Binary predictions."""
        X_encoded = X.copy()
        
        for col in X.columns:
            if col in self.label_encoders:
                le = self.label_encoders[col]
                X_encoded[col] = X[col].astype(str).map(
                    lambda x: le.transform([x])[0] if x in le.classes_ else -1
                )
        
        return self.model.predict(X_encoded)


class MajorityClassBaseline:
    """
    Majority class baseline: Always predicts negative (no shortage).
    
    This is the absolute simplest baseline. Given the severe class imbalance
    (98.5% negative), this achieves high accuracy but 0% recall. It establishes
    the floor that any useful model must beat.
    """
    
    def __init__(self):
        self.name = "Majority Class"
        self.majority_class = 0
        
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series) -> None:
        """Determine majority class."""
        self.majority_class = int(y_train.mode()[0])
        logger.info(f"  Majority class: {self.majority_class}")
        
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Predict majority class with certainty.
        
        Returns:
            Probability array with 1.0 for majority class
        """
        n_samples = len(X)
        proba = np.zeros((n_samples, 2))
        proba[:, self.majority_class] = 1.0
        return proba
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Always predict majority class."""
        return np.full(len(X), self.majority_class)


class RandomBaseline:
    """
    Random baseline: Predicts based on training set positive rate.
    
    This baseline randomly predicts positive/negative with probabilities matching
    the training set distribution. It represents random guessing calibrated to
    the class distribution.
    """
    
    def __init__(self):
        self.name = "Random (Stratified)"
        self.positive_rate = 0.5
        
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series) -> None:
        """Learn positive rate from training data."""
        self.positive_rate = float(y_train.mean())
        logger.info(f"  Training positive rate: {self.positive_rate:.4f}")
        
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Predict with constant probability equal to training positive rate.
        
        Returns:
            Probability array with constant positive probability
        """
        n_samples = len(X)
        proba_positive = np.full(n_samples, self.positive_rate)
        proba_negative = 1 - proba_positive
        return np.column_stack([proba_negative, proba_positive])
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Predict randomly based on positive rate."""
        np.random.seed(42)
        return np.random.binomial(1, self.positive_rate, size=len(X))


print("\n" + "=" * 80)
print("SECTION 5: BASELINE MODELS AND EVALUATION FRAMEWORK")
print("=" * 80)

evaluator = ModelEvaluator(output_dir="models/baselines")

print("\nStep 5.1: Initializing baseline models...")
majority = MajorityClassBaseline()
random_baseline = RandomBaseline()
logistic = LogisticRegressionBaseline()

print("\nStep 5.2: Training baselines...")
print("-" * 80)

majority.fit(X_train, y_train)

random_baseline.fit(X_train, y_train)

logistic.fit(X_train, y_train)

print("\nStep 5.3: Evaluating on validation set...")
print("-" * 80)

all_metrics = []

for model in [majority, random_baseline, logistic]:
    print(f"\nEvaluating: {model.name}")
    
    y_prob = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)
    
    metrics = evaluator.compute_metrics(
        y_true=y_val.values,
        y_pred=y_pred,
        y_prob=y_prob,
        model_name=model.name
    )
    
    evaluator.save_metrics(
        metrics, 
        f"baseline_{model.name.lower().replace(' ', '_').replace('(', '').replace(')', '')}.json"
    )
    all_metrics.append(metrics)
    
    print(f"  PR-AUC:    {metrics['pr_auc']:.4f}")
    print(f"  ROC-AUC:   {metrics['roc_auc']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Coverage:  {metrics['coverage_rate']:.4f}")

print("\n" + "-" * 80)
print("Step 5.4: Generating comparison report...")
print("-" * 80)

comparison_df = evaluator.generate_comparison_table(all_metrics)
print("\n" + str(comparison_df.to_string(index=False)))

comparison_df.to_csv(evaluator.output_dir / "baseline_comparison.csv", index=False)
logger.info(f"  Comparison table saved to: {evaluator.output_dir / 'baseline_comparison.csv'}")

evaluator.plot_pr_curve(all_metrics, filename="baseline_pr_curves.png")

print("\n" + "=" * 80)
print("SECTION 5 COMPLETE")
print("=" * 80)

lr_metrics = [m for m in all_metrics if m['model_name'] == 'Logistic Regression'][0]
print(f"\nLogistic Regression Baseline Performance:")
print(f"  PR-AUC:  {lr_metrics['pr_auc']:.4f}")
print(f"  ROC-AUC: {lr_metrics['roc_auc']:.4f}")
print(f"  Recall:  {lr_metrics['recall']:.4f} (Coverage: {lr_metrics['recall']*100:.1f}% of shortages)")
print(f"\nPerformance floor established!")
print(f"Traditional ML models (Section 6) must exceed PR-AUC > {lr_metrics['pr_auc']:.4f}")
print("=" * 80)

2026-01-17 16:26:23,736 - [INFO] - Executing Section 5 (Baseline Models)...

SECTION 5: BASELINE MODELS AND EVALUATION FRAMEWORK

Step 5.1: Initializing baseline models...

Step 5.2: Training baselines...
--------------------------------------------------------------------------------
2026-01-17 16:26:23,754 - [INFO] -   Majority class: 0
2026-01-17 16:26:23,758 - [INFO] -   Training positive rate: 0.0155
2026-01-17 16:27:09,204 - [INFO] -   Logistic regression trained on 28 features

Step 5.3: Evaluating on validation set...
--------------------------------------------------------------------------------

Evaluating: Majority Class
2026-01-17 16:27:09,289 - [INFO] -   Metrics saved to: models/baselines/baseline_majority_class.json
  PR-AUC:    0.0186
  ROC-AUC:   0.5000
  Precision: 0.0000
  Recall:    0.0000
  F1:        0.0000
  Coverage:  0.0000

Evaluating: Random (Stratified)
2026-01-17 16:27:09,335 - [INFO] -   Metrics saved to: models/baselines/baseline_random_stratified.json
 

In [97]:
# ==============================================================================
# SECTION 6: TRADITIONAL ML MODELS (TREE-BASED)
# ==============================================================================

logger = logging.getLogger(__name__)
logger.info("Executing Section 6 (Traditional ML Models)...")

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    logger.warning("LightGBM not available. Install with: pip install lightgbm")


class FeatureEncoder:
    """
    Handle categorical feature encoding for tree-based models.
    
    Tree-based models (XGBoost, LightGBM, RandomForest) handle categorical
    features differently. This class provides consistent encoding across all models.
    """
    
    def __init__(self, method: str = 'label'):
        """
        Initialize encoder.
        
        Args:
            method: Encoding method ('label' for label encoding)
        """
        self.method = method
        self.encoders = {}
        
    def fit_transform(self, X: pd.DataFrame, categorical_cols: List[str]) -> pd.DataFrame:
        """
        Fit encoders and transform categorical features.
        
        Args:
            X: Feature matrix
            categorical_cols: List of categorical column names
            
        Returns:
            Transformed feature matrix
        """
        X_encoded = X.copy()
        
        for col in categorical_cols:
            if col in X_encoded.columns:
                le = LabelEncoder()
                X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
                self.encoders[col] = le
        
        return X_encoded
    
    def transform(self, X: pd.DataFrame, categorical_cols: List[str]) -> pd.DataFrame:
        """
        Transform categorical features using fitted encoders.
        
        Args:
            X: Feature matrix
            categorical_cols: List of categorical column names
            
        Returns:
            Transformed feature matrix
        """
        X_encoded = X.copy()
        
        for col in categorical_cols:
            if col in self.encoders and col in X_encoded.columns:
                le = self.encoders[col]
                X_encoded[col] = X_encoded[col].astype(str).map(
                    lambda x: le.transform([x])[0] if x in le.classes_ else -1
                )
        
        return X_encoded


class XGBoostModel:
    """
    XGBoost classifier with class imbalance handling.
    
    XGBoost is a gradient boosting framework that uses level-wise tree growth.
    It's generally more stable than LightGBM but slower. Uses scale_pos_weight
    to handle class imbalance.
    """
    
    def __init__(self, scale_pos_weight: float, params: Optional[Dict] = None):
        """
        Initialize XGBoost model.
        
        Args:
            scale_pos_weight: Weight for positive class (n_negative / n_positive)
            params: Model hyperparameters
        """
        self.name = "XGBoost"
        self.scale_pos_weight = scale_pos_weight
        
        default_params = {
            'max_depth': 6,
            'learning_rate': 0.05,
            'n_estimators': 200,
            'min_child_weight': 1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'gamma': 0,
            'reg_alpha': 0,
            'reg_lambda': 1,
            'random_state': 42,
            'n_jobs': -1,
            'eval_metric': 'aucpr'
        }
        
        if params:
            default_params.update(params)
        
        default_params['scale_pos_weight'] = scale_pos_weight
        
        self.model = xgb.XGBClassifier(**default_params)
        self.params = default_params
        
    def fit(
        self, 
        X_train: pd.DataFrame, 
        y_train: pd.Series,
        X_val: Optional[pd.DataFrame] = None,
        y_val: Optional[pd.Series] = None,
        early_stopping_rounds: int = 20
    ) -> None:
        """
        Train XGBoost model with optional early stopping.
        
        Args:
            X_train: Training features
            y_train: Training labels
            X_val: Validation features (optional, for early stopping)
            y_val: Validation labels (optional, for early stopping)
            early_stopping_rounds: Patience for early stopping
        """
        if X_val is not None and y_val is not None:
            self.model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
        else:
            self.model.fit(X_train, y_train)
        
        logger.info(f"  XGBoost trained with {self.model.n_estimators} trees")
        
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """Predict class probabilities."""
        return self.model.predict_proba(X)
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Predict class labels."""
        return self.model.predict(X)
    
    def get_feature_importance(self) -> pd.DataFrame:
        """
        Extract feature importance scores.
        
        Returns:
            DataFrame with feature names and importance scores
        """
        importance_df = pd.DataFrame({
            'feature': self.model.feature_names_in_,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        return importance_df


class LightGBMModel:
    """
    LightGBM classifier with class imbalance handling.
    
    LightGBM uses leaf-wise tree growth which is faster than XGBoost but can
    overfit more easily. Generally performs better on large datasets. Uses
    is_unbalance parameter for class imbalance handling.
    """
    
    def __init__(self, scale_pos_weight: float, params: Optional[Dict] = None):
        """
        Initialize LightGBM model.
        
        Args:
            scale_pos_weight: Weight for positive class
            params: Model hyperparameters
        """
        self.name = "LightGBM"
        self.scale_pos_weight = scale_pos_weight
        
        default_params = {
            'num_leaves': 31,
            'max_depth': -1,
            'learning_rate': 0.05,
            'n_estimators': 200,
            'min_child_samples': 20,
            'subsample': 0.8,
            'subsample_freq': 1,
            'colsample_bytree': 0.8,
            'reg_alpha': 0,
            'reg_lambda': 0,
            'random_state': 42,
            'n_jobs': -1,
            'verbose': -1,
            'class_weight': {0: 1.0, 1: scale_pos_weight}
        }
        
        if params:
            default_params.update(params)
        
        self.model = lgb.LGBMClassifier(**default_params)
        self.params = default_params
        
    def fit(
        self, 
        X_train: pd.DataFrame, 
        y_train: pd.Series,
        X_val: Optional[pd.DataFrame] = None,
        y_val: Optional[pd.Series] = None,
        early_stopping_rounds: int = 20
    ) -> None:
        """
        Train LightGBM model with optional early stopping.
        
        Args:
            X_train: Training features
            y_train: Training labels
            X_val: Validation features
            y_val: Validation labels
            early_stopping_rounds: Patience for early stopping
        """
        if X_val is not None and y_val is not None:
            self.model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)]
            )
        else:
            self.model.fit(X_train, y_train)
        
        logger.info(f"  LightGBM trained with {self.model.n_estimators} trees")
        
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """Predict class probabilities."""
        return self.model.predict_proba(X)
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Predict class labels."""
        return self.model.predict(X)
    
    def get_feature_importance(self) -> pd.DataFrame:
        """
        Extract feature importance scores.
        
        Returns:
            DataFrame with feature names and importance scores
        """
        importance_df = pd.DataFrame({
            'feature': self.model.feature_name_,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        return importance_df


class RandomForestModel:
    """
    Random Forest classifier with class imbalance handling.
    
    Random Forest uses bagging (bootstrap aggregating) which makes it more
    robust to overfitting than boosting methods. Generally slower but more
    stable. Uses class_weight='balanced' for imbalance handling.
    """
    
    def __init__(self, params: Optional[Dict] = None):
        """
        Initialize Random Forest model.
        
        Args:
            params: Model hyperparameters
        """
        self.name = "Random Forest"
        
        default_params = {
            'n_estimators': 200,
            'max_depth': 20,
            'min_samples_split': 10,
            'min_samples_leaf': 5,
            'max_features': 'sqrt',
            'class_weight': 'balanced',
            'random_state': 42,
            'n_jobs': -1,
            'verbose': 0
        }
        
        if params:
            default_params.update(params)
        
        self.model = RandomForestClassifier(**default_params)
        self.params = default_params
        
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series, **kwargs) -> None:
        """
        Train Random Forest model.
        
        Args:
            X_train: Training features
            y_train: Training labels
        """
        self.model.fit(X_train, y_train)
        logger.info(f"  Random Forest trained with {self.model.n_estimators} trees")
        
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """Predict class probabilities."""
        return self.model.predict_proba(X)
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Predict class labels."""
        return self.model.predict(X)
    
    def get_feature_importance(self) -> pd.DataFrame:
        """
        Extract feature importance scores.
        
        Returns:
            DataFrame with feature names and importance scores
        """
        importance_df = pd.DataFrame({
            'feature': self.model.feature_names_in_,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        return importance_df


print("\n" + "=" * 80)
print("SECTION 6: TRADITIONAL ML MODELS (TREE-BASED)")
print("=" * 80)

print("\nStep 6.1: Feature encoding for tree models...")
print("-" * 80)

encoder = FeatureEncoder(method='label')
X_train_encoded = encoder.fit_transform(X_train, categorical_cols)
X_val_encoded = encoder.transform(X_val, categorical_cols)
X_test_encoded = encoder.transform(X_test, categorical_cols)

logger.info(f"  Encoded {len(categorical_cols)} categorical features")
logger.info(f"  Training shape: {X_train_encoded.shape}")

print("\nStep 6.2: Initializing models with class imbalance handling...")
print("-" * 80)

scale_pos_weight = imbalance_report['scale_pos_weight']
print(f"Scale pos weight: {scale_pos_weight:.2f}")

xgb_model = XGBoostModel(scale_pos_weight=scale_pos_weight)
rf_model = RandomForestModel()

models = [xgb_model, rf_model]

if LIGHTGBM_AVAILABLE:
    lgb_model = LightGBMModel(scale_pos_weight=scale_pos_weight)
    models.append(lgb_model)
else:
    print("  Note: LightGBM not available, skipping")

print("\nStep 6.3: Training models with early stopping...")
print("-" * 80)

ml_metrics = []

for model in models:
    print(f"\nTraining: {model.name}")
    start_time = time.time()
    
    if hasattr(model, 'model'):
        if LIGHTGBM_AVAILABLE and isinstance(model.model, lgb.LGBMClassifier):
            model.fit(X_train_encoded, y_train, X_val_encoded, y_val, early_stopping_rounds=20)
        elif isinstance(model.model, xgb.XGBClassifier):
            model.fit(X_train_encoded, y_train, X_val_encoded, y_val, early_stopping_rounds=20)
        else:
            model.fit(X_train_encoded, y_train)
    else:
        model.fit(X_train_encoded, y_train)
    
    train_time = time.time() - start_time
    print(f"  Training time: {train_time:.1f}s")
    
    y_val_prob = model.predict_proba(X_val_encoded)[:, 1]
    y_val_pred = model.predict(X_val_encoded)
    
    metrics = evaluator.compute_metrics(
        y_true=y_val.values,
        y_pred=y_val_pred,
        y_prob=y_val_prob,
        model_name=model.name
    )
    
    metrics['train_time'] = train_time
    ml_metrics.append(metrics)
    
    print(f"  PR-AUC:       {metrics['pr_auc']:.4f}")
    print(f"  ROC-AUC:      {metrics['roc_auc']:.4f}")
    print(f"  Precision:    {metrics['precision']:.4f}")
    print(f"  Recall:       {metrics['recall']:.4f}")
    print(f"  F1:           {metrics['f1']:.4f}")
    print(f"  TP: {metrics['confusion_matrix']['tp']:,}  FP: {metrics['confusion_matrix']['fp']:,}")
    print(f"  FN: {metrics['confusion_matrix']['fn']:,}  TN: {metrics['confusion_matrix']['tn']:,}")
    print(f"  Coverage:     {metrics['coverage_rate']:.4f} ({metrics['coverage_rate']*100:.1f}% of shortages)")
    print(f"  False Alarms: {metrics['false_alarm_rate']:.4f} ({metrics['false_alarm_rate']*100:.2f}% of negatives)")
    
    evaluator.save_metrics(metrics, f"{model.name.lower().replace(' ', '_')}_metrics.json")

print("\n" + "-" * 80)
print("Step 6.4: Model comparison...")
print("-" * 80)

comparison_df = evaluator.generate_comparison_table(ml_metrics)
print("\n" + str(comparison_df.to_string(index=False)))

comparison_df.to_csv(Path("models/baselines") / "ml_comparison.csv", index=False)

evaluator.plot_pr_curve(ml_metrics, filename="ml_pr_curves.png")

print("\n" + "-" * 80)
print("Step 6.5: Feature importance analysis...")
print("-" * 80)

for model in models:
    if hasattr(model, 'get_feature_importance'):
        importance_df = model.get_feature_importance()
        print(f"\n{model.name} - Top 10 Features:")
        print(importance_df.head(10).to_string(index=False))
        
        importance_df.to_csv(
            Path("models/baselines") / f"{model.name.lower().replace(' ', '_')}_feature_importance.csv",
            index=False
        )

print("\n" + "=" * 80)
print("SECTION 6 COMPLETE - INITIAL MODEL EVALUATION")
print("=" * 80)

best_ml_model = max(ml_metrics, key=lambda x: x['pr_auc'])
lr_pr_auc = lr_metrics['pr_auc']

print(f"\nBest Performing Model (Default Hyperparameters):")
print(f"  Model:        {best_ml_model['model_name']}")
print(f"  PR-AUC:       {best_ml_model['pr_auc']:.4f}")
print(f"  ROC-AUC:      {best_ml_model['roc_auc']:.4f}")
print(f"  Precision:    {best_ml_model['precision']:.4f}")
print(f"  Recall:       {best_ml_model['recall']:.4f}")
print(f"  F1 Score:     {best_ml_model['f1']:.4f}")
print(f"  Coverage:     {best_ml_model['coverage_rate']:.4f} ({best_ml_model['coverage_rate']*100:.1f}% of shortages caught)")
print(f"  True Pos:     {best_ml_model['confusion_matrix']['tp']:,}")
print(f"  False Pos:    {best_ml_model['confusion_matrix']['fp']:,}")
print(f"  False Neg:    {best_ml_model['confusion_matrix']['fn']:,}")

improvement = (best_ml_model['pr_auc'] - lr_pr_auc) / lr_pr_auc * 100
print(f"\nPerformance vs Baseline:")
print(f"  Baseline (Logistic Regression) PR-AUC: {lr_pr_auc:.4f}")
print(f"  Best ML Model PR-AUC:                  {best_ml_model['pr_auc']:.4f}")
print(f"  Improvement:                           {improvement:+.1f}%")

if best_ml_model['pr_auc'] > lr_pr_auc:
    print(f"\nSuccess! {best_ml_model['model_name']} beats baseline by {improvement:.1f}%")
else:
    print("\nWarning: ML models did not beat baseline.")

print("\nAll Models Performance Summary (Default Settings):")
print("-" * 80)
print(f"{'Model':<20} {'PR-AUC':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("-" * 80)
for m in sorted(ml_metrics, key=lambda x: x['pr_auc'], reverse=True):
    print(f"{m['model_name']:<20} {m['pr_auc']:<10.4f} {m['precision']:<12.4f} {m['recall']:<10.4f} {m['f1']:<10.4f}")

print("\n" + "=" * 80)
print("OBSERVATIONS & NEXT STEPS")
print("=" * 80)

low_precision_models = [m for m in ml_metrics if m['precision'] < 0.30 and m['recall'] > 0.5]
if low_precision_models:
    print("\nLow Precision Alert (using default threshold=0.5):")
    for m in low_precision_models:
        fp_rate = m['confusion_matrix']['fp'] / (m['confusion_matrix']['fp'] + m['confusion_matrix']['tn'])
        print(f"  {m['model_name']}: {m['precision']:.1%} precision, {m['confusion_matrix']['fp']:,} false positives ({fp_rate:.1%} of negatives)")
    print("\n  This is EXPECTED with severe class imbalance (1.86% positive rate).")
    print("  Threshold tuning (Section 7) will optimize Precision-Recall tradeoff.")

zero_pred_models = [m for m in ml_metrics if m['confusion_matrix']['tp'] == 0]
if zero_pred_models:
    print("\nZero Predictions Alert:")
    for m in zero_pred_models:
        print(f"  {m['model_name']}: Predicts all negative (TP=0). Needs hyperparameter tuning.")

print("\nNext Steps:")
print("  Section 7: Hyperparameter Tuning (Optuna)")
print("    - Optimize each model's hyperparameters")
print("    - Try different class weight strategies")
print("    - Maximize PR-AUC on validation set")
print("\n  Section 8: Threshold Optimization")
print("    - Find optimal decision threshold for each model")
print("    - Balance Precision vs Recall based on business needs")
print("    - Multi-objective optimization (Precision, Recall, F1)")
print("\n  Section 9: Model Selection & Deployment")
print("    - Compare tuned models on test set")
print("    - Select final model for production")

print("=" * 80)

2026-01-17 17:13:42,440 - [INFO] - Executing Section 6 (Traditional ML Models)...

SECTION 6: TRADITIONAL ML MODELS (TREE-BASED)

Step 6.1: Feature encoding for tree models...
--------------------------------------------------------------------------------
2026-01-17 17:14:03,998 - [INFO] -   Encoded 2 categorical features
2026-01-17 17:14:04,000 - [INFO] -   Training shape: (522629, 28)

Step 6.2: Initializing models with class imbalance handling...
--------------------------------------------------------------------------------
Scale pos weight: 63.46

Step 6.3: Training models with early stopping...
--------------------------------------------------------------------------------

Training: XGBoost
2026-01-17 17:14:10,664 - [INFO] -   XGBoost trained with 200 trees
  Training time: 6.7s
  PR-AUC:       0.6942
  ROC-AUC:      0.9736
  Precision:    0.2870
  Recall:       0.8018
  F1:           0.4227
  TP: 1,254  FP: 3,115
  FN: 310  TN: 79,616
  Coverage:     0.8018 (80.2% of shortag

In [103]:
# ==============================================================================
# SECTION 7: HYPERPARAMETER TUNING WITH OPTUNA
# ==============================================================================

logger = logging.getLogger(__name__)
logger.info("Executing Section 7 (Hyperparameter Tuning)...")

try:
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    logger.error("Optuna not available. Install with: pip install optuna")
    raise ImportError("Optuna is required for Section 7. Install with: pip install optuna")


class OptunaOptimizer:
    """
    Hyperparameter optimization using Optuna with TPE sampler.
    
    This class provides a unified interface for hyperparameter tuning across
    different model types. Uses Tree-structured Parzen Estimator (TPE) for
    efficient search and includes early stopping to prevent overfitting.
    
    Design Principles:
        - Model-specific search spaces (XGBoost != LightGBM != RF)
        - PR-AUC as optimization metric (handles imbalance)
        - Pruning for computational efficiency
        - Reproducible results (fixed seed)
    """
    
    def __init__(
        self,
        model_type: str,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_val: pd.DataFrame,
        y_val: pd.Series,
        scale_pos_weight: float,
        n_trials: int = 20,
        random_state: int = 42
    ):
        """
        Initialize Optuna optimizer.
        
        Args:
            model_type: Model type ('xgboost', 'lightgbm', 'randomforest')
            X_train: Training features
            y_train: Training labels
            X_val: Validation features
            y_val: Validation labels
            scale_pos_weight: Class imbalance weight
            n_trials: Number of optimization trials
            random_state: Random seed
        """
        self.model_type = model_type
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.scale_pos_weight = scale_pos_weight
        self.n_trials = n_trials
        self.random_state = random_state
        self.best_params = None
        self.best_score = None
        
    def _objective_xgboost(self, trial: optuna.Trial) -> float:
        """
        Objective function for XGBoost hyperparameter optimization.
        
        Args:
            trial: Optuna trial object
            
        Returns:
            PR-AUC score on validation set
        """
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'gamma': trial.suggest_float('gamma', 0, 3),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
            'scale_pos_weight': self.scale_pos_weight,
            'random_state': self.random_state,
            'n_jobs': -1,
            'eval_metric': 'aucpr'
        }
        
        model = xgb.XGBClassifier(**params)
        model.fit(
            self.X_train, self.y_train,
            eval_set=[(self.X_val, self.y_val)],
            verbose=False
        )
        
        y_pred_proba = model.predict_proba(self.X_val)[:, 1]
        precision, recall, _ = precision_recall_curve(self.y_val, y_pred_proba)
        pr_auc = auc(recall, precision)
        
        return pr_auc
    
    def _objective_lightgbm(self, trial: optuna.Trial) -> float:
        """
        Objective function for LightGBM hyperparameter optimization.
        
        Args:
            trial: Optuna trial object
            
        Returns:
            PR-AUC score on validation set
        """
        params = {
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 80),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'subsample_freq': trial.suggest_int('subsample_freq', 0, 5),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
            'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0, 3),
            'class_weight': {0: 1.0, 1: self.scale_pos_weight},
            'random_state': self.random_state,
            'n_jobs': -1,
            'verbose': -1
        }
        
        model = lgb.LGBMClassifier(**params)
        model.fit(
            self.X_train, self.y_train,
            eval_set=[(self.X_val, self.y_val)],
            callbacks=[lgb.early_stopping(15, verbose=False)]
        )
        
        y_pred_proba = model.predict_proba(self.X_val)[:, 1]
        precision, recall, _ = precision_recall_curve(self.y_val, y_pred_proba)
        pr_auc = auc(recall, precision)
        
        return pr_auc
    
    def _objective_randomforest(self, trial: optuna.Trial) -> float:
        """
        Objective function for Random Forest hyperparameter optimization.
        
        Args:
            trial: Optuna trial object
            
        Returns:
            PR-AUC score on validation set
        """
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'max_depth': trial.suggest_int('max_depth', 10, 25),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
            'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0, 0.05),
            'class_weight': 'balanced',
            'random_state': self.random_state,
            'n_jobs': -1,
            'verbose': 0
        }
        
        model = RandomForestClassifier(**params)
        model.fit(self.X_train, self.y_train)
        
        y_pred_proba = model.predict_proba(self.X_val)[:, 1]
        precision, recall, _ = precision_recall_curve(self.y_val, y_pred_proba)
        pr_auc = auc(recall, precision)
        
        return pr_auc
    
    def optimize(self) -> Dict[str, Any]:
        """
        Run hyperparameter optimization.
        
        Returns:
            Dictionary with best parameters and score
        """
        sampler = TPESampler(seed=self.random_state)
        study = optuna.create_study(
            direction='maximize',
            sampler=sampler
        )
        
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        
        if self.model_type == 'xgboost':
            study.optimize(self._objective_xgboost, n_trials=self.n_trials, show_progress_bar=False)
        elif self.model_type == 'lightgbm':
            study.optimize(self._objective_lightgbm, n_trials=self.n_trials, show_progress_bar=False)
        elif self.model_type == 'randomforest':
            study.optimize(self._objective_randomforest, n_trials=self.n_trials, show_progress_bar=False)
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
        
        self.best_params = study.best_params
        self.best_score = study.best_value
        
        logger.info(f"  Best PR-AUC: {self.best_score:.4f}")
        logger.info(f"  Best params: {self.best_params}")
        
        return {
            'best_params': self.best_params,
            'best_score': self.best_score,
            'study': study
        }


def train_tuned_model(
    model_type: str,
    best_params: Dict[str, Any],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    scale_pos_weight: float
) -> Any:
    """
    Train final model with best hyperparameters.
    
    Args:
        model_type: Model type
        best_params: Optimized hyperparameters
        X_train: Training features
        y_train: Training labels
        X_val: Validation features
        y_val: Validation labels
        scale_pos_weight: Class imbalance weight
        
    Returns:
        Trained model
    """
    if model_type == 'xgboost':
        best_params['scale_pos_weight'] = scale_pos_weight
        best_params['random_state'] = 42
        best_params['n_jobs'] = -1
        best_params['eval_metric'] = 'aucpr'
        
        model = xgb.XGBClassifier(**best_params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
    elif model_type == 'lightgbm':
        best_params['class_weight'] = {0: 1.0, 1: scale_pos_weight}
        best_params['random_state'] = 42
        best_params['n_jobs'] = -1
        best_params['verbose'] = -1
        
        model = lgb.LGBMClassifier(**best_params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(15, verbose=False)]
        )
        
    elif model_type == 'randomforest':
        best_params['class_weight'] = 'balanced'
        best_params['random_state'] = 42
        best_params['n_jobs'] = -1
        best_params['verbose'] = 0
        
        model = RandomForestClassifier(**best_params)
        model.fit(X_train, y_train)
    
    return model


print("\n" + "=" * 80)
print("SECTION 7: HYPERPARAMETER TUNING WITH OPTUNA")
print("=" * 80)

tuned_models = {}
tuning_results = {}

model_configs = [
    ('xgboost', 20, "XGBoost"),
    ('lightgbm', 20, "LightGBM"),
    ('randomforest', 5, "Random Forest")
]

for model_type, n_trials, display_name in model_configs:
    print(f"\n{'-' * 80}")
    print(f"Optimizing: {display_name} ({n_trials} trials)")
    print(f"{'-' * 80}")
    
    start_time = time.time()
    
    optimizer = OptunaOptimizer(
        model_type=model_type,
        X_train=X_train_encoded,
        y_train=y_train,
        X_val=X_val_encoded,
        y_val=y_val,
        scale_pos_weight=scale_pos_weight,
        n_trials=n_trials,
        random_state=42
    )
    
    result = optimizer.optimize()
    tuning_time = time.time() - start_time
    
    print(f"\nOptimization complete in {tuning_time:.1f}s")
    print(f"  Best PR-AUC: {result['best_score']:.4f}")
    print(f"  Best params:")
    for param, value in result['best_params'].items():
        if isinstance(value, float):
            print(f"    {param}: {value:.4f}")
        else:
            print(f"    {param}: {value}")
    
    print(f"\nTraining final {display_name} model with best params...")
    tuned_model = train_tuned_model(
        model_type=model_type,
        best_params=result['best_params'],
        X_train=X_train_encoded,
        y_train=y_train,
        X_val=X_val_encoded,
        y_val=y_val,
        scale_pos_weight=scale_pos_weight
    )
    
    y_val_prob = tuned_model.predict_proba(X_val_encoded)[:, 1]
    y_val_pred = tuned_model.predict(X_val_encoded)
    
    metrics = evaluator.compute_metrics(
        y_true=y_val.values,
        y_pred=y_val_pred,
        y_prob=y_val_prob,
        model_name=f"{display_name} (Tuned)"
    )
    
    print(f"\nTuned Model Performance:")
    print(f"  PR-AUC:       {metrics['pr_auc']:.4f}")
    print(f"  ROC-AUC:      {metrics['roc_auc']:.4f}")
    print(f"  Precision:    {metrics['precision']:.4f}")
    print(f"  Recall:       {metrics['recall']:.4f}")
    print(f"  F1:           {metrics['f1']:.4f}")
    print(f"  TP: {metrics['confusion_matrix']['tp']:,}  FP: {metrics['confusion_matrix']['fp']:,}")
    print(f"  FN: {metrics['confusion_matrix']['fn']:,}  TN: {metrics['confusion_matrix']['tn']:,}")
    
    tuned_models[model_type] = tuned_model
    tuning_results[model_type] = {
        'metrics': metrics,
        'best_params': result['best_params'],
        'best_score': result['best_score'],
        'tuning_time': tuning_time
    }
    
    with open(Path("models/baselines") / f"{model_type}_tuned_params.json", 'w') as f:
        json.dump(result['best_params'], f, indent=2, default=str)
    
    evaluator.save_metrics(metrics, f"{model_type}_tuned_metrics.json")

print("\n" + "=" * 80)
print("SECTION 7 COMPLETE - HYPERPARAMETER TUNING SUMMARY")
print("=" * 80)

comparison_data = []
for model_type, results in tuning_results.items():
    metrics = results['metrics']
    comparison_data.append({
        'Model': metrics['model_name'],
        'PR-AUC': f"{metrics['pr_auc']:.4f}",
        'ROC-AUC': f"{metrics['roc_auc']:.4f}",
        'Precision': f"{metrics['precision']:.4f}",
        'Recall': f"{metrics['recall']:.4f}",
        'F1': f"{metrics['f1']:.4f}",
        'Tuning Time': f"{results['tuning_time']:.1f}s"
    })

tuned_comparison_df = pd.DataFrame(comparison_data)
print("\nTuned Models Performance:")
print(tuned_comparison_df.to_string(index=False))

tuned_comparison_df.to_csv(Path("models/baselines") / "tuned_models_comparison.csv", index=False)

best_tuned = max(tuning_results.items(), key=lambda x: x[1]['metrics']['pr_auc'])
best_model_type, best_results = best_tuned

print(f"\nBest Tuned Model: {best_results['metrics']['model_name']}")
print(f"  PR-AUC:    {best_results['metrics']['pr_auc']:.4f}")
print(f"  Precision: {best_results['metrics']['precision']:.4f}")
print(f"  Recall:    {best_results['metrics']['recall']:.4f}")

print("\nNote: All models still using default threshold=0.5")
print("Next: Section 8 will optimize decision thresholds for each model")
print("=" * 80)

2026-01-17 18:16:23,798 - [INFO] - Executing Section 7 (Hyperparameter Tuning)...


[I 2026-01-17 18:16:23,807] A new study created in memory with name: no-name-b71be8d7-6c02-4944-ab1b-64ae65902736



SECTION 7: HYPERPARAMETER TUNING WITH OPTUNA

--------------------------------------------------------------------------------
Optimizing: XGBoost (20 trials)
--------------------------------------------------------------------------------
2026-01-17 18:18:32,430 - [INFO] -   Best PR-AUC: 0.8283
2026-01-17 18:18:32,433 - [INFO] -   Best params: {'max_depth': 8, 'learning_rate': 0.1413031891597433, 'n_estimators': 255, 'min_child_weight': 5, 'subsample': 0.9917014344310534, 'colsample_bytree': 0.8430470239667245, 'gamma': 1.6277319616381094, 'reg_alpha': 3.3112243763586915, 'reg_lambda': 3.5280458067654856}

Optimization complete in 128.6s
  Best PR-AUC: 0.8283
  Best params:
    max_depth: 8
    learning_rate: 0.1413
    n_estimators: 255
    min_child_weight: 5
    subsample: 0.9917
    colsample_bytree: 0.8430
    gamma: 1.6277
    reg_alpha: 3.3112
    reg_lambda: 3.5280

Training final XGBoost model with best params...

Tuned Model Performance:
  PR-AUC:       0.8283
  ROC-AUC:   

In [105]:
# ==============================================================================
# SECTION 8: THRESHOLD OPTIMIZATION
# ==============================================================================

logger = logging.getLogger(__name__)
logger.info("Executing Section 8 (Threshold Optimization)...")


class ThresholdOptimizer:
    """
    Multi-objective threshold optimization for binary classification.
    
    This class finds optimal decision thresholds by exploring the Precision-Recall
    tradeoff. Supports multiple optimization strategies (maximize F1, maximize
    recall with precision constraint, Pareto frontier analysis).
    
    Design Principles:
        - Business-aligned: Consider false alarm costs
        - Multi-objective: Balance Precision, Recall, F1
        - Interpretable: Clear visualization of tradeoffs
        - Production-ready: Provides concrete threshold recommendations
    """
    
    def __init__(self, output_dir: str = "models/baselines"):
        """
        Initialize threshold optimizer.
        
        Args:
            output_dir: Directory for saving results
        """
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
    def find_optimal_threshold(
        self,
        y_true: np.ndarray,
        y_prob: np.ndarray,
        strategy: str = 'f1',
        min_precision: float = 0.20,
        min_recall: float = 0.60
    ) -> Dict[str, Any]:
        """
        Find optimal decision threshold using specified strategy.
        
        Args:
            y_true: True labels
            y_prob: Predicted probabilities
            strategy: Optimization strategy ('f1', 'recall_constrained', 'precision_constrained')
            min_precision: Minimum acceptable precision (for recall_constrained)
            min_recall: Minimum acceptable recall (for precision_constrained)
            
        Returns:
            Dictionary with optimal threshold and metrics
        """
        precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
        
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        
        if strategy == 'f1':
            best_idx = np.argmax(f1_scores)
            optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
            
        elif strategy == 'recall_constrained':
            valid_idx = precision >= min_precision
            if valid_idx.sum() == 0:
                logger.warning(f"No threshold achieves precision >= {min_precision}. Using F1 strategy.")
                best_idx = np.argmax(f1_scores)
            else:
                valid_recall = recall[valid_idx]
                best_valid_idx = np.argmax(valid_recall)
                best_idx = np.where(valid_idx)[0][best_valid_idx]
            
            optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
            
        elif strategy == 'precision_constrained':
            valid_idx = recall >= min_recall
            if valid_idx.sum() == 0:
                logger.warning(f"No threshold achieves recall >= {min_recall}. Using F1 strategy.")
                best_idx = np.argmax(f1_scores)
            else:
                valid_precision = precision[valid_idx]
                best_valid_idx = np.argmax(valid_precision)
                best_idx = np.where(valid_idx)[0][best_valid_idx]
            
            optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
        
        y_pred = (y_prob >= optimal_threshold).astype(int)
        
        final_precision = precision_score(y_true, y_pred, zero_division=0)
        final_recall = recall_score(y_true, y_pred, zero_division=0)
        final_f1 = f1_score(y_true, y_pred, zero_division=0)
        
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        
        return {
            'threshold': float(optimal_threshold),
            'precision': float(final_precision),
            'recall': float(final_recall),
            'f1': float(final_f1),
            'tp': int(tp),
            'fp': int(fp),
            'fn': int(fn),
            'tn': int(tn),
            'strategy': strategy
        }
    
    def plot_threshold_analysis(
        self,
        y_true: np.ndarray,
        y_prob: np.ndarray,
        model_name: str,
        optimal_results: Dict[str, Any]
    ) -> None:
        """
        Generate threshold analysis plots.
        
        Args:
            y_true: True labels
            y_prob: Predicted probabilities
            model_name: Model name for plot title
            optimal_results: Optimal threshold results
        """
        precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        axes[0, 0].plot(thresholds, precision[:-1], label='Precision', linewidth=2)
        axes[0, 0].plot(thresholds, recall[:-1], label='Recall', linewidth=2)
        axes[0, 0].axvline(optimal_results['threshold'], color='red', linestyle='--', 
                          label=f"Optimal={optimal_results['threshold']:.3f}")
        axes[0, 0].set_xlabel('Threshold', fontsize=11)
        axes[0, 0].set_ylabel('Score', fontsize=11)
        axes[0, 0].set_title('Precision & Recall vs Threshold', fontsize=12, fontweight='bold')
        axes[0, 0].legend(fontsize=10)
        axes[0, 0].grid(True, alpha=0.3)
        
        axes[0, 1].plot(thresholds, f1_scores[:-1], linewidth=2, color='green')
        axes[0, 1].axvline(optimal_results['threshold'], color='red', linestyle='--',
                          label=f"Optimal={optimal_results['threshold']:.3f}")
        axes[0, 1].set_xlabel('Threshold', fontsize=11)
        axes[0, 1].set_ylabel('F1 Score', fontsize=11)
        axes[0, 1].set_title('F1 Score vs Threshold', fontsize=12, fontweight='bold')
        axes[0, 1].legend(fontsize=10)
        axes[0, 1].grid(True, alpha=0.3)
        
        axes[1, 0].plot(recall, precision, linewidth=2)
        axes[1, 0].scatter([optimal_results['recall']], [optimal_results['precision']], 
                          color='red', s=100, zorder=5,
                          label=f"Optimal (P={optimal_results['precision']:.3f}, R={optimal_results['recall']:.3f})")
        axes[1, 0].set_xlabel('Recall', fontsize=11)
        axes[1, 0].set_ylabel('Precision', fontsize=11)
        axes[1, 0].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
        axes[1, 0].legend(fontsize=10)
        axes[1, 0].grid(True, alpha=0.3)
        
        cm = np.array([[optimal_results['tn'], optimal_results['fp']],
                       [optimal_results['fn'], optimal_results['tp']]])
        im = axes[1, 1].imshow(cm, interpolation='nearest', cmap='Blues')
        axes[1, 1].set_title('Confusion Matrix at Optimal Threshold', fontsize=12, fontweight='bold')
        tick_marks = np.arange(2)
        axes[1, 1].set_xticks(tick_marks)
        axes[1, 1].set_yticks(tick_marks)
        axes[1, 1].set_xticklabels(['Negative', 'Positive'])
        axes[1, 1].set_yticklabels(['Negative', 'Positive'])
        axes[1, 1].set_ylabel('True Label', fontsize=11)
        axes[1, 1].set_xlabel('Predicted Label', fontsize=11)
        
        for i in range(2):
            for j in range(2):
                axes[1, 1].text(j, i, f'{cm[i, j]:,}',
                              ha="center", va="center", fontsize=14,
                              color="white" if cm[i, j] > cm.max() / 2 else "black")
        
        plt.colorbar(im, ax=axes[1, 1])
        plt.tight_layout()
        
        output_path = self.output_dir / f"{model_name.lower().replace(' ', '_')}_threshold_analysis.png"
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        logger.info(f"  Threshold analysis plot saved to: {output_path}")


print("\n" + "=" * 80)
print("SECTION 8: THRESHOLD OPTIMIZATION")
print("=" * 80)

threshold_optimizer = ThresholdOptimizer(output_dir="models/baselines")

optimized_models = {}
all_threshold_results = []

strategies = [
    ('f1', 'Maximize F1', {}),
    ('recall_constrained', 'Maximize Recall (Precision >= 20%)', {'min_precision': 0.20}),
    ('precision_constrained', 'Maximize Precision (Recall >= 60%)', {'min_recall': 0.60})
]

for model_type, model in tuned_models.items():
    print(f"\n{'=' * 80}")
    print(f"Optimizing Threshold: {model_type.upper()}")
    print(f"{'=' * 80}")
    
    y_val_prob = model.predict_proba(X_val_encoded)[:, 1]
    
    print(f"\nStep 1: Testing multiple thresholds to find optimal configuration...")
    print(f"{'-' * 80}")
    
    threshold_scan_results = []
    test_thresholds = np.arange(0.05, 0.95, 0.05)
    
    for thresh in test_thresholds:
        y_pred = (y_val_prob >= thresh).astype(int)
        
        prec = precision_score(y_val.values, y_pred, zero_division=0)
        rec = recall_score(y_val.values, y_pred, zero_division=0)
        f1 = f1_score(y_val.values, y_pred, zero_division=0)
        
        tn, fp, fn, tp = confusion_matrix(y_val.values, y_pred).ravel()
        
        precision_curve, recall_curve, _ = precision_recall_curve(y_val.values, y_val_prob)
        pr_auc_val = auc(recall_curve, precision_curve)
        
        threshold_scan_results.append({
            'Model': model_type.upper(),
            'Threshold': f"{thresh:.2f}",
            'Precision': f"{prec:.4f}",
            'Recall': f"{rec:.4f}",
            'F1': f"{f1:.4f}",
            'PR-AUC': f"{pr_auc_val:.4f}",
            'TP': int(tp),
            'FP': int(fp),
            'FN': int(fn),
            'TN': int(tn)
        })
    
    threshold_scan_df = pd.DataFrame(threshold_scan_results)
    
    print(f"\nThreshold Tuning Table for {model_type.upper()}:")
    print(threshold_scan_df.to_string(index=False))
    
    threshold_scan_df.to_csv(
        Path("models/baselines") / f"{model_type}_threshold_scan.csv",
        index=False
    )
    
    all_threshold_results.extend(threshold_scan_results)
    
    print(f"\n{'-' * 80}")
    print(f"Step 2: Finding optimal thresholds using different strategies...")
    print(f"{'-' * 80}")
    
    model_results = {}
    
    for strategy_name, strategy_desc, strategy_params in strategies:
        print(f"\nStrategy: {strategy_desc}")
        
        result = threshold_optimizer.find_optimal_threshold(
            y_true=y_val.values,
            y_prob=y_val_prob,
            strategy=strategy_name,
            **strategy_params
        )
        
        model_results[strategy_name] = result
        
        print(f"  Optimal Threshold: {result['threshold']:.4f}")
        print(f"  Precision:         {result['precision']:.4f}")
        print(f"  Recall:            {result['recall']:.4f}")
        print(f"  F1 Score:          {result['f1']:.4f}")
        print(f"  TP: {result['tp']:,}  FP: {result['fp']:,}  FN: {result['fn']:,}  TN: {result['tn']:,}")
    
    with open(Path("models/baselines") / f"{model_type}_threshold_results.json", 'w') as f:
        json.dump(model_results, f, indent=2)
    
    best_strategy_result = model_results['f1']
    threshold_optimizer.plot_threshold_analysis(
        y_true=y_val.values,
        y_prob=y_val_prob,
        model_name=f"{model_type}_tuned",
        optimal_results=best_strategy_result
    )
    
    optimized_models[model_type] = {
        'model': model,
        'threshold_results': model_results,
        'best_threshold': best_strategy_result['threshold'],
        'threshold_scan': threshold_scan_df
    }

print("\n" + "=" * 80)
print("SECTION 8 COMPLETE - THRESHOLD OPTIMIZATION SUMMARY")
print("=" * 80)

print("\n" + "-" * 80)
print("COMPLETE THRESHOLD SCAN RESULTS (ALL MODELS)")
print("-" * 80)

all_threshold_df = pd.DataFrame(all_threshold_results)
print(all_threshold_df.to_string(index=False))

all_threshold_df.to_csv(
    Path("models/baselines") / "all_models_threshold_scan.csv",
    index=False
)

summary_data = []
for model_type, opt_results in optimized_models.items():
    for strategy_name, result in opt_results['threshold_results'].items():
        summary_data.append({
            'Model': model_type.upper(),
            'Strategy': strategy_name.replace('_', ' ').title(),
            'Threshold': f"{result['threshold']:.4f}",
            'Precision': f"{result['precision']:.4f}",
            'Recall': f"{result['recall']:.4f}",
            'F1': f"{result['f1']:.4f}",
            'TP': result['tp'],
            'FP': result['fp']
        })

summary_df = pd.DataFrame(summary_data)

print("\n" + "-" * 80)
print("OPTIMAL THRESHOLD CONFIGURATIONS BY STRATEGY")
print("-" * 80)
print(summary_df.to_string(index=False))

summary_df.to_csv(Path("models/baselines") / "threshold_optimization_summary.csv", index=False)

print("\n" + "=" * 80)
print("RECOMMENDED CONFIGURATIONS")
print("=" * 80)

for model_type, opt_results in optimized_models.items():
    f1_result = opt_results['threshold_results']['f1']
    recall_result = opt_results['threshold_results']['recall_constrained']
    precision_result = opt_results['threshold_results']['precision_constrained']
    
    print(f"\n{model_type.upper()}:")
    print(f"  For Balanced Performance (F1 Optimization):")
    print(f"    Threshold: {f1_result['threshold']:.4f}")
    print(f"    Precision: {f1_result['precision']:.4f}, Recall: {f1_result['recall']:.4f}, F1: {f1_result['f1']:.4f}")
    print(f"    TP: {f1_result['tp']:,}, FP: {f1_result['fp']:,}")
    
    print(f"  For High Coverage (Recall Priority, Precision >= 20%):")
    print(f"    Threshold: {recall_result['threshold']:.4f}")
    print(f"    Precision: {recall_result['precision']:.4f}, Recall: {recall_result['recall']:.4f}, F1: {recall_result['f1']:.4f}")
    print(f"    TP: {recall_result['tp']:,}, FP: {recall_result['fp']:,}")
    
    print(f"  For High Precision (Precision Priority, Recall >= 60%):")
    print(f"    Threshold: {precision_result['threshold']:.4f}")
    print(f"    Precision: {precision_result['precision']:.4f}, Recall: {precision_result['recall']:.4f}, F1: {precision_result['f1']:.4f}")
    print(f"    TP: {precision_result['tp']:,}, FP: {precision_result['fp']:,}")

print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

for model_type, opt_results in optimized_models.items():
    scan_df = opt_results['threshold_scan']
    
    scan_df_parsed = scan_df.copy()
    scan_df_parsed['Threshold'] = scan_df_parsed['Threshold'].astype(float)
    scan_df_parsed['Precision'] = scan_df_parsed['Precision'].astype(float)
    scan_df_parsed['Recall'] = scan_df_parsed['Recall'].astype(float)
    scan_df_parsed['F1'] = scan_df_parsed['F1'].astype(float)
    
    best_f1_idx = scan_df_parsed['F1'].idxmax()
    best_precision_idx = scan_df_parsed['Precision'].idxmax()
    best_recall_idx = scan_df_parsed['Recall'].idxmax()
    
    print(f"\n{model_type.upper()} Threshold Analysis:")
    print(f"  Best F1 Score:  {scan_df_parsed.loc[best_f1_idx, 'F1']:.4f} at threshold {scan_df_parsed.loc[best_f1_idx, 'Threshold']:.2f}")
    print(f"  Best Precision: {scan_df_parsed.loc[best_precision_idx, 'Precision']:.4f} at threshold {scan_df_parsed.loc[best_precision_idx, 'Threshold']:.2f}")
    print(f"  Best Recall:    {scan_df_parsed.loc[best_recall_idx, 'Recall']:.4f} at threshold {scan_df_parsed.loc[best_recall_idx, 'Threshold']:.2f}")

print("\n" + "=" * 80)
print("FILES GENERATED")
print("=" * 80)
print("Per-model threshold scans:")
for model_type in optimized_models.keys():
    print(f"  - {model_type}_threshold_scan.csv")
print("\nCombined results:")
print("  - all_models_threshold_scan.csv")
print("  - threshold_optimization_summary.csv")
print("\nVisualizations:")
for model_type in optimized_models.keys():
    print(f"  - {model_type}_tuned_threshold_analysis.png")

print("\n" + "=" * 80)
print("SECTION 8 COMPLETE")
print("Threshold optimization finished successfully!")
print("=" * 80)

2026-01-17 18:33:50,968 - [INFO] - Executing Section 8 (Threshold Optimization)...

SECTION 8: THRESHOLD OPTIMIZATION

Optimizing Threshold: XGBOOST

Step 1: Testing multiple thresholds to find optimal configuration...
--------------------------------------------------------------------------------

Threshold Tuning Table for XGBOOST:
  Model Threshold Precision Recall     F1 PR-AUC   TP   FP  FN    TN
XGBOOST      0.05    0.3403 0.8804 0.4909 0.8283 1377 2669 187 80062
XGBOOST      0.10    0.4506 0.8632 0.5921 0.8283 1350 1646 214 81085
XGBOOST      0.15    0.5028 0.8497 0.6318 0.8283 1329 1314 235 81417
XGBOOST      0.20    0.5365 0.8414 0.6552 0.8283 1316 1137 248 81594
XGBOOST      0.25    0.5783 0.8312 0.6821 0.8283 1300  948 264 81783
XGBOOST      0.30    0.6115 0.8223 0.7014 0.8283 1286  817 278 81914
XGBOOST      0.35    0.6375 0.8152 0.7155 0.8283 1275  725 289 82006
XGBOOST      0.40    0.6709 0.8069 0.7327 0.8283 1262  619 302 82112
XGBOOST      0.45    0.6810 0.8024 0.7367 